# Powderday flux catalogs — quenched galaxies in the high-res 25 Mpc box

**Goal.** Multi-aperture, dusty vs dust-free photometric catalogs (fluxes **with errors**) for
**quenched** galaxies in SIMBA high-res **m25n512** (`cis25`) at the `TARGET_REDSHIFTS`
anchors **z ≈ 0.3, 0.7, 1.0, 1.5** (snapshots 134/116/105/078), and a CIGALE fit of those
mocks in which every stellar-population parameter is **pinned per object** so that the dust
attenuation $A_V$ is the measurand rather than one free parameter among thousands.

**Sample (per anchor snapshot).** `log10 M* > 10`, **passive** by the 0.2/τ criterion
(sSFR < 0.2/t_H at the anchor), and **> 20 gas particles** (plus the usual ≥ 20 star-particle floor).
The sample is split by **weak vs strong AGN feedback over the quench window**: the AGN–ISM coupling
strength `xcoup_hist` (jet-mode strength gated by gas-poorness, §8j physics) averaged between each
galaxy's **SFT and QT** (1/t and 0.2/t crossings from `find_quenching_times`); classes: **strong** = fully coupled (`xstr_quench` = 1) through the window, **weak** = bottom tercile of the remainder, **intermediate** = the rest.

**Pipeline** (same skeleton as `test_powderday.ipynb`, selection machinery from
`quench_mode_vs_sigma_gas.ipynb`):

| Part | What | Where |
|---|---|---|
| 1 | anchors + gated `BUILD_MULTI_Z` / `BUILD_BH` history builds | cluster |
| 2–3 | selection, SFT/QT, AGN split, **sample statistics** | anywhere (needs the HDF5s) |
| 4 | Stage 0 — per-galaxy particle files | cluster |
| 4b | annulus sampling QC — star/gas/dust counts per projected annulus × sightline | anywhere (needs Stage 0) |
| 5 | Stage 1 — selection HDF5 + Slurm masters (dust_on / dust_off / agn_on) → run RT | cluster |
| 6 | aperture QC on the first `.rtout.sed` | cluster |
| 7 | Stage 2 — per-aperture flux extraction → **one catalog per aperture per RT arm** | cluster |
| 7a | the **true** $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ vs the ISM — no CIGALE | anywhere |
| 7b | observed-frame CIGALE input catalogs | anywhere |
| 7c | aperture-matched **formed-mass** SFH archive (the injected prior) | cluster |
| 7d | **one CIGALE run per object** (SFH + $Z$ + age pinned) + the chunked job array | cluster |
| 7e | aperture-matched SIMBA truth | cluster |
| 7f | merged fits vs truth: **is $A_V$ recovered?** | anywhere |

**Apertures & sightlines (mock observation).** Stage 0 cuts a **100 pkpc spherical region**
around each galaxy (everything: CGM, satellites, projected neighbours — a true mock aperture,
not just member particles); the RT grid spans ±100 kpc (`zoom_box_len`). Hyperion log-spaces
`N_AP = 5` projected apertures 1→100 kpc — the 10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc**
(central → outskirts), all extracted. Each SED is peeled along **4 sightlines**
(θ,φ) = (0,0), (45,90), (90,180), (135,270) deg — one catalog per (RT arm, aperture,
inclination). `N_AP/AP_MIN_KPC/AP_MAX_KPC` and `THETA_DEG/PHI_DEG` below must match the
parameter masters that the RT jobs copy. **Requires the one-time powderday patch documented
before Stage 1 (already applied on this cluster's install).**

**Flux errors.** Hyperion's Monte-Carlo SED uncertainty, read with
`get_sed(..., uncertainties=True)` and propagated through the filter convolution
(`<filter>_err` columns; NaN if a run stored no uncertainties).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import glob
import json
import re
import subprocess
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy import units as u
from astropy.cosmology import Planck15 as COSMO   # matches the quenching machinery

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times
from simbanator.utils.geometry import sightline_unit_vectors, projected_radius

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25/s25',\n"
        "                 catalog_dir='<...>/SIMBA_25/s25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry "
        "(SIMBA boxes share the snapshot schedule)."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map — add "
                     '"snap_z_map": "zsnap_map_caesar_box100.txt" to its entry in ~/.simbanator/config.json')

# filtered-particle filename prefix (Stage 0 == Stage 1, never let them drift)
PARTICLE_PREFIX = sim.file_format.split("_{")[0]        # 'm25n512'

# ── selection: quenched + massive + realistically gas-populated ───────────────
# 2026-08-14 expansion: 9 anchors z=0.1-2 (snaps 145/134/125/116/110/105/100/090/078)
# + a mass-matched star-forming control per anchor (pop='SF' in the selection).
# The 5 NEW anchors (0.1/0.5/0.85/1.15/1.5) need BUILD_MULTI_Z + BUILD_BH first.
TARGET_REDSHIFTS = [0.1, 0.3, 0.5, 0.7, 0.85, 1.0, 1.15, 1.5, 2]
MASS_FLOOR       = 10.0        # log10(M*/Msun) > 10
PASSIVE_FACTOR   = 0.2         # passive if sSFR < 0.2 / t_H  (== the QT threshold of find_quenching_times)
NGAS_MIN         = 21          # STRICTLY > 20 gas particles at the anchor
NSTAR_MIN        = 20          # star-particle floor (same as quench_mode_vs_sigma_gas)
DUST_TO_H2_MIN   = 1e-4        # keep only M_dust/M_H2 >= 1e-4 at the anchor (drops the dust-poor half; 2026-08-05)
SF_MATCH_RATIO   = 1           # star-forming partners per quenched galaxy (nearest log M*, no replacement)

# ── AGN / coupling constants (identical to quench_mode_vs_sigma_gas §0) ──────
JET_LOGMBH    = 7.5            # jet mode: log10(M_BH) > 7.5 ...
JET_FEDD      = 0.2            #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX = 0.02           # (kept for reference; xcoup uses the f_gas gate)
XRAY_FGAS_MAX = 0.2            # coupling gate: f_gas = Mgas/M* < 0.2
GYR = 1e9

# ── history tracking ──────────────────────────────────────────────────────────
TRACK_AGE_FRAC      = 0.09     # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS       = set()

# ── heavy-build gates (set True on the cluster, then reuse the cached HDF5s) ──
BUILD_MULTI_Z = False          # per-anchor progenitor FITS + property history HDF5
BUILD_BH      = False          # per-anchor BH (mass / mdot / f_Edd) history HDF5

# ── apertures (MUST match SED_APERTURE_* in simbanator/sed/parameters_master*.py) ──
N_AP       = 5           # SED_APERTURE_NAP: 10^(k/2) ladder -> 1, 3.16, 10, 31.6, 100 kpc
AP_MIN_KPC = 1.0
AP_MAX_KPC = 100.0
APERTURE_RADII_KPC = np.geomspace(AP_MIN_KPC, AP_MAX_KPC, N_AP)
# central -> outskirts; ALL rungs are extracted (nominal labels, true radii above)
TARGET_AP_KPC   = [1, 3, 10, 32, 100]
WANTED_AP_IDX   = list(range(N_AP))
APERTURE_LABELS = [f"ap{t:g}kpc" for t in TARGET_AP_KPC]
# annulus edges between consecutive rungs; the OUTER rung names the annulus
# (ann1kpc = the 0->1 kpc disc == ap1kpc) — shared by Parts 4b/4c/7a/7c/7f
R_EDGES        = np.concatenate([[0.0], APERTURE_RADII_KPC])   # [pkpc]
ANNULUS_LABELS = [l.replace("ap", "ann") for l in APERTURE_LABELS]

# ── the 3 broad CIGALE regions (2026-08-15) — Parts 7b2/7c/7d/7e/7f/8e ──
# The many-annulus campaign split the flux too thin; the cumulative ladder
# measured curves-of-growth, not places. Fluxes/luminosities/truth masses come
# from DIFFERENCING the cumulative rungs (ap_out − ap_in — exact: the rungs
# are cumulative and filter convolution is linear); SFHs and stellar truth
# come from the projected-radius mask r_in < R <= r_out per sightline.
# core+outskirt are the science; cgm is fitted best-effort and allowed to fail.
REGION_DEFS = {                          # radii in projected pkpc
    "core":     dict(r_in=0.0,                   r_out=APERTURE_RADII_KPC[1],
                     ap_in=None,      ap_out="ap3kpc",   ap_idx=(None, 1)),
    "outskirt": dict(r_in=APERTURE_RADII_KPC[1], r_out=APERTURE_RADII_KPC[3],
                     ap_in="ap3kpc",  ap_out="ap32kpc",  ap_idx=(1, 3)),
    "cgm":      dict(r_in=APERTURE_RADII_KPC[3], r_out=APERTURE_RADII_KPC[4],
                     ap_in="ap32kpc", ap_out="ap100kpc", ap_idx=(3, 4)),
}
REGION_LABELS = list(REGION_DEFS)        # ["core", "outskirt", "cgm"]

# ── viewing angles (MUST match THETA/PHI in the parameter masters) ──
THETA_DEG   = [0, 45, 90, 135]
PHI_DEG     = [0, 90, 180, 270]
N_INCL      = len(THETA_DEG)
INCL_LABELS = [f"i{t:g}p{p:g}" for t, p in zip(THETA_DEG, PHI_DEG)]   # i0p0, i45p90, ...
NHAT        = sightline_unit_vectors(THETA_DEG, PHI_DEG)      # LOS unit vectors

# ── Stage-0 region cutout: EVERYTHING (CGM, satellites) within this proper radius ──
# sphere radius = zoom_box_len = largest aperture (100 kpc): the grid's inscribed sphere
# is fully populated; only the outermost aperture is slightly depth-truncated at its edge
R_CUTOUT_KPC = 100.0

# ── powderday run layout (same conventions as test_powderday.ipynb) ──────────
GVFS_BASE   = ''
# '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE (see test_powderday)
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_m25_quenched'                  # MakeSED appends '.h5'
sed_output_dir = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_quenched_regions')

RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust', paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',  paramf='parameters_master-nodust.py'),
    # dust_on + AGN point sources (BH_SED=True, Hopkins+2007 template, BH_var=False:
    # L_bol = 0.1*BH_Mdot*c^2 from the SIMBA accretion rates). Needs PartType5 in the
    # Stage-0 cutouts -> re-run Part 4 (EXTRACT_OVERWRITE=True) before Stage 1.
    'agn_on':   dict(run_tag='dusty_simdust_agn', paramf='parameters_master-agn.py'),
}
# agn_on stays restricted to the ORIGINAL quenched anchors (2026-08-14 decision):
# every new source (all SF controls + quenched at the new anchors) gets
# dust_on + dust_off only — Part 5 filters on this.
AGN_ARM_SNAPS = {134, 116, 105, 78}

# ── local output tree ─────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR  = os.path.join(OUT, "plots", "powderday_quenched")
CATDIR   = os.path.join(OUT, "sed_aperture_catalogs")
for _d in (SFHDIR, TABLEDIR, PLOTDIR, CATDIR):
    os.makedirs(_d, exist_ok=True)
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")
AV_DUSTY = 0.1     # global A_V above which a galaxy counts as 'dusty' (Parts 4b/7a/7g)

# ── CIGALE tree (Parts 7b-7f) ──
CIGALE_DIR   = os.path.join(CATDIR, "cigale")   # CIGALE input files (Parts 7b/7b2)
# Part 7d writes the 3-region runs under RUN_BASE_REG. The older trees —
# RUN_BASE (pre-2026-08-10 SFH families / Z groups) and cigale_runs_pinned
# (the 2026-08-10 galaxy-pinned cumulative campaign, superseded 2026-08-15 by
# the region runs) — are kept on disk for comparison only; delete
# output/cis25/cigale_runs{,_pinned} manually when no longer needed.
RUN_BASE     = os.path.join(OUT, "cigale_runs")
RUN_BASE_REG = os.path.join(OUT, "cigale_runs_regions")

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

print(f"sim={sim.name}  data_dir={sim.data_dir}")
print(f"prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print("aperture ladder [kpc]:", np.round(APERTURE_RADII_KPC, 2))
print("extracted rungs:", {l: f"{APERTURE_RADII_KPC[i]:.3g} kpc (idx {i})"
                           for l, i in zip(APERTURE_LABELS, WANTED_AP_IDX)})
print("regions:", {r: f"{d['r_in']:.3g}-{d['r_out']:.3g} kpc"
                   for r, d in REGION_DEFS.items()})
print("sightlines:", INCL_LABELS, "  region cutout:", R_CUTOUT_KPC, "pkpc")
print("SED output:", sed_output_dir)

# Part 0b — shared helpers

Small loaders used by several parts, so each part stays runnable in a fresh session after
Parts 0/0b: the selection catalog, `(snap, gal_id)`-keyed alignment, the RT-grid centres
(Stage-1 selection HDF5, caesar fallback — identical values), the Stage-0 cutout reader
(code units → proper kpc about a given centre), anchor-epoch (row 0) history values, and
the Part 7a dusty flag.


In [ ]:
# ── shared helpers: selection, alignment, centres, cutouts, row-0 histories ──
def load_selection():
    """SELECTION_FITS (written by Part 3) -> (table, snap array, gal_id array)."""
    sel = Table.read(SELECTION_FITS)
    return sel, np.asarray(sel["snap"], int), np.asarray(sel["gal_id"], int)

def by_snap_gal(db, snaps, gids, default=np.nan):
    """Align a {(snap, gal_id): value} dict to (snaps, gids) rows -> array."""
    return np.array([db.get((int(s), int(g)), default)
                     for s, g in zip(snaps, gids)])

def rt_centers(snaps, gids):
    """RT-grid centres (code units): Stage-1 selection h5, else caesar (identical values)."""
    from simbanator.sed.makesed import read_selection_centers
    selh5 = os.path.join(sed_output_dir, RUNS["dust_on"]["run_tag"],
                         "target_selection", selection_file + ".h5")
    cen = read_selection_centers(selh5)
    if cen:
        print(f"grid centres from the Stage-1 selection h5 ({len(cen)} galaxies)")
        return cen
    print(f"[fallback] {selh5} missing -> reading centres from the caesar catalogs")
    for s in np.unique(snaps):
        cs = sim.load_catalog(snap=int(s))
        for g in np.unique(np.asarray(gids)[np.asarray(snaps) == s]):
            cen[(int(s), int(g))] = cs.galaxies[int(g)].pos.in_units("code_length").value
        del cs
        gc.collect()
    return cen

def cutout_file(snap, gid):
    """Path of one Stage-0 per-galaxy particle cutout."""
    return os.path.join(hydro_dir_base, f"snap_{int(snap):03d}",
                        f"{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")

def read_cutout(snap, gid, center_code, ptype, fields=()):
    """One Stage-0 cutout particle type -> dict(pos [proper kpc], a, h, <fields> raw).

    `center_code` (code units, ckpc/h) is subtracted before the a/h conversion,
    so `pos` is proper kpc about that centre. Requested `fields` are returned
    raw (code units); a field absent from the file comes back as None.
    Returns None if the centre, the cutout file or the particle type is missing.
    """
    pf = cutout_file(snap, gid)
    if center_code is None or not os.path.exists(pf):
        return None
    with h5py.File(pf, "r") as f:
        if ptype not in f:
            return None
        a  = float(f["Header"].attrs["Time"])
        hh = float(f["Header"].attrs["HubbleParam"])
        out = dict(a=a, h=hh,
                   pos=(np.asarray(f[f"{ptype}/Coordinates"][:], float)
                        - center_code) * a / hh)
        for fld in fields:
            out[fld] = np.asarray(f[f"{ptype}/{fld}"][:]) if fld in f[ptype] else None
    return out

def anchor_row0(keys):
    """Anchor-epoch (row 0) values from every history under SFHDIR.

    -> {(anchor_snap, gal_id): {key: value, 'z0': anchor redshift}};
    keys absent from a history are simply missing from its dicts.
    """
    db = {}
    for hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
        with h5py.File(hf, "r") as f:
            snap0 = int(f["metadata/snapshots"][0])
            gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
            z0    = float(f["redshift/Redshift"][0])
            row0  = {k: f[f"properties/{k}"][0] for k in keys
                     if f"properties/{k}" in f}
        for j, g in enumerate(gid):
            db[(snap0, int(g))] = {k: float(v[j]) for k, v in row0.items()}
            db[(snap0, int(g))]["z0"] = z0
    return db

def row0_arr(db, snaps, gids, key):
    """anchor_row0 value `key` aligned to (snaps, gids) rows (NaN where absent)."""
    return np.array([db.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(snaps, gids)])

def dusty_flags(snaps, gids):
    """Part 7a global A_V aligned to (snaps, gids) -> (A_V array, dusty flag array).

    dusty: 1 (A_V > AV_DUSTY), 0 (transparent), -1 (not measured yet — run
    Part 7a, then re-run the caller to get the dusty/non-dusty split).

    This is Part 7a's ONE-SIGHTLINE global A_V (ATTEN_INCL = INCL_LABELS[0], the
    fiducial aperture) — a coarse per-galaxy label for splitting figures. Part 7f
    builds its own A_V per (aperture, sightline) from the same catalogs; do not
    confuse the two.
    """
    avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
    db = {}
    if os.path.exists(avf):
        at = Table.read(avf)
        db = {(int(s), int(g)): float(a)
              for s, g, a in zip(at["snap"], at["gal_id"], at["A_V"])}
    else:
        print(f"[dusty split] {os.path.basename(avf)} not found — run Part 7a first")
    av = by_snap_gal(db, snaps, gids)
    return av, np.where(np.isnan(av), -1, (av > AV_DUSTY).astype(int))


# Part 1 — Anchors & gated cluster builds

Each anchor (z ≈ 0.3, 0.6, 0.7, 1.0, 2.0 → nearest snapshot) gets its **own** progenitor table +
property history with that snapshot as row 0, and a BH history aligned to the same rows — exactly
the `quench_mode_vs_sigma_gas.ipynb` machinery, pointed at `cis25`. Histories are pre-selected to
**massive + passive** at the anchor (the gas/star floors are applied later so the statistics can
count them).

In [ ]:
# ── anchor table: snapshot, track end, per-anchor product paths ──
_sall, _zall = [], []
for _s in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                        z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                        bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

In [ ]:
# ── property list tracked per anchor (superset of what selection + coupling need) ──
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "pos", "ngas", "nstar", "ages.mass_weighted",
    ],
    "halo_data": ["masses.total"],
}

# ── GATED (cluster): per-anchor progenitor table + property history ──
# Verbatim port of quench_mode_vs_sigma_gas 1z·build, with the (stricter) M*>10 pre-selection.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) > MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+passive at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:   # drop-and-retry: some catalog versions miss some fields
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── loaders (verbatim from quench_mode_vs_sigma_gas): row 0 = the anchor epoch ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

In [ ]:
# ── BH history: per-anchor build (GATED) + loader (verbatim quench_mode §4b) ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

if BUILD_BH:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["bh_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['bh_path'])}"); continue
        if not os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] no history yet -> run BUILD_MULTI_Z first"); continue
        _H = load_anchor_history(A)
        build_bh_for_anchor(A, _H["galaxy_ids"], _H["snaps_arr"], len(_H["galaxy_ids"]))
        del _H; gc.collect()
else:
    print("BUILD_BH=False -> expecting per-anchor BH histories under", SFHDIR)

# Part 2 — Selection, quench events (SFT/QT) & the weak/strong AGN split

- **Selection** (at row 0 = the anchor): `log10 M* > 10`, passive (`sSFR < 0.2/t_H`), `ngas > 20`,
  `nstar ≥ 20`.
- **SFT/QT** per galaxy from `find_quenching_times` on the tracked sSFR history (SFT = crossing
  below 1/t, QT = subsequent crossing below 0.2/t with persistence); the **last** event is kept.
- **AGN split**: `xstr_quench` = mean of `xcoup_hist` (jet strength `clip(log10(0.2/f_Edd),0,1)`
  for `log M_BH > 7.5`, gated by `f_gas < 0.2`) over snapshots with `t_SFT ≤ t ≤ t_QT`; if the
  window is narrower than the snapshot spacing, the finite snapshot nearest SFT is used.
  **strong = fully coupled** (`xstr_quench` = 1 through the window; plain terciles degenerate
  into this tie-clump — 15–20 gals per anchor sit exactly at 1); **weak** = bottom tercile of the
  non-saturated remainder (per anchor); `intermediate` = the rest; no finite coupling = `no_AGN`;
  no detected quench event = `no_event`.

In [ ]:
# ── selection mask at the anchor epoch (row 0) ──
def selection_mask(P, t_cosmic_yr):
    mstar0 = P["masses.stellar"][0]
    sfr0   = P["sfr"][0]
    ngas0  = P["ngas"][0]
    nstar0 = P["nstar"][0] if "nstar" in P else np.full_like(mstar0, np.inf)
    _has_dh2 = ("masses.dust" in P) and ("masses.H2" in P)
    if not _has_dh2:
        print("[selection_mask] WARNING: masses.dust/masses.H2 missing from history -> dust/H2 cut skipped")
    mdust0 = P["masses.dust"][0] if _has_dh2 else None
    mh2_0  = P["masses.H2"][0]   if _has_dh2 else None
    with np.errstate(all="ignore"):
        ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
        cuts = {
            "massive":  np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
            "passive":  ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr[0]),
            "gas>20":   ngas0 >= NGAS_MIN,
            "star>=20": nstar0 >= NSTAR_MIN,
            # multiplicative form so M_H2=0 rows pass instead of dividing by zero
            "dust/H2":  (mdust0 >= DUST_TO_H2_MIN * mh2_0) if _has_dh2
                        else np.ones_like(mstar0, dtype=bool),
        }
    m = (cuts["massive"] & cuts["passive"] & cuts["gas>20"] & cuts["star>=20"]
         & cuts["dust/H2"])
    return m, cuts

# ── SFT/QT per selected galaxy (trimmed from quench_mode build_records) ──
def quench_records(P, t_cosmic_yr, redshift, galaxy_ids, cols):
    """One record per selected column; galaxies without a detected quench event keep NaN times."""
    records = []
    for col in np.asarray(cols, int):
        gid = galaxy_ids[col]
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        with np.errstate(all="ignore"):
            ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        rec = dict(gid=int(gid), col=int(col), t_sft=np.nan, t_qt=np.nan,
                   tau_q=np.nan, tau_q_over_tH=np.nan, z_qt=np.nan)
        if valid.sum() >= 5:
            t = t_cosmic_yr[valid]; s = ssfr[valid]
            o = np.argsort(t); t, s = t[o], s[o]
            tu, ui = np.unique(t, return_index=True); su = s[ui]
            if len(tu) >= 5:
                qts, sfts, _, dbg = find_quenching_times(
                    tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts))                     # last (surviving) quench event
                    rec["t_qt"], rec["t_sft"] = float(qts[k]), float(sfts[k])
                    rec["tau_q"] = rec["t_qt"] - rec["t_sft"]
                    z_qt = float(np.interp(rec["t_qt"], t_cosmic_yr[::-1], redshift[::-1]))
                    rec["z_qt"] = z_qt
                    rec["tau_q_over_tH"] = rec["tau_q"] / (COSMO.age(z_qt).value * 1e9)
        records.append(rec)
    return records

In [ ]:
# ── AGN–ISM coupling over the quench window [SFT, QT] (physics verbatim from §8j build_coupling) ──
def coupling_quench_window(BH, P, records, t_cosmic_yr):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok  = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xstr_q = np.full(n, np.nan)
    for i, r in enumerate(records):
        if not (np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"])):
            continue                                   # no quench event -> stays NaN ('no_event')
        cs = xcoup_hist[_ord, r["col"]].astype(float)
        fin = np.isfinite(cs)
        win = (t_inc >= r["t_sft"]) & (t_inc <= r["t_qt"]) & fin
        if not win.any() and fin.any():
            # quench window narrower than the snapshot spacing -> nearest finite snapshot to SFT
            j = np.where(fin)[0]
            win = np.zeros_like(fin); win[j[np.argmin(np.abs(t_inc[j] - r["t_sft"]))]] = True
        if win.any():
            xstr_q[i] = np.nanmean(cs[win])
    bx = np.isfinite(xstr_q)
    # Physical classes (2026-08-03): xstr_quench piles up at exactly 1.0 (fully
    # coupled through the whole quench window; 15-20 gals per anchor), so plain
    # terciles degenerate into the tie-clump at 1. Instead: strong = the fully
    # coupled clump, weak = bottom tercile of the non-saturated remainder,
    # intermediate = the rest. xstr_quench stays continuous in the catalogs.
    XSTR_FULL_EPS = 1e-6
    strong = bx & (xstr_q >= 1.0 - XSTR_FULL_EPS)
    weak = np.zeros(n, bool); lo_q = np.nan; hi_q = 1.0
    _rest = bx & ~strong
    if _rest.sum() >= 3:
        lo_q = float(np.nanquantile(xstr_q[_rest], 1.0 / 3.0))
        weak = _rest & (xstr_q <= lo_q)
    inter = bx & ~strong & ~weak
    no_fb = ~bx
    return dict(xstr_quench=xstr_q, strong=strong, weak=weak, inter=inter, no_fb=no_fb,
                tercile=(lo_q, hi_q))

def agn_class_labels(CO, records):
    """Per-record string label; galaxies without a quench event are 'no_event'."""
    n = len(records)
    has_event = np.array([np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]) for r in records])
    lab = np.array(["unclassified"] * n, dtype=object)
    if CO is not None:
        lab[CO["no_fb"]] = "no_AGN"
        lab[CO["inter"]] = "intermediate"
        lab[CO["weak"]]  = "weak"
        lab[CO["strong"]] = "strong"
    lab[~has_event] = "no_event"
    return lab

In [ ]:
# ── driver: per anchor -> selection, records, coupling, labels (+ SF control) ──
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] MISSING history -> run BUILD_MULTI_Z on the cluster; skipped")
        continue
    H = load_anchor_history(A)
    m, cuts = selection_mask(H["P"], H["t_cosmic_yr"])
    cols = np.where(m)[0]
    recs = quench_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"], cols)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    CO = coupling_quench_window(BH, H["P"], recs, H["t_cosmic_yr"]) if BH is not None else None
    labels = agn_class_labels(CO, recs)
    if BH is None:
        print(f"[{A['tag']}] WARNING: no BH history -> AGN split = 'unclassified' (run BUILD_BH)")
    RESULTS[_zt] = dict(A=A, H=H, mask=m, cuts=cuts, cols=cols, records=recs, CO=CO, labels=labels)
    n_ev = int(np.isfinite([r["t_qt"] for r in recs]).sum())
    print(f"[{A['tag']}] snap {A['snap']} (z={A['z']:.3f}): pool={m.size} "
          f"selected={len(cols)} with_event={n_ev} "
          f"classes={dict(zip(*np.unique(labels, return_counts=True))) if len(labels) else {}}")

# ── star-forming control: mass-matched at the anchor epoch (2026-08-14) ──
# SF galaxies have NO histories (BUILD_MULTI_Z pre-selects massive+passive), so
# they are read straight from the caesar Groups file (GroupID == row index).
# Same floor/ngas/nstar/dust cuts, sSFR >= PASSIVE_FACTOR/t_H at the anchor;
# SF_MATCH_RATIO nearest-log M* partners per quenched galaxy, drawn without
# replacement, most massive quenched galaxy picks first (massive SF partners
# are the scarce end of the pool).
def sf_control_for_anchor(A, q_gids, q_logm):
    with h5py.File(sim.get_caesar_file(int(A["snap"])), "r") as _f:
        _d = _f["galaxy_data"]
        _ms, _sf = _d["dicts/masses.stellar"][:], _d["sfr"][:]
        _ng, _ns = _d["ngas"][:], _d["nstar"][:]
        _md, _mh2 = _d["dicts/masses.dust"][:], _d["dicts/masses.H2"][:]
    _tH = COSMO.age(float(A["z"])).value * 1e9
    with np.errstate(all="ignore"):
        _lm = np.log10(np.where(_ms > 0, _ms, np.nan))
        _ss = np.where(_ms > 0, _sf / _ms, np.nan)
        _ok = ((_lm > MASS_FLOOR) & (_ss >= PASSIVE_FACTOR / _tH) & (_ng >= NGAS_MIN)
               & (_ns >= NSTAR_MIN) & (_md >= DUST_TO_H2_MIN * _mh2))
    _cand = np.where(_ok & ~np.isin(np.arange(len(_ms)), np.asarray(q_gids, int)))[0]
    _rows, _taken = [], set()
    for _qi in np.argsort(np.asarray(q_logm, float))[::-1]:
        if not np.isfinite(q_logm[_qi]):
            continue
        for _ in range(SF_MATCH_RATIO):
            _free = np.array([c for c in _cand if c not in _taken], int)
            if not _free.size:
                break
            _c = int(_free[np.argmin(np.abs(_lm[_free] - q_logm[_qi]))])
            _taken.add(_c)
            _rows.append(dict(gal_id=_c, log_mstar=float(_lm[_c]), ssfr=float(_ss[_c]),
                              ngas=int(_ng[_c]), nstar=int(_ns[_c]),
                              mdust=float(_md[_c]), mh2=float(_mh2[_c]),
                              match_gal_id=int(q_gids[_qi]),
                              dlogm=float(_lm[_c] - q_logm[_qi])))
    return _rows, int(_ok.sum())

for _zt, R in RESULTS.items():
    A, H = R["A"], R["H"]
    _qg = np.asarray([r["gid"] for r in R["records"]], int)
    _qm = np.array([np.log10(H["P"]["masses.stellar"][0, r["col"]])
                    if H["P"]["masses.stellar"][0, r["col"]] > 0 else np.nan
                    for r in R["records"]])
    R["sf_rows"], R["sf_pool"] = sf_control_for_anchor(A, _qg, _qm)
    if R["sf_rows"]:
        _dl = np.abs([r["dlogm"] for r in R["sf_rows"]])
        print(f"[{A['tag']}] SF control: {len(R['sf_rows'])} matched of "
              f"{R['sf_pool']} candidates "
              f"(|dlogM*| med {np.median(_dl):.2f}, max {np.max(_dl):.2f} dex)")
    else:
        print(f"[{A['tag']}] SF control: none matched ({R['sf_pool']} candidates)")


# Part 3 — Sample statistics & the selection catalog

How many galaxies survive each cut per snapshot, how many have gas at all, and how the AGN classes
populate. **Note:** the pool is the history's build-time pre-selection (massive + passive at the
anchor), not the full galaxy catalog — the funnel starts there. Also writes the per-galaxy
selection table (`powderday_quenched_selection.fits`) that Stages 0–2 read, so the RT stages never
depend on this session's memory.

In [ ]:
# ── funnel table + per-galaxy selection FITS (Q + mass-matched SF control) ──
_rows, _sel_rows = [], []
for _zt, R in RESULTS.items():
    A, H, cuts = R["A"], R["H"], R["cuts"]
    ngas0 = H["P"]["ngas"][0]
    n_pool = int(np.isfinite(H["P"]["masses.stellar"][0]).sum())
    lab = R["labels"]
    _rows.append(dict(
        z_target=_zt, snap=A["snap"], z_snap=round(A["z"], 4),
        pool_massive_passive=n_pool,
        with_any_gas=int((ngas0 > 0).sum()),
        gas_gt20=int(cuts["gas>20"].sum()),
        massive=int(cuts["massive"].sum()),
        passive=int(cuts["passive"].sum()),
        star_ge20=int(cuts["star>=20"].sum()),
        dust_h2_ok=int(cuts["dust/H2"].sum()),
        selected=len(R["cols"]),
        with_event=int(np.isfinite([r["t_qt"] for r in R["records"]]).sum()),
        strong=int((lab == "strong").sum()), weak=int((lab == "weak").sum()),
        intermediate=int((lab == "intermediate").sum()), no_AGN=int((lab == "no_AGN").sum()),
        no_event=int((lab == "no_event").sum()), unclassified=int((lab == "unclassified").sum()),
        sf_pool=int(R.get("sf_pool", 0)), sf_matched=len(R.get("sf_rows", [])),
    ))
    # per-galaxy rows — quenched
    P0 = H["P"]
    for i, (r, l) in enumerate(zip(R["records"], lab)):
        c = r["col"]
        with np.errstate(all="ignore"):
            _ms = float(P0["masses.stellar"][0, c])
            _sf = float(P0["sfr"][0, c])
            _md  = float(P0["masses.dust"][0, c]) if "masses.dust" in P0 else np.nan
            _mh2 = float(P0["masses.H2"][0, c])   if "masses.H2"   in P0 else np.nan
            xs = R["CO"]["xstr_quench"][i] if R["CO"] is not None else np.nan
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gid"]), pop="Q",
            log_mstar=float(np.log10(_ms)) if _ms > 0 else np.nan,
            ssfr=float(_sf / _ms) if _ms > 0 else np.nan,
            ngas=int(P0["ngas"][0, c]), nstar=int(P0["nstar"][0, c]) if "nstar" in P0 else -1,
            t_sft=r["t_sft"], t_qt=r["t_qt"], tau_q=r["tau_q"],
            tau_q_over_tH=r["tau_q_over_tH"], z_qt=r["z_qt"],
            xstr_quench=float(xs), agn_class=str(l),
            mdust=_md, mh2=_mh2,
            dust_to_h2=(_md / _mh2) if (np.isfinite(_md) and np.isfinite(_mh2) and _mh2 > 0) else np.nan,
            match_gal_id=-1, dlogm_match=np.nan,
        ))
    # per-galaxy rows — star-forming control (no histories -> no quench columns;
    # match_gal_id links each SF row to its quenched partner at the same anchor)
    for r in R.get("sf_rows", []):
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gal_id"]), pop="SF",
            log_mstar=r["log_mstar"], ssfr=r["ssfr"],
            ngas=r["ngas"], nstar=r["nstar"],
            t_sft=np.nan, t_qt=np.nan, tau_q=np.nan,
            tau_q_over_tH=np.nan, z_qt=np.nan,
            xstr_quench=np.nan, agn_class="star_forming",
            mdust=r["mdust"], mh2=r["mh2"],
            dust_to_h2=(r["mdust"] / r["mh2"]) if r["mh2"] > 0 else np.nan,
            match_gal_id=int(r["match_gal_id"]), dlogm_match=r["dlogm"],
        ))

STATS = Table(_rows)
STATS.write(os.path.join(TABLEDIR, "powderday_quenched_stats.fits"), overwrite=True)
STATS.pprint(max_width=-1)

SEL = Table(_sel_rows)
SEL.write(SELECTION_FITS, overwrite=True)
_popc = np.asarray(SEL["pop"], str)
print(f"\nselection table: {len(SEL)} sources ({int((_popc == 'Q').sum())} Q + "
      f"{int((_popc == 'SF').sum())} SF) over {len(np.unique(SEL['snap']))} snapshots "
      f"-> {SELECTION_FITS}")


In [ ]:
# ── figures: selection funnel + gas-particle content + AGN classes ──
_zs   = list(RESULTS.keys())
_tags = [RESULTS[z]["A"]["tag"] for z in _zs]

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

# funnel per anchor
_steps = ["pool_massive_passive", "with_any_gas", "gas_gt20", "selected", "with_event", "sf_matched"]
_slbl  = ["massive+passive", "any gas", "gas>20", "all cuts", "SFT/QT found", "SF matched"]
_x = np.arange(len(_zs)); _w = 0.13
for j, (st, sl) in enumerate(zip(_steps, _slbl)):
    axes[0].bar(_x + (j - 2.5) * _w, [STATS[st][i] for i in range(len(STATS))], width=_w, label=sl)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_tags)
axes[0].set_ylabel("N galaxies"); axes[0].set_title("selection funnel")
axes[0].legend(fontsize=10, frameon=False)

# gas-particle histograms (pool), with the >20 floor
for z in _zs:
    ng = RESULTS[z]["H"]["P"]["ngas"][0]
    ng = ng[np.isfinite(ng) & (ng > 0)]
    if ng.size:
        axes[1].hist(np.log10(ng), bins=25, histtype="step", lw=2, label=RESULTS[z]["A"]["tag"])
axes[1].axvline(np.log10(NGAS_MIN), color="k", ls=":", label=f"ngas={NGAS_MIN}")
axes[1].set_xlabel("log10 ngas (anchor)"); axes[1].set_ylabel("N")
axes[1].set_title("pool gas content"); axes[1].legend(fontsize=10, frameon=False)

# AGN classes among the selected
_classes = ["strong", "intermediate", "weak", "no_AGN", "no_event", "unclassified"]
_bot = np.zeros(len(_zs))
for cl in _classes:
    v = np.array([STATS[cl][i] for i in range(len(STATS))], float)
    axes[2].bar(_x, v, bottom=_bot, label=cl)
    _bot += v
axes[2].set_xticks(_x); axes[2].set_xticklabels(_tags)
axes[2].set_ylabel("N selected"); axes[2].set_title("AGN-coupling classes")
axes[2].legend(fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sample_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

# Part 3b — mass–size QC: flag sources too large (or too small) for the apertures

Fixed **physical** apertures implicitly assume every source has a similar size — the mass–size
relation is the check on that. Anchor-epoch CAESAR radii come from the histories (row 0;
`radii.*` are **comoving kpc** — verified `unit: 'kpccm'` in the m25n512 catalogs — converted
with $1/(1+z)$). CAESAR $R_{50}$ is the 3D half-**mass** radius; the van der Wel+2014 quiescent
relations (projected half-light $R_e$) are drawn for context only.

Flags (written back into `SELECTION_FITS`; Part 7 carries them into every catalog):

- **`flag_too_large`** — `SIZE_FACTOR·R50 > R_CUTOUT_KPC`: the 100 pkpc cutout/grid truncates
  the stellar envelope → the "≈ total" 100 kpc aperture (and any CIGALE mass) biases low;
- **`flag_unresolved`** — `R50 < N_EPS_MIN·ε` (softening): the size is not trusted and the
  1 kpc "central" aperture is not meaningfully sub-galactic.

Nothing is dropped — the flags are one boolean away in any downstream cut. The per-rung print
shows for how much of the sample each aperture is sub-galactic (< R50) vs effectively total
(> 3 R50).


In [ ]:
# ── Part 3b — mass-size QC: flag too-large / unresolved sources for the aperture ladder ──
# Self-contained after Parts 0/0b: reads SELECTION_FITS + the anchor histories in SFHDIR.
SIZE_FACTOR    = 5.0     # envelope proxy: SIZE_FACTOR*R50 beyond the cutout -> truncated
N_EPS_MIN      = 2.0     # resolved if R50 >= N_EPS_MIN * softening
EPS_MIN_CKPC_H = 0.25    # m25n512 minimum gravitational softening [comoving kpc/h]
SIMBA_H        = 0.68

SEL, SNAPS, IDS = load_selection()
# radii straight from the Groups files: the histories only hold the massive+
# passive pool, so SF control rows would come back NaN there. GroupID == row
# index; radii are comoving kpc (no h) -> proper with each anchor's own z.
_r50s = np.full(len(SEL), np.nan)
_r50g = np.full(len(SEL), np.nan)
for _sn3b in np.unique(SNAPS):
    try:
        with h5py.File(sim.get_caesar_file(int(_sn3b)), "r") as _f3b:
            _rs3b = _f3b["galaxy_data/dicts/radii.stellar_half_mass"][:]
            _rg3b = _f3b["galaxy_data/dicts/radii.gas_half_mass"][:]
            _zc3b = float(_f3b["simulation_attributes"].attrs["redshift"])
    except (OSError, KeyError) as _e3b:
        print(f"[3b] snap {_sn3b}: {_e3b} -> R50 stays NaN")
        continue
    _m3b = np.where(SNAPS == _sn3b)[0]
    _r50s[_m3b] = _rs3b[IDS[_m3b]] / (1.0 + _zc3b)
    _r50g[_m3b] = _rg3b[IDS[_m3b]] / (1.0 + _zc3b)

_zsel = np.asarray(SEL["z_snap"], float)
_eps_pkpc = EPS_MIN_CKPC_H / SIMBA_H / (1.0 + _zsel)           # softening, proper kpc
flag_too_large  = SIZE_FACTOR * _r50s > R_CUTOUT_KPC
flag_unresolved = _r50s < N_EPS_MIN * _eps_pkpc

SEL["r50_star_kpc"]    = _r50s
SEL["r50_gas_kpc"]     = _r50g
SEL["flag_too_large"]  = flag_too_large.astype(int)
SEL["flag_unresolved"] = flag_unresolved.astype(int)
SEL.write(SELECTION_FITS, overwrite=True)

_nok = int(np.isfinite(_r50s).sum())
print(f"R50 matched: {_nok}/{len(SEL)} | median R50 = {np.nanmedian(_r50s):.2f} pkpc | "
      f"too large (R50 > {R_CUTOUT_KPC/SIZE_FACTOR:.0f} kpc): {int(flag_too_large.sum())} | "
      f"unresolved (R50 < {N_EPS_MIN:g} eps): {int(flag_unresolved.sum())}")
print("aperture rung vs the sample sizes:")
_rungs = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]
for _t, _rr in zip(APERTURE_LABELS, _rungs):
    _sub = 100 * np.nanmean(_rr < _r50s); _tot = 100 * np.nanmean(_rr > 3 * _r50s)
    print(f"  {_t:>9s} ({_rr:6.2f} kpc): sub-galactic (<R50) for {_sub:4.0f}%  |  "
          f"~total (>3 R50) for {_tot:4.0f}%")
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _why = "TOO LARGE" if flag_too_large[_k] else "unresolved"
    print(f"  [{_why}] snap {int(SEL['snap'][_k])} gal {int(SEL['gal_id'][_k])}: "
          f"R50={_r50s[_k]:.2f} pkpc, logM*={float(SEL['log_mstar'][_k]):.2f}, "
          f"z={_zsel[_k]:.2f}")

# ── mass-size relation vs the aperture ladder ──
_fig, _ax = plt.subplots(figsize=(11, 7.5))
_zt_colors = plt.cm.viridis(np.linspace(0, 0.9, len(TARGET_REDSHIFTS)))
# van der Wel+2014 Table 5, early types: R_e = A*(M*/5e10)^alpha [kpc] (context only)
_VDW = {0.25: (10**0.60, 0.75), 0.75: (10**0.42, 0.71), 1.25: (10**0.22, 0.76),
        1.75: (10**0.09, 0.76), 2.25: (10**-0.05, 0.79)}
_lm = np.asarray(SEL["log_mstar"], float)
_xmax = max(11.4, np.nanmax(_lm) + 0.15)
_mm = np.logspace(10, _xmax, 40)
for _c, _zt in zip(_zt_colors, TARGET_REDSHIFTS):
    _m = np.isclose(np.asarray(SEL["z_target"], float), _zt)
    if not _m.any():
        continue
    _ax.scatter(_lm[_m], _r50s[_m], s=22, color=_c, label=f"z\u2248{_zt:g}", zorder=3)
    _A, _al = _VDW[min(_VDW, key=lambda z: abs(z - _zt))]
    _ax.plot(np.log10(_mm), _A * (_mm / 5e10) ** _al, "--", color=_c, lw=1.1, alpha=0.7)
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _ax.scatter([_lm[_k]], [_r50s[_k]], s=95, facecolor="none",
                edgecolor="crimson", lw=1.4, zorder=4)
for _rr, _t in zip(_rungs, APERTURE_LABELS):
    _ax.axhline(_rr, color="0.78", lw=0.7, zorder=1)
    _ax.text(_xmax - 0.03, _rr * 1.04, _t, fontsize=7, va="bottom", ha="right", color="0.45")
_ax.axhline(R_CUTOUT_KPC / SIZE_FACTOR, color="crimson", ls=":", lw=1.3)
_ax.text(10.02, R_CUTOUT_KPC / SIZE_FACTOR * 1.05,
         f"too large ({SIZE_FACTOR:g}\u00b7R50 > {R_CUTOUT_KPC:.0f} kpc cutout)",
         fontsize=8, color="crimson", va="bottom")
_ax.set_yscale("log"); _ax.set_xlim(9.98, _xmax)
_ax.set_xlabel(r"$\log_{10}\,M_*/M_\odot$")
_ax.set_ylabel(r"stellar $R_{50}$ [proper kpc]")
_ax.set_title("mass\u2013size QC")
_ax.plot([], [], "--", color="0.5", lw=1.1, label="vdW+14 quiescent $R_e$")
_ax.legend(fontsize=10, frameon=False, loc="lower right")
plt.savefig(os.path.join(PLOTDIR, "mass_size_aperture_qc.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4 — Stage 0: extract the per-galaxy particle files (cluster)

One HDF5 per galaxy (gas + stars + BHs; gas keeps `Dust_Masses`, BHs carry
`BH_Mass`/`BH_Mdot` for the `agn_on` run) under
`hydro_dir_base/snap_NNN/<PREFIX>_snap<NNN>_gal<ID>.h5` — now a **100 pkpc spherical region
cutout** around each galaxy centre (CGM + satellites included; periodic-wrap safe), NOT just
the caesar member particles. Identical for all runs (`dust_on` / `dust_off` / `agn_on` — the dust treatment and the
AGN sources live in the parameter masters). Cutouts extracted before 2026-07-28 have no
`PartType5` group — re-run this part (`EXTRACT_OVERWRITE = True`) before launching `agn_on`. Reads `SELECTION_FITS`, so it can run in a fresh session once Part 3
has been executed.

⚠ Region files reuse the plist filenames, so `EXTRACT_OVERWRITE = True` below **replaces** any
old galaxy-member-only files — intended, since mixed hydro inputs would corrupt the sample.

In [ ]:
from simbanator.analysis import extract_particles

SEL, SNAPS, IDS = load_selection()
print(f"{len(SNAPS)} sources over snapshots {sorted(set(SNAPS.tolist()))}")

EXTRACT_OVERWRITE = False   # 2026-08-14: existing cutouts are already region-mode with
                            # PartType5 -> skip them; True only if the cutout FORMAT changes
EXTRACT_PTYPES    = ("PartType0", "PartType4", "PartType5")   # gas (Dust_Masses) + stars + BHs (agn_on)

bad_snaps = []
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, radius=R_CUTOUT_KPC,
                          ptypes=EXTRACT_PTYPES, sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here)))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    print(f"{len(bad_snaps)} snapshot(s) unreadable: {bad_snaps} — re-stage those files and re-run.")

# Part 4b — annulus sampling QC: particle counts per projected annulus

How well can powderday sample an **annular** SED? Each Hyperion aperture is a circle in the
**image plane**, so for every sightline the star (emission sources) and gas (dust carriers)
particles of each 100 pkpc cutout are projected perpendicular to the viewing direction and
counted in the annuli between consecutive rungs (`ann1kpc` = the 0→1 kpc disc, then 1→3.16,
3.16→10, 10→31.6, 31.6→100 kpc; as in Part 7a, the **outer** rung names the annulus).
Counts span the whole LOS depth through the sphere — exactly the geometry the RT sees (the
outermost annulus is depth-truncated like its aperture). Centres are the **exact RT grid
centres** (`code_coods` from the Stage-1 selection HDF5; caesar fallback if Part 5 has not
run yet).

Reading the numbers:

- **stars = intrinsic emitters.** `nstar = 0` → the annular flux is scattered/re-emitted
  light only and a CIGALE fit of it is meaningless; a few tens of star particles → the
  annular SED is shot-noise dominated (a handful of SSP ages/metallicities).
- **gas → dust grid.** Gas is smoothed onto the octree, so counts are indicative; `ndust`
  (gas with `Dust_Masses > 0`) counts the particles actually carrying dust.

One row per (galaxy, sightline) → `tables/annulus_particle_counts.fits`
(`nstar_/ngas_/ndust_/ntot_<annulus>` + `A_V_glob`/`dusty`); Part 7a's radial profile uses it to flag
star-free annular catalogs.

**Dusty vs non-dusty split.** If Part 7a's `attenuation_vs_ism.fits` exists, each galaxy is
flagged **dusty** (global $A_V > 0.1$, same threshold as 7a Fig 3, fiducial aperture +
sightline) or non-dusty, the figure highlights the two subsamples (red vs gray lines, separate
medians) and the summary prints their per-annulus median counts — the dusty galaxies are the
ones whose annular attenuation/CIGALE fits matter, so their sampling is the QC that counts.
Part 7a needs the RT fluxes, so on a fresh pipeline this cell first runs without the split
(`dusty = -1`) — **re-run it after Part 7a** to get the highlighted version.

In [ ]:
# ── Part 4b — annulus sampling QC: star/gas counts per projected annulus & sightline ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (Part 4).
_R_MID = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c  = _centers.get((int(_s), int(_g)))
    _st = read_cutout(_s, _g, _c, "PartType4")
    _gs = read_cutout(_s, _g, _c, "PartType0", fields=("Dust_Masses",))
    if _st is None or _gs is None:
        _skipped.append((int(_s), int(_g)))
        continue
    _dm = (_gs["Dust_Masses"] if _gs["Dust_Masses"] is not None
           else np.zeros(len(_gs["pos"])))
    _d = {"star": _st["pos"], "gas": _gs["pos"],
          "dust": _gs["pos"][np.asarray(_dm) > 0]}                # dust-carrying gas
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _pt, _pos in _d.items():                              # projected radius wrt LOS
            _cnt, _ = np.histogram(projected_radius(_pos, NHAT[_j]), R_EDGES)
            for _k, _al in enumerate(ANNULUS_LABELS):
                _row[f"n{_pt}_{_al}"] = int(_cnt[_k])
        for _al in ANNULUS_LABELS:
            _row[f"ntot_{_al}"] = _row[f"nstar_{_al}"] + _row[f"ngas_{_al}"]
        _rows.append(_row)

QC_COUNTS = Table(_rows)
QC_COUNTS.meta["R_EDGES"] = list(np.round(R_EDGES, 3))             # proper kpc

# dusty split from Part 7a (global A_V, fiducial aperture/sightline):
# dusty = 1 (A_V > AV_DUSTY), 0 (transparent), -1 (no A_V yet -> re-run after Part 7a)
_avg, QC_COUNTS["dusty"] = dusty_flags(QC_COUNTS["snap"], QC_COUNTS["gal_id"])
QC_COUNTS["A_V_glob"] = _avg
_have_av = bool(np.isfinite(_avg).any())

_out = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
QC_COUNTS.write(_out, overwrite=True)
print(f"{len(QC_COUNTS)} rows ({len(QC_COUNTS)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
_dg = np.asarray(QC_COUNTS["dusty"], int)[::N_INCL]   # per galaxy (same on all sightlines)
if _have_av:
    print(f"dusty split (Part 7a global A_V > {AV_DUSTY:g}): {int((_dg == 1).sum())} dusty / "
          f"{int((_dg == 0).sum())} non-dusty / {int((_dg == -1).sum())} unmatched galaxies")

# ── summary: how well is each annulus sampled? ──
print(f"\n{'annulus':>10s} {'r [pkpc]':>13s} | {'nstar p16/50/84':>17s} {'=0':>4s} {'<10':>4s} "
      f"{'<100':>5s} | {'ngas p50':>8s} {'=0':>4s} | {'ndust p50':>9s} {'=0':>4s}")
for _k, _al in enumerate(ANNULUS_LABELS):
    _ns = np.asarray(QC_COUNTS[f"nstar_{_al}"], int)
    _ng = np.asarray(QC_COUNTS[f"ngas_{_al}"],  int)
    _nd = np.asarray(QC_COUNTS[f"ndust_{_al}"], int)
    _p  = np.percentile(_ns, [16, 50, 84]).astype(int)
    print(f"{_al:>10s} {R_EDGES[_k]:5.1f}-{R_EDGES[_k+1]:6.1f} | "
          f"{_p[0]:5d}/{_p[1]:5d}/{_p[2]:5d} {np.mean(_ns == 0)*100:3.0f}% {np.mean(_ns < 10)*100:3.0f}% "
          f"{np.mean(_ns < 100)*100:4.0f}% | {int(np.median(_ng)):8d} {np.mean(_ng == 0)*100:3.0f}% | "
          f"{int(np.median(_nd)):9d} {np.mean(_nd == 0)*100:3.0f}%")
_nfree = int(sum((np.asarray(QC_COUNTS[f"nstar_{_al}"], int) == 0).sum()
                 for _al in ANNULUS_LABELS))
print(f"\nstar-free (annulus, galaxy, sightline) triples: {_nfree} "
      f"/ {len(QC_COUNTS) * len(ANNULUS_LABELS)} — those annular SEDs have NO intrinsic emitters")

if _have_av:                     # median counts split by the Part 7a dusty flag
    _dm_all = np.asarray(QC_COUNTS["dusty"], int)
    print(f"\nmedian counts, dusty (D, n={int((_dg == 1).sum())} gals) "
          f"vs non-dusty (N, n={int((_dg == 0).sum())}):")
    print(f"{'annulus':>10s} | {'nstar D':>8s} {'nstar N':>8s} | {'ngas D':>8s} {'ngas N':>8s} "
          f"| {'ndust D':>8s} {'ndust N':>8s}")
    for _al in ANNULUS_LABELS:
        _vals = []
        for _cc in ("nstar", "ngas", "ndust"):
            _v = np.asarray(QC_COUNTS[f"{_cc}_{_al}"], int)
            for _dd in (1, 0):
                _m = _dm_all == _dd
                _vals.append(int(np.median(_v[_m])) if _m.any() else -1)
        print(f"{_al:>10s} | {_vals[0]:8d} {_vals[1]:8d} | {_vals[2]:8d} {_vals[3]:8d} "
              f"| {_vals[4]:8d} {_vals[5]:8d}")

# ── figure: count distributions per annulus, dusty vs non-dusty highlighted ──
_dm_all = np.asarray(QC_COUNTS["dusty"], int)
_have_split = _have_av and (_dm_all >= 0).any()
_fig, _axs = plt.subplots(1, 3, figsize=(20, 6.5), sharey=True)
for _ax, _pt, _ttl in zip(_axs, ("star", "gas", "dust"),
                          ("stars", "gas", "dust-carrying gas")):
    _M = np.column_stack([np.asarray(QC_COUNTS[f"n{_pt}_{_al}"], int)
                          for _al in ANNULUS_LABELS])
    if _have_split:                # per-(gal,sightline) lines colored by the Part 7a split
        for _rowv, _dd in zip(_M, _dm_all):
            _ax.plot(_R_MID, _rowv, color={1: "#c0392b", 0: "0.75"}.get(_dd, "0.88"),
                     lw=0.5, alpha=0.45, zorder=1)
        for _dd, _col, _mk, _lab in ((1, "#c0392b", "o-", f"dusty ($A_V>{AV_DUSTY:g}$)"),
                                     (0, "#2980b9", "s--", "non-dusty")):
            _mrows = _dm_all == _dd
            if _mrows.any():
                _ax.plot(_R_MID, np.median(_M[_mrows], axis=0), _mk, color=_col, lw=2,
                         zorder=3, label=f"{_lab} median (n={int(_mrows.sum()) // N_INCL} gals)")
    else:
        for _rowv in _M:                                          # one line per (gal, sightline)
            _ax.plot(_R_MID, _rowv, color="0.75", lw=0.5, alpha=0.5, zorder=1)
        _ax.fill_between(_R_MID, np.percentile(_M, 16, axis=0), np.percentile(_M, 84, axis=0),
                         color="#2980b9", alpha=0.25, zorder=2, label="16–84%")
        _ax.plot(_R_MID, np.median(_M, axis=0), "o-", color="#2980b9", lw=2, zorder=3,
                 label="median")
    for _thr, _ls in ((10, ":"), (100, "--")):
        _ax.axhline(_thr, color="0.3", ls=_ls, lw=0.9)
        _ax.text(_R_MID[0] * 0.9, _thr * 1.15, f"N={_thr}", color="0.3", fontsize=7)
    _ax.set_xscale("log"); _ax.set_yscale("symlog", linthresh=1)
    _ax.set_xticks(_R_MID); _ax.set_xticklabels([l[3:] for l in ANNULUS_LABELS], fontsize=10)
    _ax.set_xlabel("annulus"); _ax.set_title(_ttl)
    _ax.set_ylim(bottom=-0.5)
_axs[0].set_ylabel("particles per projected annulus (full LOS depth)")
_axs[0].legend(fontsize=10, frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "annulus_particle_counts.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4c — SIMBA metallicities per aperture & annulus (CIGALE priors)

Mass-weighted **stellar** and **gas** metallicities (total metal mass fraction,
`Metallicity[:, 0]`) of each cutout, measured in the **same projected geometry as the SEDs**:
per sightline, cumulative within each aperture rung (`ap1kpc…ap100kpc`) and in each annulus
between rungs (`ann3kpc…ann100kpc`; `ann1kpc`≡`ap1kpc`). Same centres/projection as Part 4b.

These are the **metallicity pins for the CIGALE runs** (Part 7d): CIGALE's bc03
`metallicity` and nebular `zgas` are strict grids, so each galaxy's SIMBA value is snapped to
the **nearest allowed grid value in log space** (bc03: 0.0001, 0.0004, 0.004, 0.008, 0.02,
0.05 — verified against the cluster's CIGALE 2025.1 sources) and the catalog is split into
per-metallicity sub-runs: one per bc03 node, each fitted with that single stellar Z and a
`zgas` grid restricted to the group members' snapped values. The summary below shows how
the sample maps onto the bc03 nodes per aperture — since Part 7d pins ONE node per object,
this is also the quantisation floor on the metallicity prior. It reports how the sample
would have split into sub-runs under the old grouped scheme, which Part 7d
create. Empty apertures/annuli (no particles) → NaN → the `Zsfree` group (default Z grid).

One row per (galaxy, sightline) → `tables/aperture_metallicities.fits`
(`Zstar_<label>`, `Zgas_<label>` for the 9 labels).

In [ ]:
# ── Part 4c — mass-weighted Z_star / Z_gas per projected aperture & annulus ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts; same geometry as Part 4b.
from simbanator.sed.cigale import grid_options, nearest_option

Z_LABELS = list(APERTURE_LABELS) + ANNULUS_LABELS[1:]   # cumulative rungs + true annuli

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

def _mwz(z, m, sel):
    # mass-weighted metallicity over a particle selection (NaN if empty)
    if not sel.any():
        return np.nan
    mm = m[sel]
    return float(np.sum(mm * z[sel]) / np.sum(mm)) if mm.sum() > 0 else np.nan

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c = _centers.get((int(_s), int(_g)))
    _P = {}
    for _pt, _nm in (("PartType4", "star"), ("PartType0", "gas")):
        _cut = read_cutout(_s, _g, _c, _pt, fields=("Masses", "Metallicity"))
        if _cut is None:
            _P = None
            break
        _met = _cut["Metallicity"]
        _z = np.asarray(_met[:, 0] if _met.ndim == 2 else _met, float)  # col 0 = total Z
        _P[_nm] = (_cut["pos"], np.asarray(_cut["Masses"], float), _z)
    if _P is None:
        _skipped.append((int(_s), int(_g)))
        continue
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _nm, (_pos, _m, _z) in _P.items():
            _R = projected_radius(_pos, NHAT[_j])
            for _k, _lab in enumerate(APERTURE_LABELS):              # cumulative
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m, _R <= R_EDGES[_k + 1])
            for _k, _lab in enumerate(ANNULUS_LABELS[1:], start=1):  # annular
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m,
                                              (_R > R_EDGES[_k]) & (_R <= R_EDGES[_k + 1]))
        _rows.append(_row)

ZTAB = Table(_rows)
ZTAB.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
_out = os.path.join(TABLEDIR, "aperture_metallicities.fits")
ZTAB.write(_out, overwrite=True)
print(f"{len(ZTAB)} rows ({len(ZTAB)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")

# ── summary: sample vs the CIGALE grids (Part 7d pins ONE node per object) ──
ZSUN = 0.0134
_zs_grid = grid_options("bc03", "metallicity")
print(f"\nbc03 metallicity nodes: {_zs_grid}")
print(f"{'label':>10s} | {'med Z*/Zsun':>11s} {'med Zgas/Zsun':>13s} | bc03 node counts (stars)")
for _lab in Z_LABELS:
    _zsv = np.asarray(ZTAB[f"Zstar_{_lab}"], float)
    _zgv = np.asarray(ZTAB[f"Zgas_{_lab}"], float)
    _near = nearest_option(_zsv, _zs_grid)
    _cnt = {f"{_n:g}": int(np.sum(_near == _n)) for _n in _zs_grid
            if np.sum(_near == _n)}
    _nnan = int(np.sum(~np.isfinite(_near)))
    if _nnan:
        _cnt["free"] = _nnan
    print(f"{_lab:>10s} | {np.nanmedian(_zsv)/ZSUN:11.2f} {np.nanmedian(_zgv)/ZSUN:13.2f} | {_cnt}")


# Part 5 — Stage 1: selection HDF5 + Slurm masters (all three runs)

## ⚠ REQUIRED once per powderday install: the multi-aperture patch

Stock powderday gives the peeled SED a **single infinite aperture**; the parameter masters in this
repo now carry `SED_APERTURE_NAP / SED_APERTURE_MIN_KPC / SED_APERTURE_MAX_KPC`, but powderday must
be taught to read them (**already applied** in this cluster's `powderday/front_end_tools.py`,
both SED branches — redo only on a fresh install). On the cluster, locate the peeled-image setup:

```bash
grep -rn "add_peeled_images" $(python -c "import powderday, os; print(os.path.dirname(powderday.__file__))")
```

and immediately after the image-configuration lines (`set_viewing_angles` / `set_track_origin` /
`set_uncertainties`), insert (adapt `image` / `cfg.par` to the local variable names in that file):

```python
# --- multi-aperture SEDs (analize_simba_cgm patch) ---
try:
    from hyperion.util.constants import kpc as _kpc
    _nap = int(getattr(cfg.par, 'SED_APERTURE_NAP', 0))
    if _nap > 0:
        image.set_aperture_range(_nap,
                                 float(cfg.par.SED_APERTURE_MIN_KPC) * _kpc,
                                 float(cfg.par.SED_APERTURE_MAX_KPC) * _kpc)
        image.set_uncertainties(True)   # Monte-Carlo SED errors -> <filter>_err columns
except Exception as _e:
    print('[aperture patch] skipped:', _e)
```

Hyperion **log-spaces** the apertures between min and max: 1→100 kpc with `NAP=5` gives the
10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc** (central → outskirts).
Viewing angles need **no extra patch**: stock powderday already reads
`MANUAL_ORIENTATION / THETA / PHI` from the parameter master. Verify with the Part 6 QC cell
after the first galaxy finishes.

Then run this cell, launch `submit_all_snaps.sh` under each run's `powderday_sed_out/`, and come
back to Part 6/7 when the `.rtout.sed` files exist.

In [ ]:
# ── MakeSED handles (constructor only — cheap; Parts 6–7 need just this cell, not the next) ──
from simbanator.sed.makesed import MakeSED

makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=sed_output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

In [ ]:
# ── write the selection HDF5 + generate the Slurm masters (run once per sample change) ──
# The regenerated masters list EVERY selection source per snap, but the job
# template now exits early on any galaxy whose .rtout.sed already exists —
# resubmitting after the expansion only runs the gaps.
SEL, SNAPS, IDS = load_selection()
_pop5 = (np.char.strip(np.asarray(SEL["pop"], str)) if "pop" in SEL.colnames
         else np.full(len(SEL), "Q"))

for key, cfg in RUNS.items():
    ms = makeseds[key]
    if key == "agn_on":
        # 2026-08-14: agn_on stays restricted to the ORIGINAL quenched anchors;
        # SF controls + quenched at the new anchors get dust_on + dust_off only
        _m5 = (_pop5 == "Q") & np.isin(np.asarray(SEL["snap"], int), sorted(AGN_ARM_SNAPS))
    else:
        _m5 = np.ones(len(SEL), bool)
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') — "
          f"{int(_m5.sum())}/{len(SEL)} sources ===")
    ms.selection_gals(snaps=SNAPS[_m5], galaxyID=IDS[_m5])
    ms.create_master('cluster', 'region', radius=R_CUTOUT_KPC,
                     partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)


# Part 6 — Aperture QC (run after the first `.rtout.sed` exists)

Confirms the powderday patch took effect **before** burning time on the full extraction:
reads the aperture layout stored in one output file, probes every aperture index, and prints
the expected index → radius mapping.

In [ ]:
from simbanator.sed.makesed import list_sed_apertures, _read_sed

_pat = os.path.join(makeseds['dust_on'].model_dir_base, 'snap_*', 'gal_*', '*.rtout.sed')
_cands = sorted(glob.glob(_pat))
if not _cands:
    raise FileNotFoundError(f"no .rtout.sed yet under {makeseds['dust_on'].model_dir_base} — "
                            "run the RT jobs first")
_probe = _cands[0]
print("probing:", _probe, "\n")

for gname, entry in list_sed_apertures(_probe).items():
    print(f"[{gname}] seds shape = {entry.get('seds_shape')}")
    for k, v in entry.get('seds_attrs', {}).items():
        print(f"    seds.attrs[{k!r}] = {v}")
    for k, v in entry['group_attrs'].items():
        print(f"    group.attrs[{k!r}] = {v}")

print("\nexpected mapping (log-spaced, from the parameter master):")
for i, r in enumerate(APERTURE_RADII_KPC):
    mark = (f"   <- {APERTURE_LABELS[WANTED_AP_IDX.index(i)]}"
            if i in WANTED_AP_IDX else "")
    print(f"  aperture={i:2d} -> {r:7.2f} kpc{mark}")

n_ok = 0
for i in range(N_AP):
    try:
        wav, flx, unc = _read_sed(_probe, aperture=i, uncertainties=True)
        assert np.shape(flx)[0] == N_INCL, (
            f"{np.shape(flx)[0]} inclination(s) in the rtout but N_INCL={N_INCL} — "
            "THETA/PHI here disagree with the parameter master the jobs copied")
        has_unc = unc is not None and np.isfinite(np.asarray(unc)).any()
        print(f"  aperture={i}: OK  flux shape={np.shape(flx)}  MC uncertainties={'yes' if has_unc else 'NO'}")
        n_ok += 1
    except Exception as e:
        print(f"  aperture={i}: FAILED ({type(e).__name__}: {e})")
assert n_ok == N_AP, (
    f"only {n_ok}/{N_AP} apertures readable — the powderday aperture patch is NOT active "
    "(or N_AP here disagrees with SED_APERTURE_NAP in the parameter master the jobs copied)")
print(f"\nOK — {N_AP} apertures present.")

# Part 7 — Stage 2: per-aperture flux extraction → catalogs

Filter set (2026-07-10): **Subaru/HSC, CFHT/MegaCam, HST/WFC3, JWST/NIRCam+MIRI,
Spitzer/MIPS (24/70/160 µm) + Herschel/PACS+SPIRE (70–500 µm — the dust-emission
peak), VISTA/VIRCAM, JCMT/SCUBA-2, ALMA band 6** (custom 211–275 GHz top-hat,
`ALMA_band6.res`) + the Johnson V/U & 2MASS J locals kept for Part 7a.

For each dust run × aperture: convolve the SED (and its Monte-Carlo uncertainty) with the filter
set, then join the sample metadata. **One catalog per aperture per RT arm** under
`output/cis25/sed_aperture_catalogs/`, columns: `gal_id, snap, redshift`, sample metadata
(`agn_class, log_mstar, ngas, t_sft, t_qt, tau_q, xstr_quench`, …) and per-filter
`<filter>` / `<filter>_err` fluxes (mJy, rest-frame convolution as in `test_powderday.ipynb`).

In [ ]:
# ── filter set (user-chosen mock-observation instruments, 2026-07-09) ──
# optical: Subaru/HSC, CFHT/MegaCam, HST/WFC3; near-IR: JWST/NIRCam, VISTA
# (SVO lists it as Paranal/VIRCAM; CIGALE names it paranal.vircam.*);
# mid-IR: JWST/MIRI; far-IR: Spitzer/MIPS 24/70/160 um (samples the dust
# emission peak that MIRI + the sub-mm bands only straddle); sub-mm/mm:
# Herschel/PACS (70/100/160 um) + SPIRE (250/350/500 um) bracket the cold-dust
# peak (added 2026-07-10, user request); sub-mm/mm: JCMT/SCUBA-2 + ALMA band 6
# (local top-hat, 211-275 GHz — the SED grid is 0.001-1000 um REST, so
# observed-frame band 6 is covered at every anchor; at z=0.3 its red edge is
# truncated at 1.31 mm).
FACILITIES  = ['Subaru', 'CFHT', 'HST', 'JWST', 'JWST', 'Spitzer',
               'Herschel', 'Herschel', 'Paranal', 'JCMT']
INSTRUMENTS = ['HSC', 'MegaCam', 'WFC3', 'NIRCam', 'MIRI', 'MIPS',
               'PACS', 'SPIRE', 'VIRCAM', 'SCUBA2']

local_filters = {
    # Johnson V/U + 2MASS J stay for the Part 7a/7f true A_V (never fitted)
    '2MASS':   {'J': {'J': REMOTE_HOME + '/2MASS_J.res'}},
    'Johnson': {'V': {'V': REMOTE_HOME + '/maiz-apellaniz_Johnson_V.res'}},
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {'U': {'U': REMOTE_HOME + '/maiz-apellaniz_Johnson_U.res'}},
    # custom: not in SVO; the matching ALMA_band6_cigale.dat must be added to
    # the CIGALE env once: pcigale-filters add ALMA_band6_cigale.dat
    'ALMA': {'ALMA': {'band6': REMOTE_HOME + '/ALMA_band6.res'}},
}

SEL, SNAPS, IDS = load_selection()

def extract_flux_set(redshift, prefix):
    """extract_flux_batch over every (RT arm, aperture rung, sightline).

    redshift=False -> rest-frame (this part's catalogs); True -> observed
    frame (the Part 7b CIGALE inputs). One pass = 2 x 5 x 4 = 40 extractions;
    returns {(dust key, aperture label, incl label): flux-table path}.
    """
    files = {}
    for key in RUNS:
        ms = makeseds[key]
        for i, label in zip(WANTED_AP_IDX, APERTURE_LABELS):
            for j, ilab in enumerate(INCL_LABELS):
                print(f"\n=== extract: {key} / {label} / {ilab} (aperture {i}, "
                      f"inclination {j}, {'observed' if redshift else 'rest'} frame) ===")
                flux_file, _ = ms.extract_flux_batch(
                    SNAPS, IDS, FACILITIES, INSTRUMENTS,
                    filters=None, local_filters=local_filters, wave_unit='micron',
                    findx=j, aperture=i, uncertainties=True, redshift=redshift,
                    outname=f"{prefix}_{key}_{label}_{ilab}.fits")
                files[(key, label, ilab)] = flux_file
    return files

FLUX_FILES = extract_flux_set(redshift=False, prefix="fluxes")   # rest frame


In [ ]:
# ── final catalogs: fluxes+errors ⨝ sample metadata; one file per (RT arm, aperture) ──
_META = ["gal_id", "snap", "z_snap", "z_target", "agn_class", "xstr_quench",
         "log_mstar", "ngas", "nstar", "ssfr", "t_sft", "t_qt", "tau_q", "tau_q_over_tH",
         "r50_star_kpc", "flag_too_large", "flag_unresolved"]
SEL, SNAPS, IDS = load_selection()
_META = [c for c in _META if c in SEL.colnames]   # size-QC columns exist after Part 3b

CATALOGS = {}
for (key, label, ilab), ff in FLUX_FILES.items():
    t = Table.read(ff)
    if len(t) == 0:
        print(f"[{key}/{label}/{ilab}] EMPTY flux table — skipped"); continue
    t.rename_column('gal_id_at_snap', 'gal_id')
    cat = join(t, SEL[_META], keys=['snap', 'gal_id'], join_type='left')
    flux_cols = [c for c in t.colnames if c not in ('gal_id', 'snap', 'redshift')]
    cat = cat[['gal_id', 'snap', 'redshift'] + [c for c in _META if c not in ('gal_id', 'snap')]
              + flux_cols]
    out = os.path.join(CATDIR, f"catalog_{key}_{label}_{ilab}.fits")
    _k = APERTURE_LABELS.index(label)
    cat.meta['APERTURE'] = label
    cat.meta['APIDX'] = WANTED_AP_IDX[_k]
    cat.meta['APKPC'] = float(APERTURE_RADII_KPC[WANTED_AP_IDX[_k]])   # true rung radius
    cat.meta['INCL'] = ilab
    cat.meta['THETA'] = THETA_DEG[INCL_LABELS.index(ilab)]
    cat.meta['PHI'] = PHI_DEG[INCL_LABELS.index(ilab)]
    cat.meta['DUSTRUN'] = key
    cat.write(out, overwrite=True)
    CATALOGS[(key, label, ilab)] = out
    n_err = sum(1 for c in cat.colnames if c.endswith('_err'))
    print(f"[{key}/{label}/{ilab}] {len(cat)} galaxies, {n_err} error columns -> {out}")

# ── cross-check: per aperture, every run must contain the same sources as dust_on ──
print()
for label in APERTURE_LABELS:
    for ilab in INCL_LABELS:
        fon = FLUX_FILES.get(('dust_on', label, ilab))
        if fon is None:
            continue
        t_on = Table.read(fon)
        s_on = set(zip(np.asarray(t_on['snap'], int), np.asarray(t_on['gal_id_at_snap'], int)))
        for key in (k for k in RUNS if k != 'dust_on'):
            fk = FLUX_FILES.get((key, label, ilab))
            if fk is None:
                continue
            t_k = Table.read(fk)
            s_k = set(zip(np.asarray(t_k['snap'], int), np.asarray(t_k['gal_id_at_snap'], int)))
            status = "OK" if s_on == s_k else f"MISMATCH on={sorted(s_on - s_k)} {key}={sorted(s_k - s_on)}"
            print(f"{label:>10s}/{ilab}: dust_on={len(s_on)} {key}={len(s_k)} -> {status}")

# Part 7a — Dust attenuation $A_V$ from the matched dust_on / dust_off fluxes

We already have dust_on **and** dust_off fluxes for the *same* galaxies, so the rest-frame
band attenuation is a direct differential measurement — no SED fit needed:

$$A_\lambda = -2.5\,\log_{10}\!\left(\frac{F_{\rm dust\_on}}{F_{\rm dust\_off}}\right)\ \ [\mathrm{mag}]$$

measured per aperture from the Part-7 catalogs (rest-frame `Johnson.V.V` for $A_V$, plus
`Johnson2.U.U`/`2MASS.J.J` for the curve slope $A_U\!-\!A_V$). The **radial** attenuation
is built the observational way: annular fluxes $F(<r_{\rm out})-F(<r_{\rm in})$ between
consecutive aperture rungs give $A_V$ per annulus (annuli whose differential flux goes
non-positive from MC noise are masked).

$A_V$ is correlated against

- the **anchor-epoch ISM**: $f_{\rm mol}$, $M_{H_2}/M_\star$, $f_{\rm gas}$, dust-to-gas and
  $f_{\rm dust}=M_{\rm dust}/M_\star$ (histories, row 0), plus $\kappa_{\rm rot}$ of the H$_2$
  gas disk — the H$_2$-mass-weighted Sales+2012 $\kappa_{\rm rot}$ inside 20 pkpc, computed
  from the Stage-0 region cutouts (caesar's all-gas $\kappa_{\rm rot}$ kept for comparison);
- the **quench diagnostics + stellar structure at the observation epoch**: sSFR, $M_\star$,
  mass-weighted age, $\log Z_\star/Z_\odot$, stellar $B/T$ (caesar `rotation.stellar_BoverT`),
  $\tau_q$, $\tau_q/t_H$ and the AGN class.

Outputs: `tables/attenuation_vs_ism.fits` (per-galaxy $A_\lambda$ + ISM + structure +
quench/AGN, with $A_V$ per aperture **and** per annulus), a Spearman-ranked correlation
table, and three figures (`attenuation_vs_ism.png`, `attenuation_vs_quench.png`,
`attenuation_aperture_curve.png`).

*Needs Parts 0–3 (histories under `SFHDIR`), the Part-4 region cutouts (for
$\kappa_{\rm rot}^{H_2}$), the anchor caesar catalogs (for $Z$, $B/T$, $\kappa_{\rm rot}$)
and Part 7 (flux catalogs); CIGALE is **not** required.*

In [ ]:
# ── Part 7a — Dust attenuation (A_λ) from dust_on/dust_off vs ISM & quenching ──
# Since we already have matched dust_on / dust_off fluxes for the SAME galaxies,
# the rest-frame band attenuation follows directly (no SED fit needed):
#       A_λ = -2.5 log10( F_dust_on / F_dust_off )   [mag]
# measured per aperture from the Part-7 catalogs. The RADIAL profile is built the
# observational way: annular fluxes F(<r_out) - F(<r_in) between consecutive
# aperture rungs -> A_V per annulus. A_V is then correlated against the
# anchor-epoch ISM content (H2/HI/gas/dust from the histories + kappa_rot of the
# H2 disk from the Stage-0 region cutouts) and the quench diagnostics + stellar
# structure (mass, age, metallicity, B/T) at the observation epoch.
from scipy.stats import spearmanr
from simbanator.sed.flux_extraction import attenuation_mag as _atten

ATTEN_BANDS = {"A_U": "Johnson2.U.U", "A_V": "Johnson.V.V", "A_J": "2MASS.J.J"}
AGN_COLORS  = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
               "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}
ZSUN       = 0.0134     # Asplund+2009 total-Z scale (SIMBA's Solar reference)
R_KROT_KPC = 20.0       # H2-disk kappa_rot measured inside this proper radius

# ── 1. anchor-epoch ISM masses + stellar age (row 0 of each history) ──
GAS_KEYS = ["masses.H2", "masses.HI", "masses.gas", "masses.dust", "masses.stellar",
            "ages.mass_weighted"]
_gasdb = anchor_row0(GAS_KEYS)                                   # row 0 = anchor epoch
_missing_gas = [k for k in GAS_KEYS if not any(k in v for v in _gasdb.values())]
if _missing_gas:
    print("WARNING: ISM fields absent from histories (dropped at build):", _missing_gas)
print(f"ISM masses: {len(_gasdb)} (snap,gal) rows from the anchor histories")

def _gp_arr(tab, key):
    """anchor-epoch mass `key` aligned to a catalog table's (snap, gal_id) rows."""
    return row0_arr(_gasdb, tab["snap"], tab["gal_id"], key)

# ── 1b. anchor-epoch structure from the caesar catalogs (direct h5py read) ──
# metallicities / stellar B/T / gas kappa_rot are not tracked in the histories;
# GroupID == row index in the caesar files, so a plain dataset read suffices.
STRUCT_KEYS = {"Z_star": "metallicities.stellar", "Z_gas": "metallicities.mass_weighted",
               "BT_star": "rotation.stellar_BoverT", "kappa_gas": "rotation.gas_kappa_rot"}
_structdb, _posdb = {}, {}            # (snap,gid) -> {props} / (pos[kpccm], a, h)
_need = {}
for _s, _g in _gasdb:
    _need.setdefault(_s, set()).add(_g)
for _snap, _gids in sorted(_need.items()):
    _cf = sim.get_caesar_file(_snap)
    if not os.path.exists(_cf):
        print(f"WARNING: no caesar file for snap {_snap} -> structure props stay NaN")
        continue
    with h5py.File(_cf, "r") as f:
        _dcts = f["galaxy_data/dicts"]
        _vals = {k: _dcts[v][:] for k, v in STRUCT_KEYS.items() if v in _dcts}
        _pos  = f["galaxy_data/pos"][:]                          # kpccm
        _sa   = f["simulation_attributes"].attrs
        _a, _h = float(_sa["scale_factor"]), float(_sa["hubble_constant"])
    _absent = [v for k, v in STRUCT_KEYS.items() if k not in _vals]
    if _absent:
        print(f"WARNING: snap {_snap} caesar file lacks {_absent}")
    for _g in _gids:
        if _g < len(_pos):
            _structdb[(_snap, _g)] = {k: float(_arr[_g]) for k, _arr in _vals.items()}
            _posdb[(_snap, _g)]    = (np.asarray(_pos[_g], float), _a, _h)
print(f"structure props: {len(_structdb)} (snap,gal) rows from the anchor caesar catalogs")

def _sp_arr(tab, key):
    """anchor-epoch structure prop `key` aligned to a catalog table's rows."""
    return np.array([_structdb.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(tab["snap"], tab["gal_id"])])

# ── 1c. kappa_rot of the H2 gas disk from the Stage-0 region cutouts ──
def _kappa_rot_h2(snap, gid):
    """Sales+12 kappa_rot of the H2-mass-weighted gas within R_KROT_KPC (proper).

    Cutout Coordinates are code units (ckpc/h) unwrapped around the caesar centre,
    so pos_kpccm*h recovers that centre exactly; the uniform sqrt(a) factor of the
    code velocities cancels in the K_rot/K ratio.
    """
    rec = _posdb.get((int(snap), int(gid)))
    if rec is None:
        return np.nan
    pos_kpccm, a_scale, hub = rec
    cut = read_cutout(snap, gid, pos_kpccm * hub, "PartType0",
                      fields=("Velocities", "Masses", "FractionH2"))
    if cut is None or cut["FractionH2"] is None:
        return np.nan
    r   = cut["pos"]                                            # proper kpc, gal frame
    v   = np.asarray(cut["Velocities"], float)
    mh2 = np.asarray(cut["Masses"], float) * np.asarray(cut["FractionH2"], float)
    sel = (np.sqrt(np.sum(r**2, axis=1)) < R_KROT_KPC) & (mh2 > 0) & np.isfinite(mh2)
    if sel.sum() < 10:
        return np.nan
    r, vv, w = r[sel], v[sel], mh2[sel]
    r  = r  - np.average(r,  axis=0, weights=w)                 # recentre on the H2 body
    vv = vv - np.average(vv, axis=0, weights=w)
    j  = np.cross(r, vv)
    L  = np.sum(w[:, None] * j, axis=0)
    if not np.isfinite(L).all() or np.linalg.norm(L) == 0:
        return np.nan
    zhat = L / np.linalg.norm(L)
    jz   = j @ zhat
    Rcyl = np.sqrt(np.maximum(np.sum(r**2, axis=1) - (r @ zhat)**2, 0.0))
    ok   = Rcyl > 1e-3
    Krot = 0.5 * np.sum(w[ok] * (jz[ok] / Rcyl[ok])**2)
    Ktot = 0.5 * np.sum(w * np.sum(vv**2, axis=1))
    return float(Krot / Ktot) if Ktot > 0 else np.nan

# ── 2. per-aperture attenuation table (dust_on ⨝ dust_off on snap+gal_id) ──
ATTEN_INCL = INCL_LABELS[0]     # fiducial sightline for the A_λ analysis
def _load_atten(label, incl=None):
    incl = ATTEN_INCL if incl is None else incl
    fon  = os.path.join(CATDIR, f"catalog_dust_on_{label}_{incl}.fits")
    foff = os.path.join(CATDIR, f"catalog_dust_off_{label}_{incl}.fits")
    if not (os.path.exists(fon) and os.path.exists(foff)):
        return None
    on, off = Table.read(fon), Table.read(foff)
    off_cols = ["snap", "gal_id"] + [c for c in ATTEN_BANDS.values() if c in off.colnames]
    m = join(on, off[off_cols], keys=["snap", "gal_id"],
             table_names=["on", "off"], metadata_conflicts="silent")
    for aname, col in ATTEN_BANDS.items():
        m[aname] = (_atten(m[f"{col}_on"], m[f"{col}_off"])
                    if f"{col}_on" in m.colnames and f"{col}_off" in m.colnames
                    else np.full(len(m), np.nan))
    return m

_atab = {lab: _load_atten(lab) for lab in APERTURE_LABELS}
_atab = {k: v for k, v in _atab.items() if v is not None and len(v)}
if not _atab:
    raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
FID_AP = APERTURE_LABELS[-1] if APERTURE_LABELS[-1] in _atab else list(_atab)[-1]
print(f"apertures with catalogs: {list(_atab)}  |  fiducial (global A_V) = {FID_AP}"
      f"  |  sightline = {ATTEN_INCL}")

# ── 3. fiducial-aperture analysis table: A_λ + ISM tracers + structure + quench ──
base  = _atab[FID_AP]
_MH2  = _gp_arr(base, "masses.H2");   _MHI = _gp_arr(base, "masses.HI")
_Mgas = _gp_arr(base, "masses.gas");  _Md  = _gp_arr(base, "masses.dust")
_Mst  = _gp_arr(base, "masses.stellar")
with np.errstate(all="ignore"):
    f_mol       = _MH2 / (_MH2 + _MHI)          # molecular fraction of neutral gas
    f_H2_star   = _MH2 / _Mst                    # specific molecular content
    f_gas       = _Mgas / (_Mgas + _Mst)         # gas fraction
    DGR         = _Md / _Mgas                     # dust-to-gas ratio
    f_dust_star = _Md / _Mst                      # f_dust: specific dust content

ATTEN = Table()
ATTEN["snap"]     = np.asarray(base["snap"], int)
ATTEN["gal_id"]   = np.asarray(base["gal_id"], int)
ATTEN["z_target"] = np.asarray(base["z_target"], float)
for aname in ATTEN_BANDS:
    ATTEN[aname] = np.asarray(base[aname], float)
ATTEN["S_UV"] = ATTEN["A_U"] - ATTEN["A_V"]      # attenuation-curve slope proxy (mag)
for lab, tt in _atab.items():                    # enclosed A_V at every aperture
    idx = {(int(s), int(g)): k for k, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
    col = np.full(len(base), np.nan)
    for k, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
        j = idx.get((int(s), int(g)))
        if j is not None:
            col[k] = tt["A_V"][j]
    ATTEN[f"A_V_{lab}"] = col
with np.errstate(all="ignore"):
    ATTEN["log_MH2"]   = np.log10(np.where(_MH2 > 0, _MH2, np.nan))
    ATTEN["log_Mgas"]  = np.log10(np.where(_Mgas > 0, _Mgas, np.nan))
    ATTEN["log_Mdust"] = np.log10(np.where(_Md > 0, _Md, np.nan))
ATTEN["f_mol"] = f_mol; ATTEN["f_H2_star"] = f_H2_star; ATTEN["f_gas"] = f_gas
ATTEN["DGR"] = DGR; ATTEN["f_dust_star"] = f_dust_star
for c in ("log_mstar", "ssfr", "tau_q", "tau_q_over_tH", "xstr_quench"):
    ATTEN[c] = np.asarray(base[c], float)
_agn = np.asarray(base["agn_class"])
ATTEN["agn_class"] = np.array([a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a)
                               for a in _agn])

# anchor-epoch stellar structure + gas-disk rotation (observation time)
ATTEN["age_mw"] = _gp_arr(base, "ages.mass_weighted")          # Gyr, mass-weighted
with np.errstate(all="ignore"):
    ATTEN["logZ_star"] = np.log10(_sp_arr(base, "Z_star") / ZSUN)
    ATTEN["logZ_gas"]  = np.log10(_sp_arr(base, "Z_gas") / ZSUN)
ATTEN["BT_star"]   = _sp_arr(base, "BT_star")
ATTEN["kappa_gas"] = _sp_arr(base, "kappa_gas")
ATTEN["kappa_H2"]  = np.array([_kappa_rot_h2(s, g)
                               for s, g in zip(base["snap"], base["gal_id"])])
print(f"kappa_H2 (<{R_KROT_KPC:g} pkpc): measured for "
      f"{int(np.isfinite(np.asarray(ATTEN['kappa_H2'])).sum())}/{len(ATTEN)} galaxies "
      f"(needs the Stage-0 cutouts + >=10 H2-bearing gas particles)")

# ── 3b. annular A_V — the observational radial profile: F(<r_out) - F(<r_in) ──
_labs_all = [l for l in APERTURE_LABELS if l in _atab]
_r_out    = np.array([APERTURE_RADII_KPC[WANTED_AP_IDX[APERTURE_LABELS.index(l)]]
                      for l in _labs_all])                     # TRUE rung radii [pkpc]
_r_in     = np.concatenate([[0.0], _r_out[:-1]])
_r_mid    = np.where(_r_in > 0, np.sqrt(_r_in * _r_out), _r_out / 2.0)

def _band_matrix(col):
    """(n_gal, n_ap) matrix of catalog column `col`, aligned to the base rows."""
    M = np.full((len(base), len(_labs_all)), np.nan)
    for k, lab in enumerate(_labs_all):
        tt = _atab[lab]
        if col not in tt.colnames:
            continue
        idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
        for i, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
            j = idx.get((int(s), int(g)))
            if j is not None:
                M[i, k] = tt[col][j]
    return M

_Von, _Voff = _band_matrix("Johnson.V.V_on"), _band_matrix("Johnson.V.V_off")
_dVon  = np.column_stack([_Von[:, :1],  np.diff(_Von,  axis=1)])   # annular fluxes
_dVoff = np.column_stack([_Voff[:, :1], np.diff(_Voff, axis=1)])
AV_ANN = _atten(_dVon, _dVoff)
for k, lab in enumerate(_labs_all):
    ATTEN[f"A_V_ann_{lab}"] = AV_ANN[:, k]
_nneg = int(np.sum(((_dVon <= 0) | (_dVoff <= 0)) & np.isfinite(_Von) & np.isfinite(_Voff)))
print(f"annular A_V: {len(_labs_all)} annuli/galaxy; {_nneg} annuli with non-positive "
      f"differential flux (MC noise / empty annulus) -> NaN")

_out = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
ATTEN.write(_out, overwrite=True)
_av = np.asarray(ATTEN["A_V"], float)
print(f"attenuation table: {len(ATTEN)} galaxies ({FID_AP}) -> {_out}")
print(f"A_V [{FID_AP}]  median={np.nanmedian(_av):.3f}  90th={np.nanpercentile(_av,90):.3f}  "
      f"max={np.nanmax(_av):.3f}   (A_V>0.1 mag: {int(np.nansum(_av>0.1))}/{len(ATTEN)})")

# ── 4. Spearman correlations of the global A_V with everything ──
_targets = [("f_mol","f_mol"), ("f_H2_star","M_H2/M*"), ("f_gas","f_gas"),
            ("DGR","dust/gas"), ("f_dust_star","f_dust"), ("log_MH2","log M_H2"),
            ("log_Mgas","log M_gas"), ("kappa_H2","kappa_H2"), ("kappa_gas","kappa_gas"),
            ("log_mstar","log M*"), ("age_mw","age_mw"), ("logZ_star","log Z*/Zsun"),
            ("logZ_gas","log Zg/Zsun"), ("BT_star","B/T"), ("ssfr","sSFR"),
            ("tau_q","tau_q"), ("tau_q_over_tH","tau_q/t_H"), ("S_UV","A_U-A_V")]
_ranked = []
for col, lbl in _targets:
    x = np.asarray(ATTEN[col], float); ok = np.isfinite(_av) & np.isfinite(x)
    if ok.sum() >= 5:
        rho, p = spearmanr(_av[ok], x[ok]); _ranked.append((lbl, rho, p, int(ok.sum())))
_ranked.sort(key=lambda r: -abs(r[1]))
print("\nSpearman  A_V  vs …   (fiducial aperture, sorted by |rho|)")
print(f"  {'quantity':12s} {'rho':>7s} {'p':>10s} {'n':>4s}")
for lbl, rho, p, n in _ranked:
    flag = "***" if p < 0.01 else "** " if p < 0.05 else "*  " if p < 0.1 else ""
    print(f"  {lbl:12s} {rho:+7.3f} {p:10.2e} {n:4d}  {flag}")

# per-aperture robustness of the two headline ISM correlations
print("\nrobustness across apertures  (rho[p]):")
for lab in _labs_all:
    tt = _atab[lab]; av = np.asarray(tt["A_V"], float)
    with np.errstate(all="ignore"):
        fh2 = _gp_arr(tt, "masses.H2") / _gp_arr(tt, "masses.stellar")
        dgr = _gp_arr(tt, "masses.dust") / _gp_arr(tt, "masses.gas")
    def _rp(x):
        ok = np.isfinite(av) & np.isfinite(x)
        return spearmanr(av[ok], x[ok]) if ok.sum() >= 5 else (np.nan, np.nan)
    (r1, p1), (r2, p2) = _rp(fh2), _rp(dgr)
    print(f"  {lab:16s}  M_H2/M*: {r1:+.2f}[{p1:.2g}]   dust/gas: {r2:+.2f}[{p2:.2g}]")

# ── 5. figures ──
_cls_present = [c for c in ["strong","intermediate","weak","no_AGN","no_event","unclassified"]
                if c in set(ATTEN["agn_class"])]
def _scatter(ax, xcol, xlabel, xlog=False):
    x = np.asarray(ATTEN[xcol], float); y = _av
    for cls in _cls_present:
        s = ATTEN["agn_class"] == cls
        ax.scatter(x[s], y[s], s=28, c=AGN_COLORS.get(cls, "#333"), label=cls,
                   edgecolor="k", linewidth=0.3, alpha=0.85)
    ok = np.isfinite(x) & np.isfinite(y)
    if xlog: ok &= x > 0
    if ok.sum() >= 5:
        rho, p = spearmanr(x[ok], y[ok])
        ax.text(0.04, 0.95, f"$\\rho$={rho:+.2f}\np={p:.2g}", transform=ax.transAxes,
                va="top", fontsize=8.5, bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    if xlog:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$A_V$ [mag]")

# Fig 1 — attenuation vs ISM / dust content (the H2 connection)
_p1 = [("f_mol", r"$f_{\rm mol}=M_{H_2}/(M_{H_2}\!+\!M_{HI})$", False),
       ("f_H2_star", r"$M_{H_2}/M_\star$", True),
       ("f_gas", r"$f_{\rm gas}=M_{\rm gas}/(M_{\rm gas}\!+\!M_\star)$", False),
       ("DGR", r"dust-to-gas $M_{\rm dust}/M_{\rm gas}$", True),
       ("f_dust_star", r"$f_{\rm dust}=M_{\rm dust}/M_\star$", True),
       ("kappa_H2", r"$\kappa_{\rm rot}^{H_2}$ ($<$%g pkpc)" % R_KROT_KPC, False)]
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
for ax, (c, xl, xlog) in zip(axes.flat, _p1):
    _scatter(ax, c, xl, xlog)
axes.flat[0].legend(fontsize=10, loc="upper right", framealpha=0.9)
fig.suptitle(f"$A_V$ vs ISM ({FID_AP})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f1 = os.path.join(PLOTDIR, "attenuation_vs_ism.png")
fig.savefig(_f1, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f1)

# Fig 2 — attenuation vs quenching + stellar structure at the observation epoch
_p2 = [("ssfr", "sSFR [yr$^{-1}$]", True),
       ("log_mstar", r"$\log_{10} M_\star\,[M_\odot]$", False),
       ("age_mw", "mass-weighted age [Gyr]", False),
       ("logZ_star", r"$\log_{10} Z_\star/Z_\odot$", False),
       ("BT_star", r"stellar $B/T$", False),
       ("tau_q", r"$\tau_q$ [yr]", False),
       ("tau_q_over_tH", r"$\tau_q/t_H$", False)]
fig, axes = plt.subplots(2, 4, figsize=(26, 12))
for ax, (c, xl, xlog) in zip(axes.flat[:7], _p2):
    _scatter(ax, c, xl, xlog)
ax = axes.flat[7]
for i, cls in enumerate(_cls_present):
    s = ATTEN["agn_class"] == cls; yv = _av[np.asarray(s)]
    xj = i + np.random.uniform(-0.16, 0.16, size=int(np.sum(s)))
    ax.scatter(xj, yv, c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, s=28)
    if np.isfinite(yv).any():
        ax.hlines(np.nanmedian(yv), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_xticks(range(len(_cls_present)))
ax.set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=10)
ax.set_ylabel(r"$A_V$ [mag]"); ax.set_title("by AGN class")
fig.suptitle(f"$A_V$ vs quenching ({FID_AP})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f2 = os.path.join(PLOTDIR, "attenuation_vs_quench.png")
fig.savefig(_f2, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f2)

# Fig 3 — radial A_V from annular fluxes + attenuation-curve slope
fig, (axL, axR) = plt.subplots(1, 2, figsize=(17, 7))
_AVap  = np.vstack([np.asarray(ATTEN[f"A_V_{l}"], float) for l in _labs_all]).T
_AVann = np.vstack([np.asarray(ATTEN[f"A_V_ann_{l}"], float) for l in _labs_all]).T
_dusty = _av > AV_DUSTY                                   # most quenched gals are transparent;
for row in _AVann[_dusty]:                           # the radial trend only matters there
    axL.plot(_r_mid, row, "-", color="0.7", lw=0.8, alpha=0.7, zorder=1)
axL.plot(_r_mid, np.nanmedian(_AVann[_dusty], axis=0), "o-", color="#c0392b", lw=2,
         label=f"annular median, $A_V\\!>\\!0.1$ (n={int(_dusty.sum())})", zorder=3)
axL.plot(_r_mid, np.nanmedian(_AVann, axis=0), "s--", color="#2980b9",
         label=f"annular median, all (n={len(_av)})", zorder=2)
axL.plot(_r_out, np.nanmedian(_AVap[_dusty], axis=0), ":", color="0.35", lw=1.5,
         label=r"enclosed $A_V(<r)$, $A_V\!>\!0.1$", zorder=2)
axL.set_xscale("log"); axL.set_xlabel("radius [pkpc]")
axL.set_ylabel(r"$A_V$ [mag]")
axL.set_title(r"radial $A_V$ (annuli)")
axL.legend(fontsize=10, frameon=False)
for cls in _cls_present:
    s = ATTEN["agn_class"] == cls
    axR.scatter(_av[np.asarray(s)], np.asarray(ATTEN["S_UV"])[np.asarray(s)], s=28,
                c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, alpha=0.85, label=cls)
axR.axhline(0, color="0.6", lw=0.8, ls="--")
axR.set_xlabel(r"$A_V$ [mag]"); axR.set_ylabel(r"$A_U-A_V$ [mag]  (curve slope)")
axR.legend(fontsize=10, frameon=False)
fig.tight_layout()
_f3 = os.path.join(PLOTDIR, "attenuation_aperture_curve.png")
fig.savefig(_f3, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f3)

# Part 7b — CIGALE input files (one per RT arm × aperture × sightline)

Writes `output/cis25/sed_aperture_catalogs/cigale/cigale_{dust_on|dust_off}_{ap…}.fits` in the
exact input format of **CIGALE 2025.0**. All the format/mapping logic lives in
**`simbanator.sed.cigale`** (band names verified against the 2025.0 filter database):

- columns `id` (`snapNNN_galID`), `redshift`, `distance` (Mpc, Planck13 — the same D_L used to
  normalize the fluxes), then per band the flux **in mJy** + its `<band>_err`;
- band names match the CIGALE DB exactly (`jwst.nircam.F200W`, `hst.wfc3.ir.F160W`,
  `spitzer.irac.I1`, `herschel.pacs.green`, `2mass.J`, `generic.johnson.U/V`, …); bands with no
  CIGALE counterpart (grisms, quad filters) are dropped and reported;
- missing fluxes are NaN; make an error negative by hand for upper-limit treatment.

Two deliberate choices:

1. **Observed frame.** CIGALE compares redshifted models to observed photometry, so the
   extraction reruns with `redshift=True` (the Part 7 catalogs stay rest-frame).
2. **Raw MC errors (`err_floor=0`).** CIGALE itself adds `additionalerror` (10 % by default,
   set in Part 7d's `prepare_run`) in quadrature at fit time — a floor here too would be
   double-counted. The Hyperion MC error alone is just RT convergence noise.

These per-aperture files are no longer fitted directly: **Part 7b2** differences them into the
3 region catalogs (core / outskirt / cgm) that Part 7d actually runs on. The
`cigale_fluxes_*` intermediates this cell writes are what 7b2 reads, so run 7b first.

In [ ]:
# ── Part 7b: CIGALE 2025.0 input files — observed-frame fluxes+errors, one per (RT arm, aperture, sightline) ──
# All three arms are written; Part 7d fits dust_on and agn_on (plus a small
# dust_off zero-point control). dust_off is otherwise the A_V reference only.
# Format + band mapping live in simbanator.sed.cigale (verified against the 2025.0 filter DB).
# Needs the Part 5 MakeSED handles + the Part 7 filter/extractor cell in this session.
from simbanator.sed.cigale import write_cigale_input

os.makedirs(CIGALE_DIR, exist_ok=True)

# CIGALE compares redshifted models to observed photometry -> observed frame
# (the Part 7 catalogs stay rest-frame)
CIGALE_FLUX_FILES = extract_flux_set(redshift=True, prefix="cigale_fluxes")

CIGALE_FILES = {}
for (key, label, ilab), flux_file in CIGALE_FLUX_FILES.items():
    # err_floor=0: CIGALE adds its own 10% 'additionalerror' in quadrature at fit time
    CIGALE_FILES[(key, label, ilab)] = write_cigale_input(
        flux_file, os.path.join(CIGALE_DIR, f"cigale_{key}_{label}_{ilab}.fits"),
        err_floor=0.0)

print(f"\n{len(CIGALE_FILES)} CIGALE input files -> {CIGALE_DIR}")


# Part 7b2 — region CIGALE inputs: core / outskirt / cgm

The many-annulus CIGALE campaign split the flux too thin to fit (5 annuli × 4 sightlines), and
the cumulative ladder that replaced it fitted curves-of-growth, not places. The fits now run on
**3 broad regions** — enough flux each, and each a physically distinct zone:

| region | radii | flux construction |
|---|---|---|
| `core` | 0 → 3.16 kpc | the `ap3kpc` cumulative catalog as-is |
| `outskirt` | 3.16 → 31.6 kpc | `F(<31.6) − F(<3.16)` (`annular_flux_table`) |
| `cgm` | 31.6 → 100 kpc | `F(<100) − F(<31.6)` |

- Differencing conventions are Part 7a's (`simbanator.sed.flux_extraction.annular_flux_table`):
  non-positive/non-finite region flux → **NaN flux + error** (missing band to CIGALE); errors
  `sqrt(err_out² − err_in²)`, quadrature-sum fallback where MC noise inverts it.
- **Closure is exact by construction**: the rungs are cumulative and filter convolution is linear,
  so core + outskirt + cgm telescopes back to the `ap100kpc` catalog band by band. What can
  actually break is source matching between the rung tables — that is what the check below guards.
- **Sampling QC**: with Part 4b's `annulus_particle_counts.fits` present, (source, sightline)
  pairs with **zero star particles** in a region are counted — their region flux is scattered
  light only and the fit is meaningless. The `cgm` rows will dominate both this census and the
  NaN-band one; that is expected and accepted (core + outskirt are the science, and Part 7d's
  `MIN_FIT_BANDS` drops photometrically empty rows instead of failing).

Reads the Part 7b observed-frame `cigale_fluxes_*` intermediates from disk (run Part 7b once
first); no MakeSED handles needed.

In [ ]:
# ── Part 7b2: region CIGALE inputs — core / outskirt / cgm ──
# core = the ap3kpc cumulative catalog as-is; outskirt/cgm = differences of the
# cumulative rungs (annular_flux_table: NaN on non-positive, subtracted errors).
# Reads the Part 7b observed-frame cigale_fluxes_* intermediates from disk.
from simbanator.sed.flux_extraction import annular_flux_table
from simbanator.sed.cigale import write_cigale_input

os.makedirs(CIGALE_DIR, exist_ok=True)

_qcf = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
_qc = Table.read(_qcf) if os.path.exists(_qcf) else None
if _qc is None:
    print("[QC] annulus_particle_counts.fits not found — run Part 4b to flag "
          "star-free regions")

# region star counts from the annulus census (annuli are disjoint; ann1kpc is
# the 0->1 kpc disc, so a region's count is the sum over its annuli)
_REGION_ANNS = {"core": ("ann1kpc", "ann3kpc"),
                "outskirt": ("ann10kpc", "ann32kpc"),
                "cgm": ("ann100kpc",)}

CIGALE_REGION_FILES = {}
_nan_census = {}
for _key in RUNS:
    _fluxdir = os.path.join(sed_output_dir, RUNS[_key]['run_tag'], 'sed_fluxes')
    for _ilab in INCL_LABELS:
        _fap = {_l: os.path.join(_fluxdir,
                                 f"cigale_fluxes_{_key}_{_l}_{_ilab}.fits")
                for _l in APERTURE_LABELS}
        _miss = [os.path.basename(p) for p in _fap.values()
                 if not os.path.exists(p)]
        if _miss:
            raise FileNotFoundError(
                f"{_key}/{_ilab}: run Part 7b first — missing {_miss}")
        # closure guard: the differencing is exact ONLY if the rung tables hold
        # the same sources in the same rows (annular_flux_table re-matches on
        # (snap, gal), but a dropped source silently loses its region row)
        _ids = None
        for _l in ("ap3kpc", "ap32kpc", "ap100kpc"):
            _tt = Table.read(_fap[_l])
            _cur = list(zip(np.asarray(_tt["snap"], int),
                            np.asarray(_tt["gal_id_at_snap"], int)))
            if _ids is not None and _cur != _ids:
                print(f"  [closure] {_key}/{_ilab}: source list differs "
                      f"between rungs at {_l} — differencing keeps only the "
                      "intersection")
            _ids = _cur
        for _reg, _d in REGION_DEFS.items():
            if _d["ap_in"] is None:            # core == the cumulative disc
                _t = Table.read(_fap[_d["ap_out"]])
            else:                              # outskirt / cgm: rung difference
                _t = annular_flux_table(_fap[_d["ap_in"]], _fap[_d["ap_out"]],
                                        verbose=False)
            _bands = [c for c in _t.colnames
                      if c not in ("gal_id_at_snap", "snap", "redshift")
                      and not c.endswith("_err")]
            _fl = np.array([np.asarray(_t[c], float) for c in _bands])
            _nan_census[(_key, _reg, _ilab)] = float((~np.isfinite(_fl)).mean())
            # err_floor=0: CIGALE adds its own 10% additionalerror at fit time
            CIGALE_REGION_FILES[(_key, _reg, _ilab)] = write_cigale_input(
                _t, os.path.join(CIGALE_DIR,
                                 f"cigale_{_key}_{_reg}_{_ilab}.fits"),
                err_floor=0.0, verbose=False)

if _qc is not None:
    print("\n[QC] sources with ZERO star particles in a region "
          "(region flux = scattered light only):")
    for _reg, _anns in _REGION_ANNS.items():
        _tot, _n_all = 0, 0
        for _ilab in INCL_LABELS:
            _m = np.char.strip(np.asarray(_qc["incl"], str)) == _ilab
            _n = np.zeros(int(_m.sum()), int)
            for _a in _anns:
                _n = _n + np.asarray(_qc[f"nstar_{_a}"], int)[_m]
            _tot += int((_n == 0).sum())
            _n_all += int(_m.sum())
        print(f"   {_reg:>9s}: {_tot} of {_n_all} (source, sightline) pairs")

print("\n[NaN bands] fraction of NaN region fluxes (MC noise / empty region):")
for _reg in REGION_LABELS:
    _v = [_nan_census[k] for k in _nan_census if k[1] == _reg]
    print(f"   {_reg:>9s}: median {np.median(_v):.1%}, max {np.max(_v):.1%} "
          "over (arm, sightline)")
print(f"\n{len(CIGALE_REGION_FILES)} region CIGALE input files -> {CIGALE_DIR}")

# Part 7c — region-matched **formed-mass** SFH archive (the injected prior)

Part 7d fits every region with **its own** star-formation history: `sfhfromfile` with a single
column per run. This cell builds the archive those runs read.

For each selected galaxy, sightline and **region** (core / outskirt / cgm, plus a `galaxy` entry —
the whole 100 kpc cutout — as the QC reference) it takes the **archaeological** SFH — mass formed
per 100 Myr from the star-particle formation times in the Stage-0 cutout, i.e. the exact particles
the RT saw — smooths it (25 Myr grid, 150 Myr Gaussian kernel) and stores it as
`cigale/sfh_smoothed_regions.h5`, keyed `snapNNN_galID/<sightline>/<region>`.

**2026-08-15 — the builder lives in simbanator now.** `build_mfrac_lookup` / `mfrac_of` /
`projected_region_sfh` in **`simbanator.analysis.sfh_utils`** are this notebook's tested code
promoted verbatim (same FSPS settings, same fixed `0→t_obs` 100 Myr histogram grid, same
`smooth_resample_sfh` smoothing) — the cell is now a thin driver. The existing
`tables/fsps_mfrac_lookup.npz` cache is reused unchanged. Note this is deliberately *not*
`simbanator.analysis.sfh_fsps` (snapshot-scoped, hardcoded cosmology, log-space FSPS grid,
data-dependent bin edges — none of which match what powderday rendered).

**Why per region and per sightline.** The photometry Part 7b2 differences is region- and
sightline-resolved, so the stellar population inside the core along `i0p0` is *not* the one the
global history describes — cores are older and more quenched. Injecting the global SFH into a
region fit would import exactly the age–dust degeneracy this exercise removes. The masking reuses
the Part 7e geometry (`read_cutout` → `projected_radius` → `r_in < R ≤ r_out`), so archive and
`aperture_truth.fits` describe the same particles.

## Formed mass, not surviving mass (2026-08-10, unchanged)

The archive stores the **formed**-mass SFR. Powderday scales each star particle's FSPS spectrum by

```python
lum = trapz(fnu, nu) * star.mass / mfrac        # source_creation.py:63
mfrac = sp.stellar_mass                          # SED_gen.py:407
```

— FSPS SSPs are normalised to **1 M<sub>⊙</sub> formed**, so powderday divides by the surviving
fraction to recover the initial mass. A histogram of the *current* particle masses sits low by
`mfrac ≈ 0.55` at the old end: an **~80 % tilt** across the curve that `normalise=True` cannot
remove — and with the SFH injected there is nothing left to absorb it except $A_V$. CIGALE wants
the same correction independently (`sfhfromfile` feeds `bc03.convolve`, which applies BC03's own
mass loss). The lookup reproduces powderday's FSPS population exactly (Chabrier `imf_type=1`,
`pagb=1`, `add_agb_dust_model=True`, remnants at the FSPS default, metallicity snapped in
**linear** space to match `find_nearest_zmet`), and the HDF5 records `sfh_mass_kind` /
`mfrac_source` in its attrs — **Part 7d refuses an archive that is not `formed`**.

**Caveats.** Regions holding fewer than `SFH_NSTAR_MIN` star particles are left out; those
(galaxy, region, sightline) combinations get **no run**, so the coverage table below is a sample
cut and is reported as one — the `cgm` region will be the worst hit, by design. The smoothed grid
spans the histogram bin *centres*, so the last ~50 Myr is edge-held by the interpolation in Part
7d; `sfr_hold_frac` there records how much of the recent SFR that affects. Times are on the
notebook's `COSMO` (Planck15); Part 7d re-caps the age against pcigale's own Planck18 ceiling.

In [ ]:
# ── Part 7c — REGION-matched FORMED-mass SFH archive (the injected prior) ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster). Writes one
# smoothed archaeological SFH per (galaxy, sightline, region) — the SAME
# projected geometry Part 7b2 differenced the photometry in, and the same
# particles Part 7e measures its truth from — plus a 'galaxy' entry (the whole
# 100 kpc cutout) as the closure reference for Part 7c2. Part 7d injects the
# matching region SFH into each CIGALE run via sfhfromfile (shape only,
# normalise=True).
#
# 2026-08-15: the builder lives in simbanator.analysis.sfh_utils
# (build_mfrac_lookup / mfrac_of / projected_region_sfh — promoted verbatim
# from this notebook: same FSPS settings, same 100 Myr histogram on the fixed
# 0->t_obs grid, same 25/150 Myr smoothing). Masses are divided by the FSPS
# surviving fraction before the histogram, so the stored SFR is the FORMED
# mass per year — what powderday actually rendered (source_creation.py:63
# scales each SSP by mass/mfrac) and what bc03 expects. See the markdown.
from simbanator.analysis.sfh_utils import (build_mfrac_lookup, mfrac_of,
                                           projected_region_sfh)

SFH_REGIONS_H5        = os.path.join(CIGALE_DIR, "sfh_smoothed_regions.h5")
OVERWRITE_SFH_ARCHIVE = False
SFH_ARCH_BIN_MYR      = 100.0    # archaeological bin width before smoothing
SFH_ARCH_DT_MYR       = 25.0     # smoothed output grid step
SFH_ARCH_KERNEL_MYR   = 150.0    # Gaussian kernel sigma
SFH_NSTAR_MIN         = 20       # fewer star particles -> shot-noise, not an SFH
SFH_ARCH_LABELS       = REGION_LABELS + ["galaxy"]   # 'galaxy' = whole cutout
_SFH_MASKS = {**{r: (d["r_in"], d["r_out"]) for r, d in REGION_DEFS.items()},
              "galaxy": (0.0, R_CUTOUT_KPC)}

# ── powderday's FSPS settings: simbanator/sed/parameters_master.py, the file
#    makesed.py copies into every model dir. NOT ~/powderday/parameters_master.py
#    (imf_type=2, add_agb_dust_model=False) — that one is never used here.
#    build_mfrac_lookup leaves add_stellar_remnants at the FSPS default (1),
#    i.e. mfrac INCLUDES remnants — matching powderday exactly.
PD_IMF_TYPE    = 1            # Chabrier
PD_PAGB        = 1
PD_AGB_DUST    = True
FSPS_MFRAC_NPZ = os.path.join(TABLEDIR, "fsps_mfrac_lookup.npz")
APPLY_MFRAC    = True
MFRAC_FALLBACK = "none"       # 'analytic' -> Chabrier R(t); 'none' -> refuse

os.makedirs(CIGALE_DIR, exist_ok=True)


def load_sfh_archive(path=None):
    """{id: {incl: {region: (t_gyr, sfr, t_obs_gyr)}}} from the archive."""
    path = path or SFH_REGIONS_H5
    out = {}
    with h5py.File(path, "r") as f:
        for sid in f:
            for il in f[sid]:
                for lab in f[sid][il]:
                    d = f[sid][il][lab]
                    arr = np.asarray(d[:], float)
                    out.setdefault(str(sid), {}).setdefault(str(il), {})[str(lab)] = (
                        arr[:, 0], arr[:, 1], float(d.attrs["t_obs_gyr"]))
    return out


def sfh_archive_attrs(path=None):
    """Provenance attrs of the archive (empty dict if it does not exist)."""
    path = path or SFH_REGIONS_H5
    if not os.path.exists(path):
        return {}
    with h5py.File(path, "r") as f:
        return {k: (v.decode() if isinstance(v, bytes) else v)
                for k, v in f.attrs.items()}


if os.path.exists(SFH_REGIONS_H5) and not OVERWRITE_SFH_ARCHIVE:
    _at = sfh_archive_attrs()
    _arch = load_sfh_archive()
    _n = sum(len(a) for g in _arch.values() for a in g.values())
    print(f"cached: {_n} region SFHs for {len(_arch)} galaxies -> "
          f"{SFH_REGIONS_H5}  (OVERWRITE_SFH_ARCHIVE=True rebuilds)")
    print(f"   sfh_mass_kind={_at.get('sfh_mass_kind', '?')}  "
          f"mfrac_source={_at.get('mfrac_source', '?')}")
    if str(_at.get("sfh_mass_kind", "")) != "formed":
        print("   *** this archive predates the formed-mass fix — Part 7d will "
              "refuse it. Set OVERWRITE_SFH_ARCHIVE=True and re-run. ***")
else:
    # ── the surviving-mass fraction, exactly as powderday computed it ──
    _MF_TAG = "UNCORRECTED"
    _MF_LOOKUP = None
    if APPLY_MFRAC:
        try:
            _MF_LOOKUP = build_mfrac_lookup(FSPS_MFRAC_NPZ,
                                            imf_type=PD_IMF_TYPE, pagb=PD_PAGB,
                                            add_agb_dust_model=PD_AGB_DUST)
            _ZLEG, _LAGE, _MFRAC, _MF_TAG = _MF_LOOKUP
            print(f"[mfrac] {_MF_TAG}")
            print(f"[mfrac] {len(_ZLEG)} Z nodes x {len(_LAGE)} ages; "
                  f"mfrac(1 Gyr, Zsol)={np.interp(9.0, _LAGE, _MFRAC[np.argmin(np.abs(_ZLEG - 0.019))]):.3f}  "
                  f"mfrac(10 Gyr, Zsol)={np.interp(10.0, _LAGE, _MFRAC[np.argmin(np.abs(_ZLEG - 0.019))]):.3f}")
        except ImportError as _exc:
            if MFRAC_FALLBACK != "analytic":
                raise RuntimeError(
                    f"fsps is not importable in this kernel ({_exc}) and "
                    "MFRAC_FALLBACK='none'. Without it the archive would hold "
                    "CURRENT-mass SFHs, tilted ~80% against the oldest bins "
                    "relative to what powderday rendered — and with the SFH "
                    "injected that tilt lands directly in A_V. Run this cell "
                    "in the pd39 kernel, or set MFRAC_FALLBACK='analytic'.") from _exc
            # Chabrier return fraction R(t) = C ln(t/lam + 1) (Jungwiert+2001):
            # within ~5% of FSPS/MIST beyond 1 Gyr, and normalise=True only
            # ever sees the SHAPE.
            _MF_TAG = "analytic-chabrier"
            print(f"[mfrac] fsps unavailable ({_exc}) — {_MF_TAG} fallback")

    def _mfrac(age_gyr, zstar):
        if not APPLY_MFRAC:
            return np.ones_like(np.asarray(age_gyr, float))
        if _MF_LOOKUP is not None:
            return mfrac_of(age_gyr, zstar, _MF_LOOKUP)
        return np.clip(1.0 - 0.05 * np.log(np.asarray(age_gyr, float) * 1e3 / 0.4
                                           + 1.0), 0.05, 1.0)

    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)
    _ag = np.linspace(0.02, 1.0, 4096)               # a -> cosmic time grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value           # Gyr

    _nw = _ngal_ok = _nskip = 0
    _nthin = {}                                      # (incl, label) -> skips
    _demo = {}                                       # QC figure payload
    with h5py.File(SFH_REGIONS_H5, "w") as f5:
        f5.attrs["bin_myr"] = SFH_ARCH_BIN_MYR
        f5.attrs["dt_myr"] = SFH_ARCH_DT_MYR
        f5.attrs["kernel_myr"] = SFH_ARCH_KERNEL_MYR
        f5.attrs["nstar_min"] = SFH_NSTAR_MIN
        f5.attrs["cosmology"] = COSMO.name
        f5.attrs["mfrac_applied"] = bool(APPLY_MFRAC)
        f5.attrs["mfrac_source"] = _MF_TAG
        f5.attrs["region_edges"] = json.dumps(
            {r: [float(d["r_in"]), float(d["r_out"])]
             for r, d in REGION_DEFS.items()})
        f5.attrs["sfh_mass_kind"] = ("formed" if APPLY_MFRAC and
                                     _MF_TAG != "UNCORRECTED" else "current")
        for _snap in np.unique(SNAPS):
            _z = float(sim.get_z_from_snap(int(_snap)))
            _t_obs = float(COSMO.age(_z).value)
            _ns = 0
            _mfs, _mcur, _mform = [], 0.0, 0.0
            for _gid in np.unique(IDS[SNAPS == _snap]):
                _cut = read_cutout(_snap, _gid,
                                   _cen.get((int(_snap), int(_gid))), "PartType4",
                                   fields=("Masses", "StellarFormationTime",
                                           "Metallicity"))
                if _cut is None or _cut.get("StellarFormationTime") is None:
                    print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre/stars")
                    _nskip += 1
                    continue
                _a = np.asarray(_cut["StellarFormationTime"], float)
                _ok = np.isfinite(_a) & (_a > 0) & (_a <= 1)
                if int(_ok.sum()) < SFH_NSTAR_MIN:
                    _nskip += 1
                    continue
                _tf = np.interp(np.clip(_a[_ok], _ag[0], 1.0), _ag, _tg)   # Gyr
                _m0 = np.asarray(_cut["Masses"], float)[_ok] * 1e10 / _cut["h"]
                _pos = _cut["pos"][_ok]
                # current -> FORMED mass (powderday's mass/mfrac scaling)
                _Zp = _cut.get("Metallicity")
                if _Zp is None:
                    _Zs = np.full(_m0.shape, 0.019)   # solar-ish fallback
                else:
                    _Zp = np.asarray(_Zp, float)
                    _Zs = (_Zp[:, 0] if _Zp.ndim == 2 else _Zp)[_ok]
                _mf = _mfrac(np.clip(_t_obs - _tf, 1e-4, None), _Zs)
                _mm = _m0 / _mf
                _mfs.append(_mf)
                _mcur += float(_m0.sum())
                _mform += float(_mm.sum())
                _sid = f"snap{int(_snap):03d}_gal{int(_gid)}"
                _ngal_ok += 1
                _ns += 1
                for _il, _nv in zip(INCL_LABELS, NHAT):
                    for _lab in SFH_ARCH_LABELS:
                        _rin, _rout = _SFH_MASKS[_lab]
                        _ts, _ss, _nap = projected_region_sfh(
                            _tf, _mm, _pos, _nv, _rin, _rout, _t_obs,
                            nstar_min=SFH_NSTAR_MIN, bin_myr=SFH_ARCH_BIN_MYR,
                            dt_myr=SFH_ARCH_DT_MYR,
                            kernel_myr=SFH_ARCH_KERNEL_MYR)
                        if _ts is None:
                            _nthin[(_il, _lab)] = _nthin.get((_il, _lab), 0) + 1
                            continue
                        _d = f5.create_dataset(f"{_sid}/{_il}/{_lab}",
                                               data=np.column_stack([_ts, _ss]))
                        _d.attrs["t_obs_gyr"] = _t_obs
                        _d.attrs["nstar"] = _nap
                        _d.attrs["r_in_kpc"] = float(_rin)
                        _d.attrs["r_out_kpc"] = float(_rout)
                        _nw += 1
                        # QC: keep the uncorrected twin for 3 galaxies, 'galaxy'
                        if (len(_demo) < 3 and _il == INCL_LABELS[0]
                                and _lab == "galaxy"):
                            _t0, _s0, _ = projected_region_sfh(
                                _tf, _m0, _pos, _nv, _rin, _rout, _t_obs,
                                nstar_min=SFH_NSTAR_MIN,
                                bin_myr=SFH_ARCH_BIN_MYR,
                                dt_myr=SFH_ARCH_DT_MYR,
                                kernel_myr=SFH_ARCH_KERNEL_MYR)
                            _demo[_sid] = (_t0, _s0, _ts, _ss)
            _mfa = np.concatenate(_mfs) if _mfs else np.array([np.nan])
            print(f"snap {_snap:3d} (z={_z:.2f}, t_obs={_t_obs:.2f} Gyr): "
                  f"{_ns} galaxies archived | mfrac median {np.median(_mfa):.3f} "
                  f"[{np.min(_mfa):.3f}, {np.max(_mfa):.3f}] | "
                  f"M_formed/M_current = {(_mform / _mcur if _mcur else np.nan):.3f}")
    print(f"\n[SFH archive] {_nw} region SFHs for {_ngal_ok} galaxies "
          f"({_nskip} galaxies skipped) -> {SFH_REGIONS_H5}")
    print(f"[SFH archive] sfh_mass_kind="
          f"{'formed' if APPLY_MFRAC and _MF_TAG != 'UNCORRECTED' else 'current'}"
          f"  ({_MF_TAG})")
    print("   M_formed/M_current ~1.5-1.8 is expected for these old systems; "
          "~1.0 means the correction did NOT apply")

    # ── QC figure: what the formed-mass correction actually does to the shape ──
    if _demo:
        fig, axs = plt.subplots(1, len(_demo), figsize=(7 * len(_demo), 5.2),
                                squeeze=False)
        for _k, (_sid, (_t0, _s0, _ts, _ss)) in enumerate(_demo.items()):
            ax = axs[0][_k]
            ax.plot(_t0, _s0 / max(_s0.sum(), 1e-30), "-", lw=1.5, color="0.55",
                    label="current mass (what the RT did NOT use)")
            ax.plot(_ts, _ss / max(_ss.sum(), 1e-30), "-", lw=1.8, color="C3",
                    label="formed mass (injected)")
            ax.set(xlabel="cosmic time [Gyr]", ylabel="normalised SFR",
                   title=_sid)
            ax.grid(alpha=0.3)
            if _k == 0:
                ax.legend(fontsize=10, frameon=False)
        fig.tight_layout()
        fig.savefig(os.path.join(PLOTDIR, "sfh_mfrac_correction.png"), dpi=150,
                    bbox_inches="tight")
        plt.show()
    _arch = load_sfh_archive()

# ── coverage: this is a SAMPLE CUT under a per-region injection, not a footnote ──
_cov = {}
for _sid, _g in _arch.items():
    for _il, _labs in _g.items():
        for _lab in _labs:
            _cov[(_il, _lab)] = _cov.get((_il, _lab), 0) + 1
_ngal = len(_arch)
print(f"\nregion SFH coverage — galaxies with an archived SFH (of {_ngal}); "
      "an empty cell means NO Part 7d run for that (galaxy, region, sightline):")
print("   " + "".join(f"{l:>10s}" for l in SFH_ARCH_LABELS))
for _il in INCL_LABELS:
    print(f"{_il:>7s}" + "".join(f"{_cov.get((_il, l), 0):>10d}"
                                 for l in SFH_ARCH_LABELS))
print("   deficit vs the full sample:", {
    l: _ngal - _cov.get((INCL_LABELS[0], l), 0) for l in SFH_ARCH_LABELS})

# ── figure: the radial SFH gradient the per-region injection preserves ──
_pick = [s for s in sorted(_arch) if len(_arch[s].get(INCL_LABELS[0], {}))
         == len(SFH_ARCH_LABELS)][:3]
if _pick:
    fig, axs = plt.subplots(1, len(_pick), figsize=(7 * len(_pick), 5.2),
                            squeeze=False)
    _cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(SFH_ARCH_LABELS)))
    for _k, _sid in enumerate(_pick):
        ax = axs[0][_k]
        for _lab, _c in zip(SFH_ARCH_LABELS, _cmap):
            _e = _arch[_sid][INCL_LABELS[0]].get(_lab)
            if _e is None:
                continue
            ax.plot(_e[0], _e[1], "-", lw=1.5, color=_c, label=_lab)
        ax.set(xlabel="cosmic time [Gyr]", ylabel=r"SFR [$M_\odot$/yr]",
               title=_sid)
        ax.grid(alpha=0.3)
        if _k == 0:
            ax.legend(fontsize=10, frameon=False, title="region",
                      title_fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "sfh_region_archive.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# Part 7c2 — region SFH QC: closure + the sightline question

The galaxy-level SFH pin is gone — Part 7d injects each region's **own** history — so the old
pin-licence question ("how wrong is one SFH for five apertures?") dissolves. Two QC measurements
on the region archive replace it:

1. **Closure.** The three regions partition the 100 kpc cutout, and the whole builder (histogram
   on a fixed grid → linear smoothing) is linear in the particle weights, so
   $\sum_{\rm regions} {\rm SFR}(t)$ must equal the `galaxy` entry **bin-exactly** wherever all
   three regions pass `SFH_NSTAR_MIN`. Any residual beyond float noise is a mask/geometry bug
   (the cell raises); the deficit from `nstar_min`-skipped regions — cgm-dominated, by design —
   is reported separately.
2. **Sightline spread.** How much does a region's SFH *shape* (mass-weighted age, recent-mass
   fraction) vary over the 4 sightlines? The `galaxy` entry must be exactly sightline-degenerate
   (it is the whole sphere along every line of sight — the cell raises otherwise, same proof as
   before). If the region spreads are also negligible, `SFH_REGION_IL` in Part 7d may collapse
   the sightline axis of the SFH + umin pins to save runs; the photometry stays per-sightline
   either way.

Output: `tables/cigale_sfh_region_shape.fits`, one row per (galaxy, sightline, region).

In [ ]:
# ── Part 7c2 — region SFH QC: closure vs the whole-cutout SFH + sightline spread ──
# Needs only the Part 7c archive (kernel-restart safe: reloads it if needed).
# (i)  CLOSURE — regions partition the cutout and the builder is linear in the
#      weights, so Sum_regions SFR(t) == the 'galaxy' SFH bin-exactly wherever
#      all three regions pass nstar_min. Residual beyond float noise = bug.
# (ii) SIGHTLINE SPREAD — negligible region spread licenses SFH_REGION_IL in
#      Part 7d (collapse the SFH/umin-pin sightline axis to save runs).
SFH_SHAPE_FITS = os.path.join(TABLEDIR, "cigale_sfh_region_shape.fits")
F_RECENT_GYR = 0.3      # 'recent' window for the UV/nebular-driving mass
_trapz = getattr(np, "trapezoid", None) or np.trapz

if "_arch" not in globals() or not _arch:
    _arch = load_sfh_archive()


def _sfh_shape_stats(t_gyr, sfr, t_obs):
    """(mass-weighted age [Myr], f_recent, normalised cumulative mass)."""
    w = np.clip(np.asarray(sfr, float), 0.0, None)
    tot = w.sum()
    if not np.isfinite(tot) or tot <= 0:
        return np.nan, np.nan, None
    age_mw = (t_obs - float((w * t_gyr).sum() / tot)) * 1e3
    f_rec = float(w[t_gyr > t_obs - F_RECENT_GYR].sum() / tot)
    return age_mw, f_rec, np.cumsum(w) / tot


# ── (i) closure: Sum_regions vs the 'galaxy' entry ──────────────────────────
_clo, _npart = [], 0
for _sid, _g in sorted(_arch.items()):
    for _il in INCL_LABELS:
        _e = _g.get(_il, {})
        _gal = _e.get("galaxy")
        if _gal is None:
            continue
        if not all(_r in _e for _r in REGION_LABELS):
            _npart += 1          # a region fell below nstar_min: partial sum
            continue
        _t, _sg = _gal[0], _gal[1]
        if any(_e[_r][0].shape != _t.shape for _r in REGION_LABELS):
            continue
        _sum = np.sum([_e[_r][1] for _r in REGION_LABELS], axis=0)
        _den = max(float(_trapz(_sg, _t)), 1e-30)
        _clo.append(float(_trapz(np.abs(_sum - _sg), _t)) / _den)
_clo = np.asarray(_clo, float)
if _clo.size:
    print(f"[closure] |Sum_regions - galaxy| L1 / formed mass: "
          f"median {np.median(_clo):.2e}, max {np.max(_clo):.2e} over "
          f"{_clo.size} (galaxy, sightline) with all 3 regions archived "
          f"({_npart} partial)")
    if np.max(_clo) > 1e-6:
        raise RuntimeError(
            "region SFHs do not sum to the whole-cutout SFH — the builder is "
            "linear, so this is a mask/geometry bug (check REGION_DEFS vs "
            "R_CUTOUT_KPC and projected_region_sfh), not physics.")
    print("   -> exact partition confirmed: the per-region injection loses "
          "nothing the whole-cutout pin had, it only localises it")
else:
    print(f"[closure] no (galaxy, sightline) had all 3 regions archived "
          f"({_npart} partial) — the cgm nstar_min cut dominates; "
          "core+outskirt coverage is what carries the science")

# ── (ii) per-region sightline spread of the SFH shape ───────────────────────
_rows = []
for _sid, _g in sorted(_arch.items()):
    for _il in INCL_LABELS:
        for _lab in REGION_LABELS + ["galaxy"]:
            _e = _g.get(_il, {}).get(_lab)
            if _e is None:
                continue
            _a, _f, _c = _sfh_shape_stats(_e[0], _e[1], _e[2])
            if _c is None:
                continue
            _rows.append(dict(
                id=_sid, snap=int(_sid.split("_gal")[0][4:]),
                gal_id=int(_sid.split("_gal")[1]), incl=_il, region=_lab,
                age_mw_myr=_a, f_recent=_f))
if not _rows:
    raise RuntimeError("no region SFHs to compare — is the Part 7c archive "
                       "built? (OVERWRITE_SFH_ARCHIVE=True)")
SHAPE = Table(rows=_rows)
SHAPE.write(SFH_SHAPE_FITS, overwrite=True)
print(f"\n[shape] {len(SHAPE)} (galaxy, sightline, region) rows -> "
      f"{SFH_SHAPE_FITS}")

_reg_col = np.char.strip(np.asarray(SHAPE["region"], str))
_id_col = np.asarray(SHAPE["id"], str)
_age_col = np.asarray(SHAPE["age_mw_myr"], float)


def _p2p_by_region(lab):
    """per-galaxy peak-to-peak of t_mw over the sightlines, for one region"""
    v = []
    for _sid in set(_id_col):
        _m = (_id_col == _sid) & (_reg_col == lab)
        if _m.sum() > 1:
            v.append(np.nanmax(_age_col[_m]) - np.nanmin(_age_col[_m]))
    return np.asarray(v, float)


_spread = {}
print("\nsightline-to-sightline spread of the mass-weighted age [Myr] "
      "(median / p95 peak-to-peak over the 4 sightlines):")
for _lab in REGION_LABELS + ["galaxy"]:
    _v = _p2p_by_region(_lab)
    _spread[_lab] = _v
    _md = np.nanmedian(_v) if _v.size else np.nan
    _p95 = np.nanpercentile(_v, 95) if _v.size else np.nan
    print(f"   {_lab:>9s}  {_md:8.1f} / {_p95:8.1f}"
          + ("   <- must be ~0: the whole cutout is sightline-independent"
             if _lab == "galaxy" else ""))
_gal_md = (np.nanmedian(_spread["galaxy"]) if _spread["galaxy"].size else np.nan)
if np.isfinite(_gal_md) and _gal_md > 1e-6:
    raise RuntimeError(
        "the whole-cutout SFH differs between sightlines — it is the entire "
        f"{R_CUTOUT_KPC:g} kpc sphere along every line of sight, so the "
        "archive and the aperture geometry have drifted apart; check "
        "projected_region_sfh / the cutout radius before fitting.")
print("   -> if the region spreads are also ~0, SFH_REGION_IL in Part 7d can "
      "collapse the SFH/umin-pin sightline axis (the photometry stays "
      "per-sightline either way); leave it None otherwise")

# ── figure: closure residuals + sightline spread ────────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(15, 5.6))
if _clo.size:
    axs[0].hist(np.log10(np.clip(_clo, 1e-20, None)), bins=30, color="C0",
                alpha=0.8)
axs[0].set(xlabel=r"$\log_{10}$ closure residual (L1 / formed mass)",
           ylabel="(galaxy, sightline)", title="region-sum closure")
axs[0].grid(alpha=0.3)
_bx = [_spread[_l] for _l in REGION_LABELS if _spread[_l].size]
_lb = [_l for _l in REGION_LABELS if _spread[_l].size]
if _bx:
    axs[1].boxplot(_bx, showfliers=False)
    axs[1].set_xticklabels(_lb)
axs[1].set(ylabel=r"sightline peak-to-peak of $t_{\rm mw}$ [Myr]",
           title="does the SFH shape depend on the sightline?")
axs[1].grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sfh_region_qc.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# Part 7d0 — region dust-mass priors: pinning the dl2014 `umin` node

CIGALE has **no dust-mass input** — `dust.mass` is an output. The dl2014 module scales a template
of fixed emissivity $\varepsilon(q_{\rm PAH}, U_{\rm min}, \gamma)$ [W per kg of dust], so the
fitted mass is just $L_{\rm dust}/\varepsilon$. That is the lever: given a region's **true** dust
mass (Part 8a's `annulus_ism_truth.fits`, region = difference of the cumulative rungs) and its RT
dust luminosity (the `.rtout.sed` integrated beyond 3 µm, `dust_on − dust_off`, differenced across
the same rungs), the target emissivity is $L/M$ — and snapping `umin` to the node that reproduces
it anchors the fitted `dust.mass` to the simulation value. The exact analogue of the per-region
`bc03.metallicity` pin. `qpah` and `gamma` stay free in the fit; the pin is computed at the
fiducial `qpah = 2.5`, `gamma = 0.02`, and the printed node spread across the gamma grid shows
what that choice costs.

- The emissivity table is exported **once by the CIGALE env python**
  (`simbanator.sed.dl2014_fit --emissivity-out`, subprocessed below — this kernel never imports
  `pcigale`) and cached to `tables/dl2014_emissivity.fits`, every DL2014 `umin` node tabulated.
- Output: `tables/region_umin_pins.fits`, per (galaxy, sightline, region): the pinned node, the
  target emissivity, `offset_dex` = $\log_{10}(\varepsilon({\rm node})/\varepsilon({\rm target}))$
  — the dex by which the pinned `dust.mass` will sit off the truth — and `pin_status`
  (`ok` / `no_dust` / `no_lum` / `clamped_lo` / `clamped_hi`). Part 7d falls back to the free
  11-node `umin` grid wherever the pin failed (cgm-dominated, by design); Part 7f closes the loop
  with `dlogMdust` vs `offset_dex`.
- Needs Part 8a's cache (**run Part 8a first**) and the `.rtout.sed` files → cluster.

In [ ]:
# ── Part 7d0 — region dust-mass prior: dl2014 umin pins (cluster, cached) ──
# CIGALE has no dust-mass input: dl2014 scales a template of fixed emissivity
# eps(qpah, umin, gamma) [W/kg], so the fitted dust.mass is L_dust/eps. Given
# the region's TRUE dust mass (Part 8a) and its RT dust luminosity (rtout SED
# integrated beyond 3 um, dust_on - dust_off), the target emissivity is L/M and
# the umin node that reproduces it anchors dust.mass to the simulation value —
# the dl2014 analogue of the per-region bc03 metallicity pin.
from simbanator.sed import cigale as cg
from simbanator.sed.makesed import _read_sed
from simbanator.sed.flux_extraction import dust_luminosity
from simbanator.sed import dl2014_fit as _dlf

EMIS_FITS           = os.path.join(TABLEDIR, "dl2014_emissivity.fits")
UMIN_PIN_FITS       = os.path.join(TABLEDIR, "region_umin_pins.fits")
OVERWRITE_UMIN_PINS = False
UMIN_PIN_QPAH       = 2.50    # fiducial template family for the pin
UMIN_PIN_GAMMA      = 0.02    # (the fit grids for qpah/gamma stay free)
PIN_GAMMA_GRID      = (0.01, 0.02, 0.05, 0.1)
MSUN_KG             = 1.98892e30

# 1. the emissivity table — exported ONCE by the CIGALE env python (this kernel
#    never imports pcigale); every umin node of the DL2014 database is tabulated
if not os.path.exists(EMIS_FITS):
    _py = os.path.join(os.path.dirname(cg.find_pcigale()), "python")
    _cmd = [_py, _dlf.__file__, "--emissivity-out", EMIS_FITS,
            "--qpah", f"{UMIN_PIN_QPAH:g}",
            "--gamma", ",".join(f"{g:g}" for g in PIN_GAMMA_GRID)]
    print("[dl2014] " + " ".join(_cmd))
    subprocess.run(_cmd, check=True)
EMIS = Table.read(EMIS_FITS)
_esel = (np.isclose(np.asarray(EMIS["qpah"], float), UMIN_PIN_QPAH)
         & np.isclose(np.asarray(EMIS["gamma"], float), UMIN_PIN_GAMMA))
if not _esel.any():
    raise RuntimeError(f"{EMIS_FITS} has no rows at qpah={UMIN_PIN_QPAH}, "
                       f"gamma={UMIN_PIN_GAMMA} — delete the file and let this "
                       "cell regenerate it with the right --qpah/--gamma")
_EMIN = float(np.min(EMIS["emissivity"][_esel]))
_EMAX = float(np.max(EMIS["emissivity"][_esel]))
print(f"[dl2014] emissivity table: {len(EMIS)} rows; at the pin family "
      f"eps spans {_EMIN:.3g}-{_EMAX:.3g} W/kg over umin "
      f"{np.min(EMIS['umin'][_esel]):g}-{np.max(EMIS['umin'][_esel]):g}")

# 2. truth dust masses (Part 8a): region = difference of the cumulative rungs
_ismf = os.path.join(TABLEDIR, "annulus_ism_truth.fits")
if not os.path.exists(_ismf):
    raise RuntimeError(f"{_ismf} missing — run Part 8a first (it caches the "
                       "per-aperture ISM truth this pin is anchored to)")
_ISM = Table.read(_ismf)
for _c in ("incl", "aperture"):
    _ISM[_c] = np.char.strip(np.asarray(_ISM[_c], str))


def _mdust_map(label):
    _m = np.asarray(_ISM["aperture"], str) == label
    return {(int(s), int(g), str(i)): float(v) for s, g, i, v in
            zip(_ISM["snap"][_m], _ISM["gal_id"][_m], _ISM["incl"][_m],
                np.asarray(_ISM["M_dust"], float)[_m])}


_MD_AP = {_l: _mdust_map(_l) for _l in
          sorted({d["ap_out"] for d in REGION_DEFS.values()}
                 | {d["ap_in"] for d in REGION_DEFS.values() if d["ap_in"]})}

if os.path.exists(UMIN_PIN_FITS) and not OVERWRITE_UMIN_PINS:
    UMIN_PINS = Table.read(UMIN_PIN_FITS)
    print(f"cached ({len(UMIN_PINS)} rows) -> {UMIN_PIN_FITS}  "
          "(OVERWRITE_UMIN_PINS=True rebuilds)")
else:
    SEL, SNAPS, IDS = load_selection()
    _sed_on = os.path.join(sed_output_dir, RUNS['dust_on']['run_tag'],
                           'powderday_sed_out')
    _sed_off = os.path.join(sed_output_dir, RUNS['dust_off']['run_tag'],
                            'powderday_sed_out')
    _ap_need = sorted({i for d in REGION_DEFS.values()
                       for i in d["ap_idx"] if i is not None})
    _rows, _nmiss = [], 0
    for _s, _g in zip(SNAPS, IDS):
        _rel = os.path.join(f"snap_{int(_s):03d}", f"gal_{int(_g)}",
                            f"snap{int(_s):03d}.galaxy{int(_g):06d}.rtout.sed")
        _fon, _foff = os.path.join(_sed_on, _rel), os.path.join(_sed_off, _rel)
        if not (os.path.exists(_fon) and os.path.exists(_foff)):
            _nmiss += 1
            continue
        _L_ap = {}
        for _iap in _ap_need:                    # cumulative L_dust per rung
            _won, _von, _ = _read_sed(_fon, aperture=int(_iap),
                                      uncertainties=False)
            _woff, _voff, _ = _read_sed(_foff, aperture=int(_iap),
                                        uncertainties=False)
            _L_ap[_iap] = np.atleast_1d(
                dust_luminosity(_won, _von, _voff))   # (n_incl,) [W]
        for _j, _il in enumerate(INCL_LABELS):
            for _reg, _d in REGION_DEFS.items():
                _iin, _iout = _d["ap_idx"]
                _L = float(_L_ap[_iout][_j]) - (float(_L_ap[_iin][_j])
                                                if _iin is not None else 0.0)
                _L = max(_L, 0.0)                # MC noise can go negative
                _k3 = (int(_s), int(_g), _il)
                _Mout = _MD_AP[_d["ap_out"]].get(_k3, np.nan)
                _Min = (_MD_AP[_d["ap_in"]].get(_k3, np.nan)
                        if _d["ap_in"] else 0.0)
                _M = (max(_Mout - _Min, 0.0) * MSUN_KG
                      if np.isfinite(_Mout) and np.isfinite(_Min) else np.nan)
                _node, _et, _off = cg.pin_umin(_L, _M, EMIS,
                                               qpah=UMIN_PIN_QPAH,
                                               gamma=UMIN_PIN_GAMMA)
                if not np.isfinite(_M) or _M <= 0:
                    _st = "no_dust"
                elif _L <= 0:
                    _st = "no_lum"
                elif not np.isfinite(_node):
                    _st = "no_table"
                elif _et < _EMIN:
                    _st = "clamped_lo"
                elif _et > _EMAX:
                    _st = "clamped_hi"
                else:
                    _st = "ok"
                _rows.append(dict(snap=int(_s), gal_id=int(_g), incl=_il,
                                  region=_reg, L_dust_W=_L, M_dust_kg=_M,
                                  emis_target=_et, umin_node=_node,
                                  offset_dex=_off, pin_status=_st))
    if _nmiss:
        print(f"[7d0] {_nmiss} galaxies without both dust_on+dust_off rtout SEDs")
    UMIN_PINS = Table(rows=_rows)
    UMIN_PINS.write(UMIN_PIN_FITS, overwrite=True)
    print(f"{len(UMIN_PINS)} (galaxy, sightline, region) pins -> {UMIN_PIN_FITS}")

# ── census + the cost of the fiducial (qpah, gamma) choice ──────────────────
_stat = np.char.strip(np.asarray(UMIN_PINS["pin_status"], str))
_regc = np.char.strip(np.asarray(UMIN_PINS["region"], str))
print("\npin status per region:")
for _reg in REGION_LABELS:
    _m = _regc == _reg
    _u, _n = np.unique(_stat[_m], return_counts=True)
    print(f"   {_reg:>9s}: " + "  ".join(f"{a}:{b}" for a, b in zip(_u, _n)))
_okm = np.isin(_stat, ("ok", "clamped_lo", "clamped_hi"))
if _okm.any():
    _offv = np.asarray(UMIN_PINS["offset_dex"], float)[_okm]
    print(f"\n[offset] pinned dust.mass will sit off the truth by "
          f"median {np.median(np.abs(_offv)):.3f} dex "
          f"(p95 {np.percentile(np.abs(_offv), 95):.3f}) — Part 7f closes the "
          "loop with dlogMdust vs offset_dex")
    # gamma sensitivity: re-snap the usable rows at every gamma of the fit grid
    _Lok = np.asarray(UMIN_PINS["L_dust_W"], float)[_okm]
    _Mok = np.asarray(UMIN_PINS["M_dust_kg"], float)[_okm]
    _nodes0 = np.asarray(UMIN_PINS["umin_node"], float)[_okm]
    _vary = np.zeros(int(_okm.sum()), bool)
    for _gam in PIN_GAMMA_GRID:
        if np.isclose(_gam, UMIN_PIN_GAMMA):
            continue
        _alt = np.array([cg.pin_umin(_l, _mm, EMIS, qpah=UMIN_PIN_QPAH,
                                     gamma=_gam)[0]
                         for _l, _mm in zip(_Lok, _Mok)])
        _vary |= np.isfinite(_alt) & (_alt != _nodes0)
    print(f"[gamma spread] {int(_vary.sum())}/{int(_okm.sum())} pins land on a "
          f"different umin node at another gamma of {list(PIN_GAMMA_GRID)} — "
          "the free gamma grid can shift the anchored mass by about one node")

# ── QC figure: node distribution + target emissivity vs the table range ─────
fig, axs = plt.subplots(1, 2, figsize=(15, 5.6))
for _reg, _co in zip(REGION_LABELS, ("C0", "C1", "C2")):
    _m = (_regc == _reg) & _okm
    _v = np.asarray(UMIN_PINS["umin_node"], float)[_m]
    _v = _v[np.isfinite(_v) & (_v > 0)]
    if _v.size:
        axs[0].hist(np.log10(_v), bins=20, histtype="step", lw=1.8, color=_co,
                    label=f"{_reg} (n={_v.size})")
axs[0].set(xlabel=r"$\log_{10}$ pinned $U_{\rm min}$", ylabel="pins",
           title="pinned umin nodes per region")
axs[0].legend(fontsize=10, frameon=False)
axs[0].grid(alpha=0.3)
for _reg, _co in zip(REGION_LABELS, ("C0", "C1", "C2")):
    _m = _regc == _reg
    _v = np.asarray(UMIN_PINS["emis_target"], float)[_m]
    _v = _v[np.isfinite(_v) & (_v > 0)]
    if _v.size:
        axs[1].hist(np.log10(_v), bins=25, histtype="step", lw=1.8, color=_co,
                    label=_reg)
for _e, _ls in ((_EMIN, "--"), (_EMAX, ":")):
    axs[1].axvline(np.log10(_e), color="k", ls=_ls, lw=1)
axs[1].set(xlabel=r"$\log_{10}$ target emissivity $L/M$ [W/kg]",
           ylabel="(galaxy, sightline, region)",
           title="targets vs the DL2014 range (dashed/dotted = table ends)")
axs[1].legend(fontsize=10, frameon=False)
axs[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "region_umin_pins.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# Part 7d — region CIGALE runs (cluster)

CIGALE never fits inside the notebook: this cell **prepares** the run directories
(`simbanator.sed.cigale.prepare_run` — a complete, validated `pcigale.ini` + `.spec`) and writes
**one SLURM job array**; you then `sbatch` it from a login node.

## 2026-08-15 — three regions, each with its own SFH and its own dust-mass anchor

The cumulative galaxy-pinned campaign (2026-08-10) measured curves-of-growth: every fitted
aperture contained the core, so the radial statement was always a mixture. It is replaced by
**one run per (galaxy, region, sightline, chain)** on the Part 7b2 region photometry
(core 0–3.2 kpc / outskirt 3.2–31.6 kpc / cgm 31.6–100 kpc), with three per-region pins:

| axis | how it is constrained | why |
|---|---|---|
| **SFH** | the region's **own** formed-mass history (Part 7c archive), injected as the single `sfh.fits` column, shape-only (`normalise=True`) | a core is older and more quenched than its outskirts; injecting the global history would re-import the age–dust degeneracy. $M_*$ stays the fitted normalisation, so the Part 7f recovery is still honest |
| **metallicity** | `bc03.metallicity` at the nearest node to the region's own $Z_\star$ (Part 4c; outskirt/cgm = mass-weighted over their annulus columns); `nebular.zgas` from the region $Z_{\rm gas}$ | age–metallicity–dust all redden the optical; two of the three are known |
| **dust mass** | `dl2014.umin` pinned to the node whose emissivity matches the region's true $L_{\rm dust}/M_{\rm dust}$ (Part 7d0), so the fitted `dust.mass` is anchored to the simulation; `qpah`/`gamma` stay free | with the SFH fixed, any FIR mismatch leaks into $A_V$ through energy balance — anchoring the dust mass constrains that channel with simulation truth instead of leaving 11 umin nodes to wander. Rows without a usable pin (`no_dust`/`no_lum`, cgm-dominated) fall back to the free grid and are tagged `_ufree` |
| **redshift / age** | fixed at the snapshot $z$; `sfh.age` capped at pcigale's Planck18 ceiling with the time axis shifted so the *recent* end survives | unchanged from the pinned design |
| **attenuation** | **the measurand, free**: `Av_ISM` 18 nodes over 0–3 mag, `slope_ISM`, `mu`, `slope_BC` | unchanged |
| **AGN** | `skirtor2016` only in the agn chain (`fracAGN` free incl. 0.0, `i = 30°`) | unchanged |

`dust_on` + `dust_off` still share one run (identical module chain — the pcigale grid is built once
per run and shared by every catalog row); `agn_on` keeps its own. The **run count grows**
(~3 regions × 4 sightlines per galaxy × chain instead of ~1) but the umin pin shrinks the dust
grid **11×** — 2 592 models/run pinned vs 28 512 free — so the campaign cost stays comparable;
the census below prints both numbers. `SFH_REGION_IL` can collapse the SFH/umin-pin sightline axis
(merging runs 4×) **only** if Part 7c2 showed a negligible sightline spread.

### The energy-balance caveat, restated

A region does not contain all the dust heated by its own stars, nor only its own heating — light
crosses region boundaries, and CIGALE's energy balance (the pathway that sets $A_V$) is only
approximately closed per region. With 3 broad zones the leakage is far smaller than for the old
5-annulus design, and it is **accepted**: core+outskirt are the science, and the cgm region is
allowed to fail (skipped rows, NaN-band drops, large $\chi^2$ — never fatal). Part 7f quotes the
per-region residuals; do not over-interpret cgm fits.

### Identity: run name vs row id

Constant over a run → the directory name:
`reg_{dust|agn}_{region}_{incl}_snapNNN_galID_z{node}_u{node|free}` (`parse_pin_run`). Varies row
to row (the RT arm) → the source `id`, re-keyed by `cg.stack_cigale_inputs` to
`snapNNN_galID__{arm}_{region}_{sightline}` (`parse_pin_id`). Part 7f reads identity from the id
column and the chain/Z/umin pins from the run name.

### Scheduling

A pinned dust-chain run is 2 592 models (agn ×4 = 10 368) — trivially within `max_block_models`;
sources add only χ² work. The run count can exceed `MAX_ARRAY_TASKS = 1000`;
`cg.write_slurm_array` folds runs into tasks (`runs_per_task`) automatically. `SKIP_IF_DONE=True`
stays the default. Pre-flight: `PILOT=True` (one galaxy), `cg.check()`, one interactive
`cg.run()`, then `sbatch output/cis25/cigale_runs_regions/submit_cigale_regions.job`.

Workflow: **Part 7b2** → **Part 7c** (+ 7c2) → **Part 7e** (region truth → the Z weights) →
**Part 8a** → **Part 7d0** (umin pins) → this cell → sbatch → **Part 7f**.

In [ ]:
# ── Part 7d: region CIGALE runs — ONE run per (galaxy, region, sightline, chain, pins) ─
# CIGALE never runs in this kernel: the cell only writes run dirs + the job file.
# 2026-08-15 — the cumulative galaxy-pinned campaign is replaced by REGION runs:
# each run fits ONE region's photometry (Part 7b2: core / outskirt / cgm) with
#   (1) the region's OWN formed-mass SFH (Part 7c) via sfhfromfile,
#   (2) the region's bc03 metallicity node (Part 4c, annulus columns mass-
#       weighted for outskirt/cgm),
#   (3) the dl2014 umin node pinned from the region's true dust mass (Part 7d0)
#       so the fitted dust.mass is anchored to the simulation — rows without a
#       usable pin fall back to the free umin grid and are tagged '_ufree'.
# The pcigale fact that shaped the old design still holds — the model grid is
# built once per run and shared by every catalog row — so dust_on + dust_off
# remain rows of ONE run (identical chain); agn_on keeps its own (skirtor2016).
# The run count grows (one per region x sightline) but the umin pin shrinks the
# dust grid 11x; the census below prints both numbers.
from astropy.cosmology import Planck18
from simbanator.sed import cigale as cg

PCIGALE_CMD     = cg.find_pcigale()   # dedicated conda env; see the markdown
CORES_PER_TASK  = 8
PLOT_SEDS       = False   # pcigale-plots on every SED ~doubles the campaign
SKIP_IF_DONE    = True    # resubmits then only run what is missing
USE_Z_PRIORS    = True
BROADBANDS_ONLY = True
PILOT           = False   # True -> one snapshot x one galaxy

# ── what is pinned, and at what level ──
SFH_REGION_IL = None      # None -> every sightline injects its own region SFH
                          # (a label, e.g. 'i0p0', collapses the SFH+umin pins
                          #  to that sightline and merges the runs — ONLY if
                          #  Part 7c2 showed a negligible sightline spread)
Z_PIN_LEVEL   = "row"     # 'row' = the region's own Z node per run
                          # 'galaxy' = one Z node per galaxy (ap100kpc, fewer runs)

# every arm is FITTED: dust_off rides the dust chain as the zero-point control.
FIT_ARMS    = ("dust_on", "dust_off", "agn_on")
ARM_REGIONS = {a: tuple(REGION_LABELS) for a in FIT_ARMS}
ARM_INCLS   = {a: tuple(INCL_LABELS) for a in FIT_ARMS}
CHAIN_OF    = {"dust_on": "dust", "dust_off": "dust", "agn_on": "agn"}

# --- scheduling (write_slurm_array folds runs into tasks if > MAX_ARRAY_TASKS) ---
ARRAY_THROTTLE   = 48          # simultaneous tasks
RUNS_PER_TASK    = 1
MAX_ARRAY_TASKS  = 1000        # scontrol show config | grep -i MaxArraySize
PARTITIONS       = "INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL"
WALLTIME         = "0-12:00"   # PHI cores are ~3-4x slower than Cascade

RUN_BASE_REG = globals().get("RUN_BASE_REG",
                             os.path.join(OUT, "cigale_runs_regions"))
os.makedirs(RUN_BASE_REG, exist_ok=True)

_DROP_BAND = re.compile(r"\.F\d+[NM]$|\.NB\d+$|_ext$|^generic\.|^2mass\.")
_TAG_RE = re.compile(r"^(dust_on|dust_off|agn_on)_(core|outskirt|cgm)_(i\d+p\d+)$")
# run-dir name -> what is CONSTANT over the run (chain + region + sightline +
# Z node + umin pin). The arm varies row to row — see parse_pin_id.
PIN_RE = re.compile(r"^reg_(dust|agn)_(core|outskirt|cgm)_(i\d+p\d+)"
                    r"_snap(\d+)_gal(\d+)_z(\d+)_u(\d+|free)$")


def parse_pin_run(name):
    """Run-dir basename -> {chain, region, incl, snap, gal_id, zs_idx, umin_tag} or None."""
    m = PIN_RE.match(name)
    if m is None:
        return None
    return dict(chain=m.group(1), region=m.group(2), incl=m.group(3),
                snap=int(m.group(4)), gal_id=int(m.group(5)),
                zs_idx=int(m.group(6)), umin_tag=m.group(7))


def parse_pin_id(row_id):
    """Row id 'snapNNN_galID__arm_region_incl' -> the identity of ONE fitted SED."""
    sid, tag = cg.parse_stacked_id(str(row_id))
    m = _TAG_RE.match(tag)
    if m is None or "_gal" not in sid:
        return None
    return dict(id=sid, snap=int(sid.split("_gal")[0][4:]),
                gal_id=int(sid.split("_gal")[1]), arm=m.group(1),
                region=m.group(2), incl=m.group(3))


# manual fitted-band list (unselected bands are still PREDICTED as bayes.<band>)
FIT_BANDS = [
    "subaru.hsc.g", "subaru.hsc.r", "subaru.hsc.i", "subaru.hsc.z",
    "subaru.hsc.Y",
    "paranal.vircam.Y", "paranal.vircam.J", "paranal.vircam.H",
    "paranal.vircam.Ks",
    "hst.wfc3.uvis1.F606W", "hst.wfc3.uvis1.F814W",
    "jwst.nircam.F070W", "jwst.nircam.F090W", "jwst.nircam.F115W",
    "jwst.nircam.F150W", "jwst.nircam.F150W2", "jwst.nircam.F200W",
    "jwst.nircam.F277W", "jwst.nircam.F322W2", "jwst.nircam.F356W",
    "jwst.nircam.F444W",
    "jwst.miri.F560W", "jwst.miri.F770W", "jwst.miri.F1000W",
    "jwst.miri.F1130W", "jwst.miri.F1280W", "jwst.miri.F1500W",
    "jwst.miri.F1800W", "jwst.miri.F2100W", "jwst.miri.F2550W",
    "spitzer.mips.24mu", "spitzer.mips.70mu", "spitzer.mips.160mu",
    "herschel.pacs.blue", "herschel.pacs.green", "herschel.pacs.red",
    "herschel.spire.PSW", "herschel.spire.PMW",
    "jcmt.scuba2.450GHz", "jcmt.scuba2.850GHz",
    "alma.band6",
]
MIN_FIT_BANDS = 8      # fewer finite fluxes than this -> the ROW is dropped

SED_MODULES_DUST = ("sfhfromfile", "bc03", "nebular", "dustatt_modified_CF00",
                    "dl2014", "restframe_parameters", "redshifting")
SED_MODULES_AGN  = ("sfhfromfile", "bc03", "nebular", "dustatt_modified_CF00",
                    "dl2014", "skirtor2016", "restframe_parameters",
                    "redshifting")

# ── the measurand: attenuation, free ──
AV_ISM_GRID    = [0.0, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.55,
                  0.7, 0.9, 1.1, 1.4, 1.7, 2.1, 2.6, 3.0]
SLOPE_ISM_GRID = [-0.7, -0.48, -0.3]      # CF00 default + greyer ISM curves
MU_GRID        = [0.2, 0.44, 0.7]         # low mu = FIR powered by birth clouds
SLOPE_BC_GRID  = [-1.3, -0.7]             # CF00 default + a greyer BC curve
# ── nuisance: dust emission. umin is PINNED per run from Part 7d0 (free
#    UMIN_GRID is the fallback where the pin failed); qpah/gamma stay free ──
QPAH_GRID  = [2.50, 5.95]                 # powderday PAH_frac['usg'] + 1 lower
UMIN_GRID  = sorted({float(v) for v in cg.nearest_option(
    [0.1, 0.2, 0.35, 0.6, 1.0, 1.7, 3.0, 5.0, 8.0, 12.0, 25.0],
    cg.grid_options("dl2014", "umin"), log=True)})
GAMMA_GRID = [0.01, 0.02, 0.05, 0.1]
_UOPTS     = cg.grid_options("dl2014", "umin")   # node index for the dir tag
# ── nuisance: AGN (agn chain only). fracAGN keeps 0.0 reachable ──
FRAC_AGN_GRID = [0.0, 0.1, 0.2, 0.4]
SKIRTOR_I     = 30                        # type-1 view (i < 90 - oa = 50 deg)

# per-region metallicity pins
_ZS_GRID   = cg.grid_options("bc03", "metallicity")
_ZG_GRID   = cg.grid_options("nebular", "zgas")
Z_FALLBACK = 0.02      # solar-ish node when Part 4c has no value

# rest-frame line EWs (CIGALE 2025.1 label/blue/line/red format)
EW_SPEC = ("OIII5007/497.7/499.7/499.7/501.7/501.7/503.7 & "
           "Halpha/653.3/655.3/655.3/657.3/657.3/659.3 & "
           "HdeltaA/404.160/407.975/408.350/412.225/412.850/416.100")
# NOTE: no sfh.index (single column -> zero-width delta) and no sfh.tau_main /
# age_main / age_bq / r_sfr (there is no parametric SFH).
VARIABLES_BASE = ["stellar.m_star", "stellar.metallicity", "stellar.age_m_star",
                  "sfh.sfr", "sfh.sfr10Myrs", "sfh.sfr100Myrs",
                  "attenuation.Av_ISM", "attenuation.Av_BC",
                  "attenuation.generic.bessell.V",
                  "attenuation.generic.bessell.B",
                  "dust.luminosity", "dust.mass", "dust.umean",
                  "param.Dn4000", "param.EW(OIII5007)", "param.EW(Halpha)",
                  "param.EW(HdeltaA)",
                  "param.restframe_Lnu(galex.FUV)",
                  "param.restframe_Lnu(generic.bessell.V)",
                  "param.restframe_generic.johnson.U-generic.johnson.V",
                  "param.restframe_generic.johnson.V-generic.johnson.J",
                  "param.restframe_galex.FUV-galex.NUV",
                  "param.restframe_galex.NUV-sloan.sdss.r"]
VARIABLES = {"dust": VARIABLES_BASE,
             "agn":  VARIABLES_BASE + ["agn.fracAGN"]}

_n_att  = (len(AV_ISM_GRID) * len(SLOPE_ISM_GRID) * len(MU_GRID)
           * len(SLOPE_BC_GRID))
_n_dust_free = len(QPAH_GRID) * len(UMIN_GRID) * len(GAMMA_GRID)
_n_dust_pin  = len(QPAH_GRID) * 1 * len(GAMMA_GRID)
print("pcigale:", PCIGALE_CMD)
print(f"[grid] attenuation {_n_att} x dl2014 {_n_dust_pin} = "
      f"{_n_att * _n_dust_pin:,} models/run with the umin pin "
      f"({_n_att * _n_dust_free:,} on the free fallback grid)")
print(f"[grid]   x fracAGN {len(FRAC_AGN_GRID)} for the agn chain "
      f"(skirtor2016 @ i={SKIRTOR_I} deg)")
print("[grid] the grid is built ONCE per run and shared by every source in it")

# ── the injected SFHs (Part 7c) — region archive, formed mass enforced ──
SFH_REGIONS_H5 = globals().get(
    "SFH_REGIONS_H5", os.path.join(CIGALE_DIR, "sfh_smoothed_regions.h5"))
if not os.path.exists(SFH_REGIONS_H5):
    raise RuntimeError(f"{SFH_REGIONS_H5} missing — run Part 7c first")
if "load_sfh_archive" not in globals():
    def load_sfh_archive(path=None):
        """{id: {incl: {region: (t_gyr, sfr, t_obs_gyr)}}} — see Part 7c."""
        out = {}
        with h5py.File(path or SFH_REGIONS_H5, "r") as f:
            for sid in f:
                for il in f[sid]:
                    for lab in f[sid][il]:
                        d = f[sid][il][lab]
                        arr = np.asarray(d[:], float)
                        out.setdefault(str(sid), {}).setdefault(
                            str(il), {})[str(lab)] = (arr[:, 0], arr[:, 1],
                                                      float(d.attrs["t_obs_gyr"]))
        return out
with h5py.File(SFH_REGIONS_H5, "r") as _f:
    _kind = _f.attrs.get("sfh_mass_kind", b"current")
    _kind = _kind.decode() if isinstance(_kind, bytes) else str(_kind)
    _msrc = _f.attrs.get("mfrac_source", b"?")
    _msrc = _msrc.decode() if isinstance(_msrc, bytes) else str(_msrc)
if _kind != "formed":
    raise RuntimeError(
        f"{SFH_REGIONS_H5} holds '{_kind}'-mass SFHs (mfrac_source={_msrc}). "
        "powderday rendered the FORMED mass (mass/mfrac), so injecting "
        "surviving-mass rates tilts the oldest bins ~80% low — and with the "
        "SFH injected that tilt goes straight into Av. Re-run Part 7c with "
        "OVERWRITE_SFH_ARCHIVE=True.")
SFH_ARCH = load_sfh_archive()
print(f"[sfh] archive: {len(SFH_ARCH)} galaxies, mass kind '{_kind}' ({_msrc})")
print("[sfh] injection: each run gets its region's own SFH"
      + (f", sightline collapsed to {SFH_REGION_IL} (licensed by Part 7c2)"
         if SFH_REGION_IL else ", per sightline"))

# ── the umin pins (Part 7d0) ──
_upinf = os.path.join(TABLEDIR, "region_umin_pins.fits")
_UPIN = {}
if os.path.exists(_upinf):
    _ut = Table.read(_upinf)
    for _r in _ut:
        _st = str(_r["pin_status"]).strip()
        if (_st in ("ok", "clamped_lo", "clamped_hi")
                and np.isfinite(float(_r["umin_node"]))):
            _UPIN[(int(_r["snap"]), int(_r["gal_id"]),
                   str(_r["incl"]).strip(), str(_r["region"]).strip())] = \
                float(_r["umin_node"])
    print(f"[umin] {len(_UPIN)} usable pins of {len(_ut)} rows ({_upinf}); "
          f"unpinned rows fall back to the free {len(UMIN_GRID)}-node grid")
else:
    print(f"[umin] {_upinf} missing — run Part 7d0 first; EVERY run falls back "
          "to the free umin grid (dust.mass is then unanchored)")


def _sfh_file_one(run_dir, sid, incl, region, age_myr, shift_myr=0):
    """Region SFH -> run_dir/sfh.fits via cg.write_sfhfromfile.

    Returns (path, hold_frac) or (None, reason) — the writer itself (strict
    1 Myr grid, oldest-Myr shift, all-zero guard) lives in simbanator now.
    """
    entry = SFH_ARCH.get(str(sid), {}).get(incl, {}).get(region)
    if entry is None:
        return None, f"no archived SFH at {region}/{incl}"
    return cg.write_sfhfromfile(run_dir, sid, entry[0], entry[1], age_myr,
                                shift_myr=shift_myr)


def _snap_z(value, grid, fallback=None):
    """SIMBA metallicity -> the single nearest node of a strict CIGALE grid."""
    v = float(value) if np.isfinite(value) else np.nan
    if not np.isfinite(v):
        v = float(fallback) if fallback is not None else Z_FALLBACK
    out = float(cg.nearest_option([v], grid, log=True)[0])
    return out if np.isfinite(out) else float(
        cg.nearest_option([Z_FALLBACK], grid, log=True)[0])


# ── per-region metallicity priors (Part 4c), mass-weighted over the annuli ──
_ztabf = os.path.join(TABLEDIR, "aperture_metallicities.fits")
_ztab = Table.read(_ztabf) if (USE_Z_PRIORS and os.path.exists(_ztabf)) else None
if USE_Z_PRIORS and _ztab is None:
    print(f"[Z priors] {_ztabf} missing — run Part 4c first; every object "
          f"falls back to Z_FALLBACK={Z_FALLBACK}")

# weights for the annulus combination: stellar mass per annulus (Part 7e) and
# gas mass per annulus (Part 8a). Optional — unweighted mean if missing.
_wstar, _wgas = {}, {}
_aptf = os.path.join(TABLEDIR, "aperture_truth.fits")
if os.path.exists(_aptf):
    _wt = Table.read(_aptf)
    for _s2, _g2, _i2, _a2, _v2 in zip(
            np.asarray(_wt["snap"], int), np.asarray(_wt["gal_id"], int),
            np.char.strip(np.asarray(_wt["incl"], str)),
            np.char.strip(np.asarray(_wt["aperture"], str)),
            np.asarray(_wt["mstar"], float)):
        _wstar[(_s2, _g2, _i2, _a2)] = _v2
else:
    print(f"[Z priors] {_aptf} missing — outskirt/cgm Z_star combined "
          "UNWEIGHTED over their annuli (run Part 7e for mass weights)")
_ismf2 = os.path.join(TABLEDIR, "annulus_ism_truth.fits")
if os.path.exists(_ismf2):
    _wt2 = Table.read(_ismf2)
    for _s2, _g2, _i2, _a2, _v2 in zip(
            np.asarray(_wt2["snap"], int), np.asarray(_wt2["gal_id"], int),
            np.char.strip(np.asarray(_wt2["incl"], str)),
            np.char.strip(np.asarray(_wt2["aperture"], str)),
            np.asarray(_wt2["M_gas"], float)):
        _wgas[(_s2, _g2, _i2, _a2)] = _v2
else:
    print(f"[Z priors] {_ismf2} missing — outskirt/cgm Z_gas combined "
          "UNWEIGHTED over their annuli (run Part 8a for gas weights)")


def _z_maps(label, ilab):
    """Part 4c column pair for ONE label -> ({id: Zstar}, {id: Zgas})."""
    if _ztab is None:
        return {}, {}
    m = np.char.strip(np.asarray(_ztab["incl"], str)) == ilab
    ids = [f"snap{int(s):03d}_gal{int(g)}" for s, g in
           zip(np.asarray(_ztab["snap"], int)[m],
               np.asarray(_ztab["gal_id"], int)[m])]
    return (dict(zip(ids, np.asarray(_ztab[f"Zstar_{label}"], float)[m])),
            dict(zip(ids, np.asarray(_ztab[f"Zgas_{label}"], float)[m])))


# which Part 4c columns feed each region: core == the ap3kpc cumulative disc;
# outskirt/cgm combine their (disjoint) annulus columns
_REGION_ZSRC = {"core": ("ap3kpc",),
                "outskirt": ("ann10kpc", "ann32kpc"),
                "cgm": ("ann100kpc",)}


def _region_z_maps(reg, ilab):
    _labs = _REGION_ZSRC[reg]
    if len(_labs) == 1:
        return _z_maps(_labs[0], ilab)
    _parts = [_z_maps(_l, ilab) for _l in _labs]
    _ids = set().union(*[set(p[0]) for p in _parts])
    _zs_out, _zg_out = {}, {}
    for _id in _ids:
        _sn = int(_id.split("_gal")[0][4:])
        _gd = int(_id.split("_gal")[1])
        for _out, _wmap, _pi in ((_zs_out, _wstar, 0), (_zg_out, _wgas, 1)):
            _zv = np.array([_parts[_k][_pi].get(_id, np.nan)
                            for _k in range(len(_labs))], float)
            _wv = np.array([_wmap.get((_sn, _gd, ilab, _l), np.nan)
                            for _l in _labs], float)
            _okz = np.isfinite(_zv) & np.isfinite(_wv) & (_wv > 0)
            if _okz.any():
                _out[_id] = float(np.sum(_zv[_okz] * _wv[_okz])
                                  / np.sum(_wv[_okz]))
            elif np.isfinite(_zv).any():
                _out[_id] = float(np.nanmean(_zv))    # weights missing
    return _zs_out, _zg_out


_ZMAPS = {(_reg, _il): _region_z_maps(_reg, _il)
          for _reg in REGION_LABELS for _il in INCL_LABELS}

# ── 1. every Part 7b2 catalog: cleaned, tagged and stacked ONCE ──────────────
_items, _nocat = [], []
for _arm in FIT_ARMS:
    for _reg in ARM_REGIONS[_arm]:
        for _il in ARM_INCLS[_arm]:
            _f = os.path.join(CIGALE_DIR, f"cigale_{_arm}_{_reg}_{_il}.fits")
            if not os.path.exists(_f):
                _nocat.append(os.path.basename(_f))
                continue
            cg.sanitize_input_errors(_f, verbose=False)   # legacy neg. errors
            _t = Table.read(_f)
            if BROADBANDS_ONLY:
                _t = _t[[c for c in _t.colnames
                         if not _DROP_BAND.search(c.removesuffix("_err"))]]
            # NaN errors -> 10% of the flux (CIGALE adds additionalerror too)
            for _b in [c for c in _t.colnames
                       if c not in ("id", "redshift", "distance")
                       and not c.endswith("_err")]:
                _fl = np.asarray(_t[_b], float)
                _er = np.asarray(_t[f"{_b}_err"], float)
                _bad = np.isfinite(_fl) & ~np.isfinite(_er)
                if _bad.any():
                    _t[f"{_b}_err"][_bad] = 0.1 * np.abs(_fl[_bad])
            _items.append((_t, (_arm, _reg, _il)))
if _nocat:
    print(f"[7b2] {len(_nocat)} catalog(s) missing, e.g. {_nocat[:3]}")
if not _items:
    raise RuntimeError(f"no Part 7b2 catalogs under {CIGALE_DIR} — run 7b2 first")
ALLCAT = cg.stack_cigale_inputs(_items)
_bandcols = [c for c in ALLCAT.colnames
             if c not in ("id", "redshift", "distance")
             and not c.endswith("_err")]
_fitcols = [c for c in _bandcols if c in FIT_BANDS]
print(f"[7b2] {len(_bandcols)} bands kept, {len(_fitcols)} fitted")

# ── 1b. dust_off rows: drop the IR bands from the FIT (2026-08-11) ──────────
# The dust_off SED has no dust emission by construction, but the shared grid
# keeps dl2014, and at IR wavelengths the model-vs-mock residual is the
# BC03-vs-FSPS STELLAR TAIL — factor-level mismatches against additionalerror's
# 10% floor. chi2_min then exceeds ~1.5e3, exp(-chi2/2) underflows to 0 for
# EVERY model and pcigale masks the whole row. The zero-point is an
# optical/NIR statement, so the IR bands are excluded PER ROW (NaN flux ==
# pcigale's per-observation band drop); dust_on/agn_on rows keep them.
_ir_fit = [c for c in _bandcols if cg.IR_BAND_RE.match(c)]
_isoff = np.char.find(np.asarray(ALLCAT["id"], str), "__dust_off_") >= 0
if _isoff.any() and _ir_fit:
    for _b in _ir_fit:
        ALLCAT[_b][_isoff] = np.nan
        ALLCAT[f"{_b}_err"][_isoff] = np.nan
    print(f"[dust_off] {len(_ir_fit)} IR fit bands -> NaN on "
          f"{int(_isoff.sum())} dust_off rows (stellar-tail chi2 underflow "
          "guard; see comment)")

# ── 2. one record per candidate SED row ─────────────────────────────────────
_recs, _skips = [], []
for _i, _rid in enumerate(np.asarray(ALLCAT["id"], str)):
    _r = parse_pin_id(_rid)
    if _r is None:
        raise RuntimeError(f"row id {_rid!r} does not parse — parse_pin_id and "
                           "cg.stack_cigale_inputs' tags have drifted apart")
    _r["row"] = _i
    _r["z"] = float(ALLCAT["redshift"][_i])
    _recs.append(_r)

if PILOT:
    _p_sn = min(r["snap"] for r in _recs)
    _p_g = min(r["gal_id"] for r in _recs if r["snap"] == _p_sn)
    _recs = [r for r in _recs if r["snap"] == _p_sn and r["gal_id"] == _p_g]
    print(f"[PILOT] restricted to snap {_p_sn}, galaxy {_p_g} "
          f"({len(_recs)} rows)")

# fitted-band cut, per ROW (a thin row is dropped; the run still happens)
_keep = []
for _r in _recs:
    _n = int(sum(np.isfinite(float(ALLCAT[c][_r["row"]])) for c in _fitcols))
    if _n < MIN_FIT_BANDS:
        _skips.append({**{k: _r[k] for k in ("arm", "region", "incl",
                                             "snap", "gal_id")},
                       "reason": f"only {_n} finite fit bands"})
    else:
        _keep.append(_r)
_recs = _keep

# ── 3. the pcigale age cap, per snapshot ────────────────────────────────────
_agecap = {}
for _sn in sorted({r["snap"] for r in _recs}):
    _z = float(np.median([r["z"] for r in _recs if r["snap"] == _sn]))
    _age_cos = int(round(COSMO.age(_z).to(u.Myr).value))
    _age_cig = int(np.floor(min(Planck18.age(_z).to(u.Myr).value,
                                Planck18.age(round(_z, 2)).to(u.Myr).value))) - 1
    _age_myr = min(_age_cos, _age_cig)
    _shift = _age_cos - _age_myr
    if _shift > 200:
        raise RuntimeError(
            f"snap {_sn}: the pcigale age cap drops {_shift} Myr of the OLDEST "
            "SFH — that is far beyond the usual few tens of Myr. Check COSMO "
            "vs Planck18 and redshift_decimals before fitting.")
    _agecap[_sn] = (_z, _age_cos, _age_cig, _shift)

# ── 4. group the rows into runs: same region+sightline SFH, Z node, umin pin,
#       chain. dust_on+dust_off share the dust chain (identical grid). ────────
_groups = {}
for _r in _recs:
    _sid, _reg = _r["id"], _r["region"]
    _il_pin = SFH_REGION_IL or _r["incl"]     # SFH + umin pin sightline
    if Z_PIN_LEVEL == "galaxy":
        _pm = _z_maps("ap100kpc", _il_pin)
        _zs_raw, _zg_raw = _pm[0].get(_sid, np.nan), _pm[1].get(_sid, np.nan)
    else:
        _zs_map, _zg_map = _ZMAPS.get((_reg, _il_pin), ({}, {}))
        _zs_raw, _zg_raw = _zs_map.get(_sid, np.nan), _zg_map.get(_sid, np.nan)
    _zs = _snap_z(_zs_raw, _ZS_GRID)
    _r["zs_node"], _r["zg_raw"] = _zs, _zg_raw
    _un = _UPIN.get((_r["snap"], _r["gal_id"], _il_pin, _reg))
    _chain = CHAIN_OF[_r["arm"]]
    _groups.setdefault((_r["snap"], _r["gal_id"], _chain, _reg, _il_pin, _zs,
                        (-1.0 if _un is None else float(_un))), []).append(_r)

print(f"\n{len(_recs)} fittable SED rows -> {len(_groups)} runs")
_nzpin = sum(1 for k in _groups if k[6] >= 0)
print(f"[umin pin] {_nzpin}/{len(_groups)} runs carry a pinned umin node; "
      f"the rest use the free {len(UMIN_GRID)}-node grid")

# ── 5. write one run dir per group ──────────────────────────────────────────
_run_dirs, _holds, _fitted = [], [], []
for _key in sorted(_groups):
    _sn, _gid, _chain, _reg, _il, _zs, _uk = _key
    _rows = _groups[_key]
    _sid = f"snap{_sn:03d}_gal{_gid}"
    _z, _age_cos, _age_cig, _shift = _agecap[_sn]
    _age_myr = min(_age_cos, _age_cig)
    _zk = _ZS_GRID.index(_zs)
    _un = None if _uk < 0 else _uk
    _utag = "free" if _un is None else str(_UOPTS.index(_un))
    _rd = os.path.join(RUN_BASE_REG,
                       f"reg_{_chain}_{_reg}_{_il}_snap{_sn:03d}_gal{_gid}"
                       f"_z{_zk}_u{_utag}")

    _sfh, _extra = _sfh_file_one(_rd, _sid, _il, _reg, _age_myr, _shift)
    if _sfh is None:
        for _r in _rows:
            _skips.append({**{k: _r[k] for k in ("arm", "region", "incl",
                                                 "snap", "gal_id")},
                           "reason": _extra})
        continue
    # keyed by RUN NAME: unique, and cg.collect_results carries it into the
    # results table as 'run'
    _holds.append(dict(run=os.path.basename(_rd), chain=_chain, region=_reg,
                       incl=_il, snap=_sn, gal_id=_gid, nrows=len(_rows),
                       umin_tag=_utag, sfr_hold_frac=_extra))

    # nebular zgas: the group's median, snapped to the nebular grid
    _zg_vals = np.array([_r["zg_raw"] for _r in _rows], float)
    _zg = _snap_z(np.nanmedian(_zg_vals) if np.isfinite(_zg_vals).any()
                  else np.nan, _ZG_GRID, fallback=_zs)

    _objf = os.path.join(_rd, "obj.fits")
    ALLCAT[[_r["row"] for _r in _rows]].write(_objf, overwrite=True)

    _mp = {
        "sfhfromfile": {"filename": _sfh, "sfr_column": [1],
                        "age": [_age_myr], "normalise": True},
        "bc03": {"imf": 1, "metallicity": [_zs]},
        "nebular": {"zgas": [_zg]},
        "dustatt_modified_CF00": {"Av_ISM": AV_ISM_GRID,
                                  "slope_ISM": SLOPE_ISM_GRID,
                                  "mu": MU_GRID,
                                  "slope_BC": SLOPE_BC_GRID},
        "dl2014": {"qpah": QPAH_GRID,
                   "umin": ([_un] if _un is not None else UMIN_GRID),
                   "gamma": GAMMA_GRID},
        "restframe_parameters": {"Dn4000": True, "EW": EW_SPEC},
    }
    _mods = SED_MODULES_DUST
    if _chain == "agn":
        _mods = SED_MODULES_AGN
        _mp["skirtor2016"] = {"fracAGN": FRAC_AGN_GRID, "i": SKIRTOR_I}
    cg.prepare_run(_rd, _objf, sed_modules=_mods, module_params=_mp,
                   analysis_params={"variables": VARIABLES[_chain],
                                    "save_best_sed": True},
                   cores=CORES_PER_TASK, fit_bands=FIT_BANDS, verbose=False)
    _run_dirs.append(_rd)
    _fitted.extend(_rows)
    if len(_run_dirs) % 200 == 0:
        print(f"   ... {len(_run_dirs)} run dirs written")

if not _run_dirs:
    raise RuntimeError("no run dirs prepared — check the Part 7b2 catalogs and "
                       "the Part 7c SFH archive")
_run_dirs = sorted(_run_dirs)

# ── the age cap, auditable ──
print("\npcigale age cap per snapshot (t_SIMBA, t_Planck18 ceiling, dropped):")
for _sn in sorted(_agecap):
    _z, _ac, _ag2, _sh = _agecap[_sn]
    print(f"   snap {_sn:3d}  z={_z:.3f}  t_cos={_ac:6d}  t_cig={_ag2:6d} Myr"
          f"  -> age={min(_ac, _ag2):6d}, oldest {_sh:3d} Myr dropped")

# ── coverage ──
_meta = [parse_pin_run(os.path.basename(d)) for d in _run_dirs]
if any(m is None for m in _meta):
    raise RuntimeError("some run-dir names do not match PIN_RE — Part 7f's "
                       "collect_results would silently drop them")
_nrow = np.array([h["nrows"] for h in _holds], float)
print(f"\n{len(_run_dirs)} run dirs prepared under {RUN_BASE_REG}")
for _c in sorted({m["chain"] for m in _meta}):
    _mult = len(FRAC_AGN_GRID) if _c == "agn" else 1
    _np_ = sum(1 for m in _meta if m["chain"] == _c and m["umin_tag"] != "free")
    _nf_ = sum(1 for m in _meta if m["chain"] == _c and m["umin_tag"] == "free")
    print(f"   {_c:5s} chain: {_np_:4d} pinned runs x "
          f"{_n_att * _n_dust_pin * _mult:,} models + {_nf_:4d} free x "
          f"{_n_att * _n_dust_free * _mult:,}")
print(f"   {int(_nrow.sum()):5d} fitted SEDs, {_nrow.min():.0f}-{_nrow.max():.0f} "
      f"per run (median {np.median(_nrow):.0f})")
print("\nfitted SEDs per (region, arm):")
_cov = {}
for _r in _fitted:
    _cov[(_r["region"], _r["arm"])] = _cov.get((_r["region"], _r["arm"]), 0) + 1
print("   " + "".join(f"{l:>10s}" for l in REGION_LABELS))
for _arm in FIT_ARMS:
    print(f"{_arm:>9s}" + "".join(f"{_cov.get((l, _arm), 0):>10d}"
                                  for l in REGION_LABELS))

SKIP_FITS = os.path.join(TABLEDIR, "cigale_region_skipped.fits")
if _skips:
    _st = Table(rows=_skips, names=("arm", "region", "incl", "snap",
                                    "gal_id", "reason"))
    _st.write(SKIP_FITS, overwrite=True)
    print(f"\n[skipped] {len(_st)} (arm, region, sightline, galaxy) rows "
          f"-> {SKIP_FITS}")
    _u, _c = np.unique(np.asarray(_st["reason"], str), return_counts=True)
    for _r, _n in sorted(zip(_u, _c), key=lambda x: -x[1]):
        print(f"   {_n:5d}  {_r}")
    print("   ^ these SEDs are not fitted. Under the per-region injection the "
          "star-poor cgm dominates here BY DESIGN — core/outskirt losses are "
          "what to actually check.")
else:
    print("\n[skipped] none")

if _holds:
    _hf = np.asarray([h["sfr_hold_frac"] for h in _holds], float)
    Table(rows=_holds).write(os.path.join(TABLEDIR,
                                          "cigale_region_sfrhold.fits"),
                             overwrite=True)
    print(f"[sfr_hold_frac] median {np.median(_hf):.4f}, p95 "
          f"{np.percentile(_hf, 95):.4f} of the tabulated time is edge-held "
          "past the archive's last sample (the smoothed grid ends at the last "
          "bin CENTRE). Part 7f checks it does not correlate with dAv.")

JOB_FILE = cg.write_slurm_array(
    _run_dirs, os.path.join(RUN_BASE_REG, "submit_cigale_regions.job"),
    pcigale_cmd=PCIGALE_CMD, partition=PARTITIONS, cores=CORES_PER_TASK,
    time=WALLTIME, array_throttle=ARRAY_THROTTLE, plots=PLOT_SEDS,
    skip_if_done=SKIP_IF_DONE, job_name="cigale_reg",
    runs_per_task=RUNS_PER_TASK, max_array_tasks=MAX_ARRAY_TASKS)
print(f"\nsbatch {JOB_FILE}")
print(f"  partitions {PARTITIONS}")
print(f"  SKIP_IF_DONE={SKIP_IF_DONE} -> a resubmit only runs what is missing "
      "(False makes pcigale keep a timestamped backup of every out/)")
print("  pre-flight before the full array: cg.check(_run_dirs[0], PCIGALE_CMD) "
      "should report the model count printed above, then time one "
      "cg.run(_run_dirs[0], PCIGALE_CMD) interactively — that time x "
      f"{len(_run_dirs)} is the campaign.")

# Part 7e — aperture-matched SIMBA truth (cluster, cached once)

CIGALE's estimates describe the stars inside **one projected selection along one sightline** —
the global caesar/history M\*/SFR/age are only the right truth for the largest aperture. One
pass over the Stage-0 region cutouts measures, per (galaxy, sightline, aperture rung, annulus
between rungs, **and the three CIGALE regions** `core`/`outskirt`/`cgm` — the region rows are
what Part 7f joins the fits against; the annuli feed the radial profiles):

- **M\*** — current stellar mass in the projected cylinder (matching Hyperion's peeled
  apertures: radius r perpendicular to the (θ, φ) sightline, full depth);
- **archaeological SFR** over the last 25 / 100 Myr — mass formed in the window from
  `StellarFormationTime` (current masses, so ≲10–15 % mass-loss bias — noted, not corrected);
- **mass-weighted age and total Z** of the same stars (→ `stellar.age_m_star`,
  `stellar.metallicity`);
- a **delayed+bq fit to the selection's own archaeological SFH** (50 Myr bins,
  `simbanator.analysis.sfh_utils.fit_delayed_bq`), giving `sfh.tau_main / age_main / age_bq /
  r_sfr` shape truth.

The per-annulus `mstar` also supplies the **mass weights** Part 7d uses to combine the annulus
metallicity columns into region pins. Everything is cached to `tables/aperture_truth.fits`
(one row per galaxy × sightline × aperture/annulus/region; `OVERWRITE_APERTURE_TRUTH=True`
rebuilds, and a cache without annulus or region rows is rebuilt automatically). Part 7f joins
this cache on `(snap, gal_id, incl, aperture)` — the `aperture` column holds the region labels
for the region rows. Requires the Stage-0 particle files and the Stage-1 selection HDF5, so it
runs on the **cluster**.

In [ ]:
# ── Part 7e — cache SIMBA properties in the SAME apertures/annuli/regions/sightlines as the RT ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster).
from simbanator.analysis.sfh_utils import fit_delayed_bq

APERTURE_TRUTH_FITS      = os.path.join(TABLEDIR, "aperture_truth.fits")
OVERWRITE_APERTURE_TRUTH = False
SFR_WINDOWS_MYR = (25.0, 100.0)     # -> sfh.sfr / sfh.sfr100Myrs truth
ARCH_BIN_MYR    = 50.0              # archaeological-SFH bin for the delayed+bq fit
NSTAR_AP_MIN    = 20                # ages/Z/fits need at least this many star particles

APERTURE_TRUTH = None
if os.path.exists(APERTURE_TRUTH_FITS) and not OVERWRITE_APERTURE_TRUTH:
    APERTURE_TRUTH = Table.read(APERTURE_TRUTH_FITS)
    _labs = set(np.char.strip(np.asarray(APERTURE_TRUTH["aperture"], str)))
    if not any(str(a).startswith("ann") for a in _labs):
        print("cache has no annulus rows (pre-annuli version) -> rebuilding")
        APERTURE_TRUTH = None
    elif not set(REGION_LABELS) <= _labs:
        print("cache has no region rows (pre-region version) -> rebuilding")
        APERTURE_TRUTH = None
    else:
        print(f"cached ({len(APERTURE_TRUTH)} rows) -> {APERTURE_TRUTH_FITS}  "
              "(OVERWRITE_APERTURE_TRUTH=True rebuilds)")
if APERTURE_TRUTH is None:
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)               # RT-grid centres (code units)

    _ag = np.linspace(0.02, 1.0, 4096)          # a -> t interpolation grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value                      # Gyr

    _rows = []
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]        # true rung radii [pkpc]
    for _snap in np.unique(SNAPS):
        _zs = float(sim.get_z_from_snap(int(_snap)))
        _t_obs = float(COSMO.age(_zs).value)
        _n_gal = 0
        for _gid in np.unique(IDS[SNAPS == _snap]):
            _cut = read_cutout(_snap, _gid, _cen.get((int(_snap), int(_gid))),
                               "PartType4",
                               fields=("Masses", "StellarFormationTime", "Metallicity"))
            if _cut is None:
                print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre or no stars")
                continue
            _pos = _cut["pos"]
            _mst = np.asarray(_cut["Masses"], float) * 1e10 / _cut["h"]
            _af  = np.asarray(_cut["StellarFormationTime"], float)
            _Zst = np.asarray(_cut["Metallicity"], float)
            _Zst = _Zst[:, 0] if _Zst.ndim == 2 else _Zst
            _tf = np.interp(np.clip(_af, _ag[0], 1.0), _ag, _tg)    # formation time [Gyr]
            _n_gal += 1
            for _il, _nv in zip(INCL_LABELS, NHAT):
                _rproj = projected_radius(_pos, _nv)

                def _measure(_msk, _lab, _r):
                    """Truth of the stars in one projected selection (aperture, annulus OR region)."""
                    _nap = int(_msk.sum())
                    _row = dict(snap=int(_snap), gal_id=int(_gid), incl=_il,
                                aperture=_lab, ap_kpc=float(_r), nstar_ap=_nap,
                                mstar=np.nan, sfr25=np.nan, sfr100=np.nan,
                                age_m_star_myr=np.nan, met_star=np.nan,
                                tau_main_myr=np.nan, age_main_myr=np.nan,
                                age_bq_myr=np.nan, r_sfr=np.nan, fit_r2=np.nan)
                    if _nap:
                        _mm, _tt, _zz = _mst[_msk], _tf[_msk], _Zst[_msk]
                        _row["mstar"] = float(_mm.sum())
                        for _w, _key in zip(SFR_WINDOWS_MYR, ("sfr25", "sfr100")):
                            _row[_key] = float(_mm[_tt >= _t_obs - _w / 1e3].sum()
                                               / (_w * 1e6))
                        if _nap >= NSTAR_AP_MIN:
                            _row["age_m_star_myr"] = float(
                                np.sum(_mm * (_t_obs - _tt)) / _mm.sum() * 1e3)
                            _row["met_star"] = float(np.sum(_mm * _zz) / _mm.sum())
                            _bins = np.arange(_tt.min(), _t_obs + ARCH_BIN_MYR / 1e3,
                                              ARCH_BIN_MYR / 1e3)
                            if _bins.size >= 8:
                                _hm, _ = np.histogram(_tt, bins=_bins, weights=_mm)
                                _tc = 0.5 * (_bins[1:] + _bins[:-1])
                                _fit = fit_delayed_bq(_tc, _hm / (ARCH_BIN_MYR * 1e6),
                                                      _t_obs)
                                if _fit:
                                    for _k2 in ("tau_main_myr", "age_main_myr",
                                                "age_bq_myr", "r_sfr"):
                                        _row[_k2] = float(_fit[_k2])
                                    _row["fit_r2"] = float(_fit["r2"])
                    return _row

                for _lab, _r in zip(APERTURE_LABELS, _apr):     # cumulative rungs
                    _rows.append(_measure(_rproj <= _r, _lab, float(_r)))
                for _ki in range(1, len(_apr)):                 # true annuli (outer rung
                    _rows.append(_measure((_rproj > _apr[_ki - 1])   # names the annulus)
                                          & (_rproj <= _apr[_ki]),
                                          ANNULUS_LABELS[_ki], float(_apr[_ki])))
                for _reg, _dd in REGION_DEFS.items():           # 3 CIGALE regions
                    _mreg = ((_rproj <= _dd["r_out"]) if _dd["r_in"] <= 0 else
                             ((_rproj > _dd["r_in"]) & (_rproj <= _dd["r_out"])))
                    _rows.append(_measure(_mreg, _reg, float(_dd["r_out"])))
        print(f"snap {_snap:3d} (z={_zs:.2f}): aperture+annulus+region truth for {_n_gal} galaxies")
    APERTURE_TRUTH = Table(rows=_rows)
    APERTURE_TRUTH.write(APERTURE_TRUTH_FITS, overwrite=True)
    print(f"{len(APERTURE_TRUTH)} rows ({len(INCL_LABELS)} sightlines x "
          f"[{len(APERTURE_LABELS)} apertures + {len(APERTURE_LABELS) - 1} annuli "
          f"+ {len(REGION_LABELS)} regions]) -> {APERTURE_TRUTH_FITS}")

# Part 7f — did the fit recover $A_V$ (and the anchored dust mass)? (merged results vs SIMBA truth)

Collects every `<run_dir>/out/results.fits` under `cigale_runs_regions/` into one table, joins it
to the region-matched SIMBA truth (Part 7e) and to the **true** attenuation measured per region
*and* per sightline from the `dust_on`/`dust_off` Johnson-V fluxes (region flux =
`F(<r_out) − F(<r_in)` on the rest-frame cumulative catalogs), then writes
`tables/cigale_region_results.fits` plus the figure set.

Run it after the array drains — `collect_results` reports how many runs are missing.

## What is actually measured here

| CIGALE output | status | truth it is checked against |
|---|---|---|
| `attenuation.Av_ISM`, `Av_BC` | **recovered — the measurand** | `A_V_true` = $-2.5\log_{10}(\Delta F^{\rm on}_V/\Delta F^{\rm off}_V)$ per (region, sightline) |
| `stellar.m_star` | **recovered** — the only free normalisation | `mstar` (region rows of Part 7e) |
| `dust.mass` | **anchored** by the Part 7d0 umin pin | region `M_dust` from Part 8a — `dlogMdust` must track `offset_dex` (the pin-closure figure); a run tagged `_ufree` is a genuine (unanchored) recovery instead |
| `agn.fracAGN` (`agn_on`) | **recovered** | `fracAGN_true_V` per region |
| `dust.luminosity` | derived (energy balance) — a restatement of $A_V$ | — |
| `stellar.age_m_star`, `stellar.metallicity` | **PINNED inputs** | region `age_m_star_myr`, `met_star` — closure test, not a recovery |
| `sfh.sfr*` | shape pinned, scale fitted | `sfr25`, `sfr100` |

## The zero-point

The `dust_off` control rows have truth $A_V \equiv 0$, so their recovered `Av_ISM` is the
**BC03-vs-FSPS zero-point**, now measured **per region** (`Av_zp` column, `dAv_zpcorr` =
`dAv` − zero-point). Quote it as a systematic; do not use the global number where the per-region
ones disagree.

## Figures

`av_recovered_vs_true_by_region.png` is the headline. `dustmass_pin_closure.png` is new: fitted
`dust.mass` vs the region truth, coloured by the recorded `offset_dex` — the direct test that the
umin pin anchored the mass. `chi2_distribution.png`: expect the `cgm` rows to sit high — quoted,
not hidden (the energy-balance caveat in the Part 7d markdown).

In [ ]:
# ── Part 7f: merged CIGALE results vs the SIMBA truth (3 regions) ────────────
# Self-contained after Part 0 (+ Part 7e's aperture_truth.fits, Part 8a's
# annulus_ism_truth.fits, Part 7d0's region_umin_pins.fits and the Part 7
# catalogs). One row per (arm, region, sightline, snapshot, galaxy):
#   bayes.*  from <run_dir>/out/results.fits            (the fit)
#   mstar/sfr/age/met from tables/aperture_truth.fits   (Part 7e region rows)
#   A_V_true from the dust_on/dust_off Johnson-V REGION fluxes (differenced
#            cumulative catalogs, per region AND sightline)
#   M_dust_true from Part 8a (region = cumulative-rung difference) — the
#            dust-mass anchor closure of the Part 7d0 umin pin
# Identity arrives on two channels: what is constant over the run (chain, Z
# node, umin tag) from the directory name, and what varies row to row (the RT
# arm) from the source id via parse_pin_id. The truth join keys on the
# 'aperture' column, which holds the REGION label for these rows.
from simbanator.sed import cigale as cg
from simbanator.sed.flux_extraction import attenuation_mag

RUN_BASE_REG = globals().get("RUN_BASE_REG",
                             os.path.join(OUT, "cigale_runs_regions"))
RESULTS_FITS = os.path.join(TABLEDIR, "cigale_region_results.fits")
AV_STATS_FITS = os.path.join(TABLEDIR, "cigale_region_av_stats.fits")
ARMS_PLOT = ("dust_on", "agn_on")
MSUN_KG = 1.98892e30

if "parse_pin_run" not in globals():        # kernel-restart safe (see Part 7d)
    _TAG_RE = re.compile(r"^(dust_on|dust_off|agn_on)_(core|outskirt|cgm)"
                         r"_(i\d+p\d+)$")
    PIN_RE = re.compile(r"^reg_(dust|agn)_(core|outskirt|cgm)_(i\d+p\d+)"
                        r"_snap(\d+)_gal(\d+)_z(\d+)_u(\d+|free)$")

    def parse_pin_run(name):
        m = PIN_RE.match(name)
        if m is None:
            return None
        return dict(chain=m.group(1), region=m.group(2), incl=m.group(3),
                    snap=int(m.group(4)), gal_id=int(m.group(5)),
                    zs_idx=int(m.group(6)), umin_tag=m.group(7))

    def parse_pin_id(row_id):
        sid, tag = cg.parse_stacked_id(str(row_id))
        m = _TAG_RE.match(tag)
        if m is None or "_gal" not in sid:
            return None
        return dict(id=sid, snap=int(sid.split("_gal")[0][4:]),
                    gal_id=int(sid.split("_gal")[1]), arm=m.group(1),
                    region=m.group(2), incl=m.group(3))


def _col(t, name, default=np.nan):
    """Column as a plain float array (masked entries -> default)."""
    if name not in t.colnames:
        return np.full(len(t), default, float)
    c = t[name]
    if hasattr(c, "filled"):
        return np.asarray(c.astype(float).filled(default), float)
    return np.asarray(c, float)


# ── 1. merge every results.fits (one collect per CHAIN keeps columns uniform;
#      the dust chain carries both dust_on and dust_off rows) ──
def _run_consts(name):
    """Only what is genuinely constant over the run — the rest is per row."""
    m = parse_pin_run(name)
    return None if m is None else {k: m[k] for k in ("chain", "zs_idx",
                                                     "umin_tag")}


_parts = []
for _chain in ("dust", "agn"):
    _dirs = sorted(glob.glob(os.path.join(RUN_BASE_REG, f"reg_{_chain}_*")))
    if not _dirs:
        print(f"[{_chain}] no run dirs — skipped")
        continue
    print(f"[{_chain}]", end=" ")
    try:
        _parts.append(cg.collect_results(_dirs, id_map=_run_consts))
    except RuntimeError as _e:
        print(f"   {_e}")
if not _parts:
    raise RuntimeError(f"no results under {RUN_BASE_REG} — has the Part 7d "
                       "array drained? (sbatch submit_cigale_regions.job)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    RES = vstack(_parts, join_type="outer", metadata_conflicts="silent")
print(f"\nmerged: {len(RES)} fits")

# per-ROW identity from the ids (the arm varies inside a run)
_ident = [parse_pin_id(i) for i in np.asarray(RES["id"], str)]
_unp = [i for i, r in zip(np.asarray(RES["id"], str), _ident) if r is None]
if _unp:
    raise RuntimeError(
        f"{len(_unp)} result row id(s) do not parse, e.g. {_unp[:3]} — Part "
        "7d's stack tags and parse_pin_id have drifted apart, and every "
        "downstream join keys off these")
for _k in ("snap", "gal_id", "arm", "incl"):
    RES[_k] = np.array([r[_k] for r in _ident])
# the truth join keys on 'aperture'; for region rows it holds the region label
RES["aperture"] = np.array([r["region"] for r in _ident])
RES["id"] = np.array([r["id"] for r in _ident])   # back to snapNNN_galID
print("   per-row identity from the ids: "
      + ", ".join(f"{a}={int((RES['arm'] == a).sum())}"
                  for a in sorted(set(np.asarray(RES["arm"], str)))))

# ── 2. region-matched SIMBA truth (Part 7e region rows) ──
_aptf = os.path.join(TABLEDIR, "aperture_truth.fits")
if not os.path.exists(_aptf):
    raise RuntimeError(f"{_aptf} missing — run Part 7e first")
APT = Table.read(_aptf)
APT = APT[[str(a).strip() in REGION_LABELS for a in APT["aperture"]]]
if len(APT) == 0:
    raise RuntimeError(f"{_aptf} has no region rows — rerun Part 7e "
                       "(OVERWRITE_APERTURE_TRUTH=True)")
APT = APT["snap", "gal_id", "incl", "aperture", "ap_kpc", "nstar_ap",
          "mstar", "sfr25", "sfr100", "age_m_star_myr", "met_star"]
for _c in ("incl", "aperture"):
    APT[_c] = np.char.strip(np.asarray(APT[_c], str))
    RES[_c] = np.char.strip(np.asarray(RES[_c], str))
TAB = join(RES, APT, keys=("snap", "gal_id", "incl", "aperture"),
           join_type="left")
print(f"joined to region truth: "
      f"{int(np.isfinite(_col(TAB, 'mstar')).sum())}/{len(TAB)} rows matched")

# ── 3. the TRUE attenuation, per REGION and sightline ──
#    Region flux = F(<r_out) - F(<r_in) on the rest-frame cumulative catalogs
#    (linear in flux, so it equals the region SED convolved), then
#    A_lambda = -2.5 log10(dF_on / dF_off); attenuation_mag NaNs non-positive.
ATTEN_BANDS = {"A_V_true": "Johnson.V.V", "A_U_true": "Johnson2.U.U",
               "A_J_true": "2MASS.J.J"}


def _band_map(arm, ap, incl, band):
    f = os.path.join(CATDIR, f"catalog_{arm}_{ap}_{incl}.fits")
    if not os.path.exists(f):
        return None
    t = Table.read(f)
    if band not in t.colnames:
        return None
    return {(int(s), int(g)): float(v)
            for s, g, v in zip(t["snap"], t["gal_id"], t[band])}


def _region_band_map(arm, reg, incl, band):
    _d = REGION_DEFS[reg]
    _out = _band_map(arm, _d["ap_out"], incl, band)
    if _out is None:
        return None
    if _d["ap_in"] is None:
        return _out
    _in = _band_map(arm, _d["ap_in"], incl, band)
    if _in is None:
        return None
    return {k: _out[k] - _in[k] for k in _out.keys() & _in.keys()}


_AV, _AGN = {k: {} for k in ATTEN_BANDS}, {"A_V_true_agn": {}, "fracAGN_true_V": {}}
for _reg in REGION_LABELS:
    for _il in INCL_LABELS:
        for _key, _band in ATTEN_BANDS.items():
            _on = _region_band_map("dust_on", _reg, _il, _band)
            _off = _region_band_map("dust_off", _reg, _il, _band)
            if not (_on and _off):
                continue
            for _k in _on.keys() & _off.keys():
                _AV[_key][(_reg, _il) + _k] = attenuation_mag(_on[_k], _off[_k])
            if _band != ATTEN_BANDS["A_V_true"]:
                continue
            _agn = _region_band_map("agn_on", _reg, _il, _band)
            if not _agn:
                continue
            for _k in _agn.keys() & _off.keys():
                _AGN["A_V_true_agn"][(_reg, _il) + _k] = attenuation_mag(
                    _agn[_k], _off[_k])
            for _k in _agn.keys() & _on.keys():
                _AGN["fracAGN_true_V"][(_reg, _il) + _k] = (
                    1.0 - _on[_k] / _agn[_k] if _agn[_k] > 0 else np.nan)

_keys = [(str(r["aperture"]), str(r["incl"]), int(r["snap"]), int(r["gal_id"]))
         for r in TAB]
for _name, _db in list(_AV.items()) + list(_AGN.items()):
    TAB[_name] = np.array([_db.get(k, np.nan) for k in _keys], float)
TAB["S_UV_true"] = TAB["A_U_true"] - TAB["A_V_true"]     # true curve slope
print(f"true A_V per (region, sightline): "
      f"{int(np.isfinite(_col(TAB, 'A_V_true')).sum())}/{len(TAB)} rows")

# ── 3b. the dust-mass anchor closure (Part 7d0 umin pin) ──
_ismf = os.path.join(TABLEDIR, "annulus_ism_truth.fits")
if os.path.exists(_ismf):
    _ISM = Table.read(_ismf)
    for _c in ("incl", "aperture"):
        _ISM[_c] = np.char.strip(np.asarray(_ISM[_c], str))

    def _md_map(label):
        _m = np.asarray(_ISM["aperture"], str) == label
        return {(int(s), int(g), str(i)): float(v) for s, g, i, v in
                zip(_ISM["snap"][_m], _ISM["gal_id"][_m], _ISM["incl"][_m],
                    np.asarray(_ISM["M_dust"], float)[_m])}

    _mdap = {l: _md_map(l) for l in {d["ap_out"] for d in REGION_DEFS.values()}
             | {d["ap_in"] for d in REGION_DEFS.values() if d["ap_in"]}}
    _mdt = np.full(len(TAB), np.nan)
    for _i, (_reg, _il, _sn, _gd) in enumerate(_keys):
        _d = REGION_DEFS.get(_reg)
        if _d is None:
            continue
        _mo = _mdap[_d["ap_out"]].get((_sn, _gd, _il), np.nan)
        _mi = (_mdap[_d["ap_in"]].get((_sn, _gd, _il), np.nan)
               if _d["ap_in"] else 0.0)
        if np.isfinite(_mo) and np.isfinite(_mi):
            _mdt[_i] = max(_mo - _mi, 0.0)                  # Msun
    TAB["M_dust_true"] = _mdt
    with np.errstate(all="ignore"):
        TAB["dlogMdust"] = (np.log10(_col(TAB, "bayes.dust.mass"))       # kg
                            - np.log10(np.clip(_mdt, 1e-30, None) * MSUN_KG))
else:
    print(f"[dust anchor] {_ismf} missing — dlogMdust not computed")
    TAB["M_dust_true"] = np.nan
    TAB["dlogMdust"] = np.nan
_upinf = os.path.join(TABLEDIR, "region_umin_pins.fits")
if os.path.exists(_upinf):
    _up = Table.read(_upinf)
    _upmap = {(int(r["snap"]), int(r["gal_id"]), str(r["incl"]).strip(),
               str(r["region"]).strip()): (float(r["offset_dex"]),
                                           str(r["pin_status"]).strip())
              for r in _up}
    TAB["umin_offset_dex"] = np.array(
        [_upmap.get((s, g, i, a), (np.nan, ""))[0]
         for (a, i, s, g) in _keys], float)
    TAB["pin_status"] = np.array(
        [_upmap.get((s, g, i, a), (np.nan, "none"))[1]
         for (a, i, s, g) in _keys])
else:
    TAB["umin_offset_dex"] = np.nan
    TAB["pin_status"] = "none"

# ── 4. residuals + the zero-point from the dust_off control ──
AV_CIG = _col(TAB, "bayes.attenuation.Av_ISM")
TAB["dAv"] = AV_CIG - _col(TAB, "A_V_true")
TAB["dlogMstar"] = (np.log10(np.clip(_col(TAB, "bayes.stellar.m_star"), 1e-30, None))
                    - np.log10(np.clip(_col(TAB, "mstar"), 1e-30, None)))
_arm_a = np.asarray(TAB["arm"], str)
_ap_a = np.char.strip(np.asarray(TAB["aperture"], str))
_ctrl = (_arm_a == "dust_off") & np.isfinite(AV_CIG)
AV_ZP = float(np.median(AV_CIG[_ctrl])) if _ctrl.any() else np.nan
AV_ZP_NMAD = float(cg.nmad(AV_CIG[_ctrl])) if _ctrl.any() else np.nan
# the FSPS-vs-BC03 offset follows the stellar population, and the core does
# not hold the same one as the outskirts -> zero point PER REGION
AV_ZP_AP = {}
for _reg in REGION_LABELS:
    _m = _ctrl & (_ap_a == _reg)
    if _m.sum() >= 5:
        AV_ZP_AP[_reg] = (float(np.median(AV_CIG[_m])),
                          float(cg.nmad(AV_CIG[_m])), int(_m.sum()))
TAB.meta["AV_ZP"] = AV_ZP
for _reg, (_z, _nm, _n) in AV_ZP_AP.items():
    TAB.meta[f"AV_ZP_{_reg}"] = _z
TAB["Av_zp"] = np.array([AV_ZP_AP.get(a, (AV_ZP,))[0] for a in _ap_a], float)
TAB["dAv_zpcorr"] = _col(TAB, "dAv") - np.asarray(TAB["Av_zp"], float)
if _ctrl.any():
    print(f"\n[zero-point] dust_off, truth A_V == 0: n={int(_ctrl.sum())}, "
          f"median Av_cig = {AV_ZP:+.3f} +- {AV_ZP_NMAD:.3f} (NMAD)")
    print("   the BC03-vs-FSPS colour offset that a fully pinned population "
          "can only absorb as dust — per region:")
    for _reg in REGION_LABELS:
        if _reg in AV_ZP_AP:
            _z, _nm, _n = AV_ZP_AP[_reg]
            print(f"      {_reg:>9s}  {_z:+.3f} +- {_nm:.3f}   (n={_n})")
    print("   'dAv_zpcorr' in the output table is dAv with the per-region "
          "zero point subtracted.")
else:
    print("\n[zero-point] no dust_off fits found — check FIT_ARMS in Part 7d; "
          "the A_V systematic is then uncalibrated")
TAB.write(RESULTS_FITS, overwrite=True)
print(f"-> {RESULTS_FITS}")

# ── 5. per (arm, region, sightline) summary ──
_rows = []
for _arm in sorted(set(_arm_a.tolist())):
    for _reg in REGION_LABELS:
        for _il in INCL_LABELS:
            _m = ((_arm_a == _arm) & (_ap_a == _reg)
                  & (np.asarray(TAB["incl"], str) == _il))
            _d = _col(TAB, "dAv")[_m]
            _d = _d[np.isfinite(_d)]
            if _d.size == 0:
                continue
            _dm = _col(TAB, "dlogMstar")[_m]
            _dm = _dm[np.isfinite(_dm)]
            _c2 = _col(TAB, "best.reduced_chi_square")[_m]
            _c2 = _c2[np.isfinite(_c2)]
            _rows.append(dict(
                arm=_arm, region=_reg, incl=_il, n=int(_d.size),
                dAv_med=float(np.median(_d)), dAv_nmad=float(cg.nmad(_d)),
                dlogM_med=float(np.median(_dm)) if _dm.size else np.nan,
                dlogM_nmad=float(cg.nmad(_dm)) if _dm.size else np.nan,
                chi2red_med=float(np.median(_c2)) if _c2.size else np.nan))
if _rows:
    Table(rows=_rows).write(AV_STATS_FITS, overwrite=True)
    print(f"-> {AV_STATS_FITS}")
    print(f"\n{'arm':9s} {'region':>9s}   n   median dAv   NMAD   chi2red")
    for _arm in ARMS_PLOT:
        for _reg in REGION_LABELS:
            _s = [r for r in _rows if r["arm"] == _arm and r["region"] == _reg]
            if not _s:
                continue
            _n = sum(r["n"] for r in _s)
            _md = float(np.median([r["dAv_med"] for r in _s]))
            _nm = float(np.median([r["dAv_nmad"] for r in _s]))
            _c2 = float(np.nanmedian([r["chi2red_med"] for r in _s]))
            print(f"{_arm:9s} {_reg:>9s} {_n:4d}   {_md:+9.3f}  {_nm:6.3f}  "
                  f"{_c2:8.2f}")

# ── 6. figures ────────────────────────────────────────────────────────────────
_SNAPC = {s: c for s, c in zip(sorted(set(np.asarray(TAB["snap"], int))),
                               plt.cm.viridis(np.linspace(0.1, 0.9, 4)))}
_MRK = dict(zip(INCL_LABELS, ["o", "s", "^", "D"]))


def _sel(arm=None, reg=None, finite=("A_V_true", "bayes.attenuation.Av_ISM")):
    m = np.ones(len(TAB), bool)
    if arm is not None:
        m &= _arm_a == arm
    if reg is not None:
        m &= _ap_a == reg
    for c in finite:
        m &= np.isfinite(_col(TAB, c))
    return m


def _scatter(ax, m, x, y):
    for _il in INCL_LABELS:
        for _sn, _c in _SNAPC.items():
            k = m & (np.asarray(TAB["incl"], str) == _il) \
                  & (np.asarray(TAB["snap"], int) == _sn)
            if k.any():
                ax.scatter(x[k], y[k], s=13, marker=_MRK[_il], color=_c,
                           alpha=0.7, lw=0.3, edgecolor="k")


_AVT = _col(TAB, "A_V_true")
_DAV = _col(TAB, "dAv")

# 1. HEADLINE — recovered vs true A_V, per arm x region
fig, axs = plt.subplots(len(ARMS_PLOT), len(REGION_LABELS),
                        figsize=(5.4 * len(REGION_LABELS), 4.8 * len(ARMS_PLOT)),
                        squeeze=False, sharex=True, sharey=True)
for _i, _arm in enumerate(ARMS_PLOT):
    for _j, _reg in enumerate(REGION_LABELS):
        ax = axs[_i][_j]
        m = _sel(_arm, _reg)
        _scatter(ax, m, _AVT, AV_CIG)
        _lim = [-0.05, max(1.0, np.nanpercentile(_AVT[m], 99) if m.any() else 1.0)]
        ax.plot(_lim, _lim, "k--", lw=1)
        if np.isfinite(AV_ZP):
            ax.plot(_lim, [v + AV_ZP for v in _lim], ":", color="C3", lw=1)
        if _i == 0:
            ax.set_title(_reg)
        if m.any():
            ax.text(0.04, 0.96, f"n={int(m.sum())}\n"
                    f"med {np.median(_DAV[m]):+.2f}\n"
                    f"NMAD {cg.nmad(_DAV[m]):.2f}", fontsize=9,
                    transform=ax.transAxes, va="top")
        ax.grid(alpha=0.3)
        if _j == 0:
            ax.set_ylabel(f"{_arm}\n" + r"CIGALE $A_V^{\rm ISM}$")
        if _i == len(ARMS_PLOT) - 1:
            ax.set_xlabel(r"true $A_V$ (region)")
fig.suptitle(f"recovered vs true $A_V$ per region  (dotted: zero point {AV_ZP:+.2f})",
             y=1.005)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_recovered_vs_true_by_region.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# 2. residual vs true A_V — the in-sample zero-point estimate
fig, axs = plt.subplots(1, len(ARMS_PLOT), figsize=(8 * len(ARMS_PLOT), 6),
                        squeeze=False, sharey=True)
for _i, _arm in enumerate(ARMS_PLOT):
    ax = axs[0][_i]
    m = _sel(_arm)
    _scatter(ax, m, _AVT, _DAV)
    if m.any():
        _b = np.nanpercentile(_AVT[m], np.linspace(0, 100, 9))
        _b = np.unique(np.round(_b, 4))
        _xc, _md, _sc = [], [], []
        for _lo, _hi in zip(_b[:-1], _b[1:]):
            k = m & (_AVT >= _lo) & (_AVT < _hi)
            if k.sum() >= 5:
                _xc.append(0.5 * (_lo + _hi))
                _md.append(np.median(_DAV[k]))
                _sc.append(cg.nmad(_DAV[k]))
        if _xc:
            ax.plot(_xc, _md, "-", color="C3", lw=2, label="running median")
            ax.fill_between(_xc, np.array(_md) - np.array(_sc),
                            np.array(_md) + np.array(_sc), color="C3", alpha=0.2)
    ax.axhline(0, color="k", ls="--", lw=1)
    if np.isfinite(AV_ZP):
        ax.axhline(AV_ZP, color="C0", ls=":", lw=1.2,
                   label=f"dust_off zero-point {AV_ZP:+.2f}")
    ax.set(xlabel=r"true $A_V$", title=_arm)
    ax.grid(alpha=0.3)
    if _i == 0:
        ax.set_ylabel(r"$\Delta A_V$ (CIGALE $-$ true)")
        ax.legend(fontsize=10, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_residual_vs_true.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 3. residual vs region
fig, ax = plt.subplots(figsize=(10, 6))
_w = 0.34
for _i, _arm in enumerate(ARMS_PLOT):
    _data = [_DAV[_sel(_arm, _reg)] for _reg in REGION_LABELS]
    _data = [d[np.isfinite(d)] for d in _data]
    _pos = np.arange(len(REGION_LABELS)) + (_i - 0.5) * _w
    _bp = ax.boxplot([d if d.size else [np.nan] for d in _data], positions=_pos,
                     widths=_w * 0.85, patch_artist=True, showfliers=False,
                     medianprops=dict(color="k"))
    for _p in _bp["boxes"]:
        _p.set(facecolor=f"C{_i}", alpha=0.55)
    ax.plot([], [], "s", color=f"C{_i}", label=_arm)
ax.axhline(0, color="k", ls="--", lw=1)
if np.isfinite(AV_ZP):
    ax.axhline(AV_ZP, color="C3", ls=":", lw=1.2, label="zero-point")
ax.set(xticks=np.arange(len(REGION_LABELS)), xticklabels=REGION_LABELS,
       ylabel=r"$\Delta A_V$")
ax.legend(fontsize=10, frameon=False)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_residual_vs_region.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 4. the dust-mass anchor closure (NEW): did the umin pin work?
_MDF = _col(TAB, "dlogMdust")
_OFF = _col(TAB, "umin_offset_dex")
m = _sel("dust_on", finite=("dlogMdust",))
if m.any():
    fig, axs = plt.subplots(1, 2, figsize=(15, 6.2))
    for _reg, _co in zip(REGION_LABELS, ("C0", "C1", "C2")):
        k = m & (_ap_a == _reg)
        if k.any():
            axs[0].hist(_MDF[k], bins=30, histtype="step", lw=1.8, color=_co,
                        label=f"{_reg} (med {np.median(_MDF[k]):+.2f})")
    axs[0].axvline(0, color="k", ls="--", lw=1)
    axs[0].set(xlabel=r"$\Delta\log M_{\rm dust}$ (CIGALE $-$ truth) [dex]",
               ylabel="fits", title="dust-mass anchor closure (dust_on)")
    axs[0].legend(fontsize=10, frameon=False)
    k2 = m & np.isfinite(_OFF)
    if k2.any():
        axs[1].scatter(_OFF[k2], _MDF[k2], s=14, alpha=0.6, lw=0.3,
                       edgecolor="k")
        _l = [np.nanmin(_OFF[k2]), np.nanmax(_OFF[k2])]
        axs[1].plot(_l, _l, "k--", lw=1, label="= recorded pin offset")
        axs[1].legend(fontsize=10, frameon=False)
    axs[1].set(xlabel="offset_dex recorded by the Part 7d0 pin",
               ylabel=r"$\Delta\log M_{\rm dust}$",
               title="residual vs the pin's own prediction")
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "dustmass_pin_closure.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# 5. the zero-point control itself
if _ctrl.any():
    fig, ax = plt.subplots(figsize=(10, 6))
    _bins = np.linspace(0, max(0.6, np.nanpercentile(AV_CIG[_ctrl], 99)), 30)
    ax.hist(AV_CIG[_ctrl], bins=_bins, color="0.75", label="all regions")
    for _reg, _c in zip(REGION_LABELS,
                        plt.cm.viridis(np.linspace(0.1, 0.9,
                                                   len(REGION_LABELS)))):
        _m = _ctrl & (_ap_a == _reg)
        if _m.sum() < 5:
            continue
        ax.hist(AV_CIG[_m], bins=_bins, histtype="step", lw=1.6, color=_c,
                label=f"{_reg} ({AV_ZP_AP[_reg][0]:+.3f})")
    ax.axvline(0, color="k", ls="--", lw=1, label="truth ($A_V \\equiv 0$)")
    ax.axvline(AV_ZP, color="C3", lw=2,
               label=f"global {AV_ZP:+.3f} $\\pm$ {AV_ZP_NMAD:.3f}")
    ax.set(xlabel=r"CIGALE $A_V^{\rm ISM}$ on dust-free photometry",
           ylabel="fits", title="zero-point control (dust_off)")
    ax.legend(fontsize=10, frameon=False, ncol=2)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "av_zeropoint_control.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# 6. stellar mass — the one other genuine recovery, and its dust degeneracy
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
m = _sel("dust_on", finite=("mstar", "bayes.stellar.m_star"))
_scatter(axs[0], m, np.log10(np.clip(_col(TAB, "mstar"), 1e-30, None)),
         np.log10(np.clip(_col(TAB, "bayes.stellar.m_star"), 1e-30, None)))
if m.any():
    _l = [np.nanmin(np.log10(np.clip(_col(TAB, "mstar")[m], 1e-30, None))),
          np.nanmax(np.log10(np.clip(_col(TAB, "mstar")[m], 1e-30, None)))]
    axs[0].plot(_l, _l, "k--", lw=1)
axs[0].set(xlabel=r"true $\log M_\star$ (region)",
           ylabel=r"CIGALE $\log M_\star$", title="dust_on")
m2 = _sel("dust_on", finite=("A_V_true", "dlogMstar"))
_scatter(axs[1], m2, _AVT, _col(TAB, "dlogMstar"))
axs[1].axhline(0, color="k", ls="--", lw=1)
axs[1].set(xlabel=r"true $A_V$", ylabel=r"$\Delta \log M_\star$",
           title="mass–dust degeneracy")
for ax in axs:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "mstar_recovery.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 7. closure test on the PINNED inputs — deviation here is a bug, not physics
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (_cig, _tru, _lab) in zip(axs, [
        ("bayes.stellar.age_m_star", "age_m_star_myr",
         r"mass-weighted age [Myr]"),
        ("bayes.stellar.metallicity", "met_star", r"stellar $Z$")]):
    m = _sel("dust_on", finite=(_cig, _tru))
    _scatter(ax, m, _col(TAB, _tru), _col(TAB, _cig))
    if m.any():
        _l = [np.nanmin(_col(TAB, _tru)[m]), np.nanmax(_col(TAB, _tru)[m])]
        ax.plot(_l, _l, "k--", lw=1)
    ax.set(xlabel=f"SIMBA {_lab}", ylabel=f"CIGALE {_lab}")
    ax.grid(alpha=0.3)
fig.suptitle("pinned-input closure (not a recovery)", y=1.0)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "pinned_closure.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 8. the AGN's effect on the dust measurement
if (_arm_a == "agn_on").any():
    fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
    m = _sel("agn_on", finite=("fracAGN_true_V", "bayes.agn.fracAGN"))
    _scatter(axs[0], m, _col(TAB, "fracAGN_true_V"),
             _col(TAB, "bayes.agn.fracAGN"))
    axs[0].plot([0, 1], [0, 1], "k--", lw=1)
    axs[0].set(xlabel=r"true $f_{\rm AGN}$ (V band, $1-F_{\rm on}/F_{\rm agn}$)",
               ylabel=r"CIGALE $f_{\rm AGN}$", title="AGN fraction")
    # per-object dAv(agn_on) - dAv(dust_on)
    _don = {(int(r["snap"]), int(r["gal_id"]), str(r["aperture"]).strip(),
             str(r["incl"]).strip()): _DAV[i]
            for i, r in enumerate(TAB) if _arm_a[i] == "dust_on"}
    _x, _y = [], []
    for i, r in enumerate(TAB):
        if _arm_a[i] != "agn_on":
            continue
        _k = (int(r["snap"]), int(r["gal_id"]), str(r["aperture"]).strip(),
              str(r["incl"]).strip())
        if _k in _don and np.isfinite(_DAV[i]) and np.isfinite(_don[_k]):
            _x.append(_col(TAB, "fracAGN_true_V")[i])
            _y.append(_DAV[i] - _don[_k])
    axs[1].scatter(_x, _y, s=14, alpha=0.7, lw=0.3, edgecolor="k")
    axs[1].axhline(0, color="k", ls="--", lw=1)
    axs[1].set(xlabel=r"true $f_{\rm AGN}$ (V)",
               ylabel=r"$\Delta A_V$(agn_on) $-$ $\Delta A_V$(dust_on)",
               title=r"AGN effect on $\Delta A_V$")
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "agn_bias.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# 9. goodness of fit — expect the cgm rows to sit high (energy-balance caveat)
_C2 = _col(TAB, "best.reduced_chi_square")
if np.isfinite(_C2).any():
    fig, ax = plt.subplots(figsize=(10, 6))
    _bins = np.logspace(np.log10(max(np.nanpercentile(_C2, 1), 1e-2)),
                        np.log10(max(np.nanpercentile(_C2, 99), 1.0)), 40)
    for _i, _reg in enumerate(REGION_LABELS):
        _v = _C2[(_ap_a == _reg) & (_arm_a == "dust_on") & np.isfinite(_C2)]
        if _v.size:
            ax.hist(_v, bins=_bins, histtype="step", lw=1.8, color=f"C{_i}",
                    label=f"{_reg} (median {np.median(_v):.2f})")
    ax.axvline(1.0, color="k", ls="--", lw=1)
    ax.set(xscale="log", xlabel=r"best $\chi^2_{\rm red}$ (dust_on)",
           ylabel="fits")
    ax.legend(fontsize=10, frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "chi2_distribution.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# ── 7. does the edge-held recent SFH bias A_V? (Part 7d's sfr_hold_frac) ──
_holdf = os.path.join(TABLEDIR, "cigale_region_sfrhold.fits")
if os.path.exists(_holdf):
    _ht = Table.read(_holdf)
    _hd = {str(r["run"]).strip(): float(r["sfr_hold_frac"]) for r in _ht}
    _hv = np.array([_hd.get(x, np.nan)
                    for x in np.char.strip(np.asarray(TAB["run"], str))], float)
    _m = np.isfinite(_hv) & np.isfinite(_DAV)
    if not np.isfinite(_hv).any():
        print("\n[sfr_hold_frac] the ledger joined to ZERO fits — its keys and "
              "the run identity have drifted apart. Not fatal, but the "
              "edge-held-SFH check below is void, so fix Part 7d's _holds "
              "before quoting the A_V residual.")
    elif _m.sum() > 20 and np.nanstd(_hv[_m]) > 0:
        print(f"\n[sfr_hold_frac] corr(hold_frac, dAv) = "
              f"{np.corrcoef(_hv[_m], _DAV[_m])[0, 1]:+.3f} over "
              f"{int(_m.sum())} fits (|r| < 0.2 -> the edge-held recent SFH is "
              "not driving the A_V residual)")

# ── 8. what the array did not deliver ──
_skipf = os.path.join(TABLEDIR, "cigale_region_skipped.fits")
if os.path.exists(_skipf):
    _sk = Table.read(_skipf)
    print(f"\n[coverage] {len(_sk)} (arm, region, sightline, galaxy) "
          f"combinations were never prepared (see {os.path.basename(_skipf)}); "
          f"{len(TAB)} fits are in hand.")
    _u, _c = np.unique(np.asarray(_sk["reason"], str), return_counts=True)
    for _r, _n in sorted(zip(_u, _c), key=lambda x: -x[1]):
        print(f"   {_n:5d}  {_r}")
    _skr = np.char.strip(np.asarray(_sk["region"], str))
    print("   per region: " + "  ".join(
        f"{_reg}:{int((_skr == _reg).sum())}" for _reg in REGION_LABELS)
        + "   (cgm dominating is by design; core/outskirt losses are the "
          "ones to chase)")

# Part 8 — Red cores: surviving ISM dust vs AGN coupling class

Where does dust survive quenching, and does the AGN's coupling to the ISM set how red the core
ends up? The chain runs: SIMBA ISM truth (surviving dust/H$_2$, 8a) → RT differential attenuation
(dust_on − dust_off, 8b — the **internal truth ladder**, *not* a claim about what an observer
sees) → the observable (pinned-CIGALE $A_V$, Part 7f). Six tests:

- **T1 (decomposition, 8c)** — is the red core dust at all? $\Delta(U\!-\!V)_{\rm dust}(R) =
  (U\!-\!V)_{\rm on} - (U\!-\!V)_{\rm off}$ isolates the dust part of the colour gradient; the
  dust_off gradient is the intrinsic (age/$Z$) part.
- **T2 (sufficiency, 8d)** — does the surviving dust column carry the attenuation? Annular $A_V$
  vs $\Sigma_{\rm dust}$ against the foreground-screen ceiling $A_{V,\rm screen} =
  1.086\,\kappa_V\,\Sigma_{\rm dust}$, with $\kappa_V$ read from the RT's own KMH94 dust model.
- **T3 (the observable clock, 8e)** — binned $\langle A_V^{\rm CIGALE}\rangle$ (zero-point
  corrected) at the **two fitted cumulative apertures** ($<3.2$ and $<32$ kpc — real fits, unlike
  the annuli), each vs the mass-weighted stellar age of the *same* aperture, one confounder per
  panel: stellar-mass halves, redshift anchors, coupling class. Plain binned medians with
  galaxy-bootstrap bands — **no rank statistics anywhere in 8e–8g**.
- **T4 (structure, 8f)** — where the dust (and H$_2$) sits: rows = RT $A_V$,
  $\log\Sigma_{\rm dust}$, $\log\Sigma_{\rm H_2}$; columns = core ($<3.2$ kpc) and outskirt
  annulus (10–32 kpc), each vs its own local stellar age, plus the radial Theil–Sen slope over
  the 5-annulus profile vs the total age. Core and outskirt share each row's y-range, so the
  gradient is read directly; matching rows show whether the $A_V$ structure simply follows the
  surviving columns.
- **T5 (the dust clock, 8g)** — everything vs the time elapsed since the quench end $t_{\rm QT}$:
  top row the attenuation clock ($A_V^{\rm CIGALE}$ at $<3.2$ and $<32$ kpc + core RT $A_V$),
  bottom row the ISM clock ($\log\Sigma_{\rm dust}$, $\log M_{\rm dust}/M_\star$,
  $\log M_{\rm H_2}/M_\star$ in the core). If coupling clears dust faster, the strong track must
  fall more steeply than the weak one — one Theil–Sen slope per class, with galaxy-bootstrap CIs.
  Fit *fidelity* is Part 7f's job and is not retested here.

- **T6 (the shape clock, 8h)** — does attenuation *peak* in the centre, and how fast does
  the peak erode? Stacked $A_V(R)$ and $\Delta(U\!-\!V)_{\rm dust}(R)$ profiles in terciles
  of time since quenching, split low-z/high-z (8h1); the concentration $C_{A_V}$ (core $-$
  outskirt annulus, $\geq 2$ finite sightlines) with one Theil–Sen erosion rate per z half
  plus the observable analogue (8h2); and the direct test that surviving core dust — not
  elapsed time — is what keeps cores red (8h3).

Inputs: **8a** builds `tables/annulus_ism_truth.fits` — the per-aperture/annulus dust, gas and
cold-gas truth that Part 4b only *counted*; **8b** generalises Part 7a's annular $A_V$ (and adds
the annular $(U\!-\!V)$ colours) to all four sightlines; **8e** additionally reads the
per-aperture stellar ages from Part 7e's `aperture_truth.fits`, the fitted $A_V$ from Part 7f and
$t_{\rm QT}$ from the selection FITS, and collapses everything to **one row per galaxy**
(table `G8`, sightline nanmedians) that 8f/8g reuse. 8c–8h are pure reads of the caches.

**Caveats.** Annuli are never fitted with CIGALE (energy balance — closed decision, see the run
order): the CIGALE $A_V$ therefore lives on *cumulative* apertures ($<3.2$, $<32$ kpc), while the
outskirt annulus and the radial slope are the RT differential truth. Ages come from
`aperture_truth.fits` and need ≥ 20 star particles in the aperture/annulus. Axes use robust
Tukey-fence ranges (quartiles ± 1.7 IQR): a handful of extreme outliers can sit outside the view
but still enter every statistic; positive-definite quantities are $\log_{10}$ with zeros dropped
as NaN (no fake floors). The four sightlines of one galaxy are not independent — 8e collapses
every quantity to a per-galaxy median before any statistic, and all CIs are bootstrapped over
galaxies. The tiny `no_AGN`/`no_event` classes are dropped from every class-split panel.


In [ ]:
# ── Part 8a — per-aperture/annulus ISM truth: dust, gas, cold gas (cluster, cached) ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (Part 4). Part 4b only
# COUNTED particles per projected annulus; this cell sums their masses — the
# dust/gas truth the red-core tests join against. Same geometry as Parts 4b/4c/7e.
# dust/HI/H2 split + temperature come verbatim from build_profiles_job.py (repo root).
from build_profiles_job import _components, _temperature, _XH, T_COLD

ANNULUS_ISM_FITS      = os.path.join(TABLEDIR, "annulus_ism_truth.fits")
OVERWRITE_ANNULUS_ISM = False
KAPPA_V_FALLBACK = 3.0e4                            # cm^2 per g of DUST (MW-like)
KAPPA_UM = {"U": 0.365, "V": 0.551, "J": 1.235}     # rest-frame pivots [um]

def _kappa_rt():
    """Extinction opacity chi [cm^2 per g of DUST] of the RT dust model.

    Resolved exactly as parameters_master.py does: $POWDERDAY_ROOT (else ~) /
    hyperion-dust/dust_files/kmh94_3.1_hg.hdf5; chi is tabulated vs frequency.
    Falls back to a MW-like constant (Part 8d's screen then carries a ~x2 model
    uncertainty and says so).
    """
    _df = os.path.join(os.environ.get("POWDERDAY_ROOT", os.path.expanduser("~")),
                       "hyperion-dust", "dust_files", "kmh94_3.1_hg.hdf5")
    try:
        with h5py.File(_df, "r") as f:
            _nu  = np.asarray(f["optical_properties"]["nu"][:], float)
            _chi = np.asarray(f["optical_properties"]["chi"][:], float)
        _lam = 2.99792458e14 / _nu                  # c [um s^-1] / nu -> um
        _o = np.argsort(_lam)
        return ({b: float(np.interp(l, _lam[_o], _chi[_o])) for b, l in KAPPA_UM.items()},
                os.path.basename(_df))
    except (OSError, KeyError) as _e:
        print(f"[kappa] {_df} unreadable ({_e}) -> MW-like fallback "
              f"kappa_V={KAPPA_V_FALLBACK:g} cm2/g (A_U/A_V=1.55, A_J/A_V=0.28)")
        return ({"U": 1.55 * KAPPA_V_FALLBACK, "V": KAPPA_V_FALLBACK,
                 "J": 0.28 * KAPPA_V_FALLBACK}, "fallback")

ANNULUS_ISM = None
if os.path.exists(ANNULUS_ISM_FITS) and not OVERWRITE_ANNULUS_ISM:
    ANNULUS_ISM = Table.read(ANNULUS_ISM_FITS)
    print(f"cached ({len(ANNULUS_ISM)} rows) -> {ANNULUS_ISM_FITS}  "
          "(OVERWRITE_ANNULUS_ISM=True rebuilds)")
if ANNULUS_ISM is None:
    KAPPA, KAPPA_SRC = _kappa_rt()
    print(f"RT dust opacity [{KAPPA_SRC}]: "
          + "  ".join(f"kappa_{b}={v:.4g} cm2/g" for b, v in KAPPA.items()))
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)
    _G_FIELDS = ("Masses", "Dust_Masses", "Metallicity", "NeutralHydrogenAbundance",
                 "FractionH2", "InternalEnergy", "ElectronAbundance", "StarFormationRate")
    _rows, _skipped, _no_thermo = [], [], 0
    for _s, _g in zip(SNAPS, IDS):
        _cut = read_cutout(_s, _g, _cen.get((int(_s), int(_g))), "PartType0",
                           fields=_G_FIELDS)
        if _cut is None:
            _skipped.append((int(_s), int(_g)))
            continue
        _hh = _cut["h"]
        _m  = np.asarray(_cut["Masses"], float) * 1e10 / _hh
        _mdp = (np.asarray(_cut["Dust_Masses"], float) * 1e10 / _hh
                if _cut["Dust_Masses"] is not None else np.zeros_like(_m))
        _Z    = (np.asarray(_cut["Metallicity"], float)
                 if _cut["Metallicity"] is not None else None)
        _fnt  = (np.asarray(_cut["NeutralHydrogenAbundance"], float)
                 if _cut["NeutralHydrogenAbundance"] is not None else None)
        _fmol = (np.asarray(_cut["FractionH2"], float)
                 if _cut["FractionH2"] is not None else None)
        _sfr  = (np.asarray(_cut["StarFormationRate"], float)
                 if _cut["StarFormationRate"] is not None else np.zeros_like(_m))
        _mdust, _mHI, _mH2 = _components(_m, _mdp, _Z, _fnt, _fmol)
        if _cut["InternalEnergy"] is not None and _cut["ElectronAbundance"] is not None:
            _T = _temperature(np.asarray(_cut["InternalEnergy"], float),
                              np.asarray(_cut["ElectronAbundance"], float),
                              _XH(_Z, len(_m)))
        else:                                       # cold cut degrades to the SFR gate
            _T, _no_thermo = np.full(len(_m), np.nan), _no_thermo + 1
        # SF gas sits on the effective EOS (its T is not physical) -> count it
        # cold; M_sf is ledgered separately so the choice stays auditable
        _cold = (_T < T_COLD) | (_sfr > 0)
        for _j, _il in enumerate(INCL_LABELS):
            _R = projected_radius(_cut["pos"], NHAT[_j])

            def _measure(_msk, _lab, _rin, _rout):
                _area = np.pi * (_rout**2 - _rin**2)
                _mg, _mdu = float(_m[_msk].sum()), float(np.nansum(_mdust[_msk]))
                _mhi, _mh2 = float(np.nansum(_mHI[_msk])), float(np.nansum(_mH2[_msk]))
                _row = dict(snap=int(_s), gal_id=int(_g), incl=_il, aperture=_lab,
                            r_in_kpc=float(_rin), r_out_kpc=float(_rout),
                            area_kpc2=float(_area),
                            ngas=int(_msk.sum()),
                            ndust=int((_mdp[_msk] > 0).sum()),   # Part 4b convention
                            ncold=int(_cold[_msk].sum()),
                            M_gas=_mg, M_dust=_mdu, M_HI=_mhi, M_H2=_mh2,
                            M_cold=float(_m[_msk & _cold].sum()),
                            M_sf=float(_m[_msk & (_sfr > 0)].sum()),
                            Sigma_dust=_mdu / _area, Sigma_gas=_mg / _area,
                            Sigma_HI=_mhi / _area, Sigma_H2=_mh2 / _area)
                _row["DGR"]    = _mdu / _mg if _mg > 0 else np.nan
                _row["f_cold"] = _row["M_cold"] / _mg if _mg > 0 else np.nan
                _row["f_mol_ann"] = _mh2 / (_mh2 + _mhi) if (_mh2 + _mhi) > 0 else np.nan
                return _row

            for _k, _lab in enumerate(APERTURE_LABELS):              # cumulative rungs
                _rows.append(_measure(_R <= R_EDGES[_k + 1], _lab, 0.0, R_EDGES[_k + 1]))
            for _k, _lab in enumerate(ANNULUS_LABELS[1:], start=1):  # true annuli
                _rows.append(_measure((_R > R_EDGES[_k]) & (_R <= R_EDGES[_k + 1]),
                                      _lab, R_EDGES[_k], R_EDGES[_k + 1]))
    ANNULUS_ISM = Table(rows=_rows)
    ANNULUS_ISM.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
    ANNULUS_ISM.meta["T_COLD"]  = T_COLD
    ANNULUS_ISM.meta["KAP_SRC"] = KAPPA_SRC
    for _b, _v in KAPPA.items():
        ANNULUS_ISM.meta[f"KAPPA_{_b}"] = _v
    ANNULUS_ISM.write(ANNULUS_ISM_FITS, overwrite=True)
    print(f"{len(ANNULUS_ISM)} rows ({len(ANNULUS_ISM) // (N_INCL * (2 * N_AP - 1))} galaxies x "
          f"{N_INCL} sightlines x {2 * N_AP - 1} labels) -> {ANNULUS_ISM_FITS}")
    if _skipped:
        print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
    if _no_thermo:
        print(f"[WARN] {_no_thermo} galaxies lack InternalEnergy/ElectronAbundance -> "
              "their cold cut is the SFR>0 gate only")

# ── QC: sampling per label + galaxy-level DGR against Part 7a ──
_labs_qc = list(APERTURE_LABELS) + ANNULUS_LABELS[1:]
print(f"\n{'label':>10s} {'med Sig_dust':>13s} {'ngas<10':>8s} {'Mdust=0':>8s}")
for _lab in _labs_qc:
    _t = ANNULUS_ISM[np.asarray(ANNULUS_ISM["aperture"], str) == _lab]
    print(f"{_lab:>10s} {np.nanmedian(np.asarray(_t['Sigma_dust'], float)):13.3g} "
          f"{np.mean(np.asarray(_t['ngas'], int) < 10) * 100:7.0f}% "
          f"{np.mean(np.asarray(_t['M_dust'], float) == 0) * 100:7.0f}%")
# membership differs (100 pkpc cutout sphere vs caesar glist): ~<0.1 dex is
# expected agreement, >0.3 dex means a units bug, not physics
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _dgr7a = {(int(s), int(g)): float(d)
              for s, g, d in zip(_at["snap"], _at["gal_id"], _at["DGR"])}
    _t100 = ANNULUS_ISM[(np.asarray(ANNULUS_ISM["aperture"], str) == "ap100kpc")
                        & (np.asarray(ANNULUS_ISM["incl"], str) == INCL_LABELS[0])]
    with np.errstate(all="ignore"):
        _dl = np.array([abs(np.log10(float(_r["DGR"]))
                            - np.log10(_dgr7a.get((int(_r["snap"]), int(_r["gal_id"])),
                                                  np.nan)))
                        for _r in _t100])
    print(f"\n[DGR cross-check vs Part 7a] median |dlog DGR| = {np.nanmedian(_dl):.3f} dex "
          f"over {int(np.isfinite(_dl).sum())} galaxies (expect <~0.1; >0.3 = units bug)")
else:
    print("\n[DGR cross-check] attenuation_vs_ism.fits not found — run Part 7a to enable it")

In [ ]:
# ── Part 8b — annular A_V and (U-V) colours for ALL four sightlines (cached) ──
# Part 7a builds the annular A_V for the fiducial sightline only; the red-core
# tests need it per (label, sightline) to join the 8a ISM truth row-for-row.
# Same differential construction: F_ann = F(<r_out) - F(<r_in), band by band
# (filter convolution is linear in flux). Catalog fluxes are mJy f_nu, so
# -2.5 log10(F_U/F_V) is directly an AB colour.
from simbanator.sed.flux_extraction import attenuation_mag

ANNULUS_AV_FITS      = os.path.join(TABLEDIR, "annulus_av_allincl.fits")
OVERWRITE_ANNULUS_AV = False
_AV_BANDS = {"U": "Johnson2.U.U", "V": "Johnson.V.V", "J": "2MASS.J.J"}  # rest-frame

ANNULUS_AV = None
if os.path.exists(ANNULUS_AV_FITS) and not OVERWRITE_ANNULUS_AV:
    ANNULUS_AV = Table.read(ANNULUS_AV_FITS)
    print(f"cached ({len(ANNULUS_AV)} rows) -> {ANNULUS_AV_FITS}  "
          "(OVERWRITE_ANNULUS_AV=True rebuilds)")
if ANNULUS_AV is None:
    SEL, SNAPS, IDS = load_selection()
    _keys = [(int(s), int(g)) for s, g in zip(SNAPS, IDS)]
    _rows = []
    for _il in INCL_LABELS:
        # (n_gal, n_ap) cumulative flux matrices, one catalog read per (arm, rung)
        _M, _miss = {}, []
        for _arm in ("dust_on", "dust_off"):
            for _k, _lab in enumerate(APERTURE_LABELS):
                _f = os.path.join(CATDIR, f"catalog_{_arm}_{_lab}_{_il}.fits")
                if not os.path.exists(_f):
                    _miss.append(os.path.basename(_f))
                    continue
                _t = Table.read(_f)
                _idx = {(int(s), int(g)): _j
                        for _j, (s, g) in enumerate(zip(_t["snap"], _t["gal_id"]))}
                for _b, _col in _AV_BANDS.items():
                    _mat = _M.setdefault((_arm, _b),
                                         np.full((len(_keys), N_AP), np.nan))
                    if _col not in _t.colnames:
                        continue
                    _v = np.asarray(_t[_col], float)
                    for _i, _key in enumerate(_keys):
                        _j = _idx.get(_key)
                        if _j is not None:
                            _mat[_i, _k] = _v[_j]
        if _miss:
            print(f"[{_il}] {len(_miss)} catalogs missing (e.g. {_miss[0]}) -> partial")
        if not _M:
            continue
        # cumulative -> annular; column 0 (the 0->1 kpc disc) IS ann1kpc == ap1kpc
        _D = {k: np.column_stack([v[:, :1], np.diff(v, axis=1)]) for k, v in _M.items()}
        for _which, _F in (("ap", _M), ("ann", _D)):
            for _k, _lab in enumerate(APERTURE_LABELS):
                if _which == "ann" and _k == 0:
                    continue                       # ann1kpc == ap1kpc: keep one copy
                _lab_out = _lab if _which == "ap" else ANNULUS_LABELS[_k]
                _fon  = {b: _F[("dust_on", b)][:, _k] for b in _AV_BANDS}
                _foff = {b: _F[("dust_off", b)][:, _k] for b in _AV_BANDS}
                with np.errstate(all="ignore"):
                    _uv_on  = np.where((_fon["U"] > 0) & (_fon["V"] > 0),
                                       -2.5 * np.log10(_fon["U"] / _fon["V"]), np.nan)
                    _uv_off = np.where((_foff["U"] > 0) & (_foff["V"] > 0),
                                       -2.5 * np.log10(_foff["U"] / _foff["V"]), np.nan)
                _A = {b: attenuation_mag(_fon[b], _foff[b]) for b in _AV_BANDS}
                for _i, (_sn, _gd) in enumerate(_keys):
                    _rows.append(dict(snap=_sn, gal_id=_gd, incl=_il, aperture=_lab_out,
                                      A_U=float(_A["U"][_i]), A_V=float(_A["V"][_i]),
                                      A_J=float(_A["J"][_i]),
                                      S_UV=float(_A["U"][_i] - _A["V"][_i]),
                                      UV_on=float(_uv_on[_i]), UV_off=float(_uv_off[_i]),
                                      F_V_on=float(_fon["V"][_i]),
                                      F_V_off=float(_foff["V"][_i])))
    if not _rows:
        raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
    ANNULUS_AV = Table(rows=_rows)
    ANNULUS_AV.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
    ANNULUS_AV.write(ANNULUS_AV_FITS, overwrite=True)
    print(f"{len(ANNULUS_AV)} rows -> {ANNULUS_AV_FITS}")

# ── closure: the fiducial sightline must reproduce Part 7a's stored A_V_ann_* ──
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _fid = ANNULUS_AV[np.char.strip(np.asarray(ANNULUS_AV["incl"], str)) == INCL_LABELS[0]]
    _mine = {(int(_r["snap"]), int(_r["gal_id"]), str(_r["aperture"]).strip()):
             float(_r["A_V"]) for _r in _fid}
    _dmax, _ncmp = 0.0, 0
    for _r in _at:
        for _k, _lab in enumerate(APERTURE_LABELS):
            if f"A_V_ann_{_lab}" not in _at.colnames:
                continue
            _lab_o = _lab if _k == 0 else ANNULUS_LABELS[_k]     # 7a names by the rung
            _v7a = float(_r[f"A_V_ann_{_lab}"])
            _v8b = _mine.get((int(_r["snap"]), int(_r["gal_id"]), _lab_o), np.nan)
            if np.isfinite(_v7a) and np.isfinite(_v8b):
                _dmax, _ncmp = max(_dmax, abs(_v7a - _v8b)), _ncmp + 1
    assert _ncmp == 0 or _dmax < 1e-6, \
        f"8b drifted from Part 7a's annular A_V (max |dA_V| = {_dmax:.2e} over {_ncmp})"
    print(f"closure vs Part 7a ({INCL_LABELS[0]}): max |dA_V_ann| = {_dmax:.2e} "
          f"over {_ncmp} annuli — OK")

# NaN'd annuli (non-positive differential flux = MC noise / empty annulus) are a
# mild TRANSPARENCY selection: report per label so T2 can quote it
_ap8b = np.char.strip(np.asarray(ANNULUS_AV["aperture"], str))
print(f"\n{'label':>10s} {'A_V NaN':>8s} {'(U-V)_on NaN':>13s}")
for _lab in list(APERTURE_LABELS) + ANNULUS_LABELS[1:]:
    _t = ANNULUS_AV[_ap8b == _lab]
    if len(_t):
        print(f"{_lab:>10s} {np.mean(~np.isfinite(np.asarray(_t['A_V'], float))) * 100:7.0f}% "
              f"{np.mean(~np.isfinite(np.asarray(_t['UV_on'], float))) * 100:12.0f}%")

In [ ]:
# ── Part 8c — master red-core table + T1: is the red core dust at all? ──
# Pure read of the 8a/8b caches + selection metadata. The dust_off colour profile
# is the INTRINSIC (age/Z) gradient; dust_on - dust_off is the dust contribution.
# f_dust_color = the fraction of the central (U-V) excess that vanishes without dust.
from scipy.stats import mannwhitneyu

NGAS_ANN_MIN  = 10          # Sigma/DGR/f_cold need >= this many gas particles
N_BOOT        = 2000        # galaxy-level bootstrap resamples
CORE_LABEL    = "ap3kpc"    # headline core: the 0-3.16 kpc disc (best sampled)
CORE_ECHO     = "ap1kpc"    # echo: the innermost disc (~3-4 softenings, noisy)
OUT_REF_LABEL = "ann32kpc"  # outer reference annulus (10-31.6 kpc)
RED_CORE_MIN  = 0.05        # mag; a 'red core' must exceed this central excess
AGN_COLORS = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
              "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7",
              "star_forming": "#16a085"}
CLS4 = ["strong", "intermediate", "weak", "no_AGN"]

_A8 = Table.read(os.path.join(TABLEDIR, "annulus_ism_truth.fits"))
_B8 = Table.read(os.path.join(TABLEDIR, "annulus_av_allincl.fits"))
for _t in (_A8, _B8):
    for _c in ("incl", "aperture"):
        _t[_c] = np.char.strip(np.asarray(_t[_c], str))
P8 = join(_A8, _B8, keys=("snap", "gal_id", "incl", "aperture"),
          metadata_conflicts="silent")
print(f"join on (snap, gal_id, incl, aperture): {len(P8)} rows "
      f"(ISM {len(_A8)}, A_V {len(_B8)})"
      + ("" if len(P8) == min(len(_A8), len(_B8)) else "  [rows lost — check labels]"))

# galaxy metadata (class labels, mass, size flags) from the selection catalog
SEL, _, _ = load_selection()
_md = {(int(_r["snap"]), int(_r["gal_id"])): _r for _r in SEL}
_pk = list(zip(np.asarray(P8["snap"], int), np.asarray(P8["gal_id"], int)))

def _selnum(row, col):
    # tolerate masked values and columns absent from older selection builds
    if col not in row.colnames:
        return np.nan
    _v = row[col]
    return np.nan if np.ma.is_masked(_v) else float(_v)

_want = ("log_mstar", "xstr_quench", "r50_star_kpc", "flag_too_large")
_absent = [c for c in _want if c not in SEL.colnames]
if _absent:
    print(f"[meta] selection FITS lacks {_absent} -> NaN "
          "(Part 3b was not re-run after the last selection rebuild)")
for _c in _want:
    P8[_c] = np.array([_selnum(_md[_k], _c) if _k in _md else np.nan for _k in _pk])
P8["agn_class"] = np.array(
    [(lambda a: a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a))
     (_md[_k]["agn_class"]) if _k in _md else "unclassified" for _k in _pk])
_z_of_snap = {int(s): float(sim.get_z_from_snap(int(s))) for s in np.unique(P8["snap"])}
P8["z_target"] = np.array([_z_of_snap[int(s)] for s in P8["snap"]])

# QC mask: surface densities and ratios are shot-noise garbage below NGAS_ANN_MIN
_lowN = np.asarray(P8["ngas"], int) < NGAS_ANN_MIN
for _c in ("Sigma_dust", "Sigma_gas", "Sigma_HI", "Sigma_H2", "DGR", "f_cold", "f_mol_ann"):
    _v = np.asarray(P8[_c], float)
    _v[_lowN] = np.nan
    P8[_c] = _v
_ap8 = np.asarray(P8["aperture"], str)
print(f"ngas < {NGAS_ANN_MIN}: Sigma/DGR/f_cold masked on {int(_lowN.sum())}/{len(P8)} rows — "
      + "  ".join(f"{_l}:{np.mean(_lowN[_ap8 == _l]) * 100:.0f}%"
                  for _l in ["ap1kpc"] + ANNULUS_LABELS[1:]))

# ── shared inference helpers (used by 8c-8g) ──
_GKEY = np.array([f"{s}_{g}" for s, g in zip(P8["snap"], P8["gal_id"])])

def _gboot_idx(keys, statfn, n=N_BOOT, seed=0):
    """Cluster bootstrap over GALAXIES: statfn(row_indices) -> (point, lo16, hi84).
    All sightline rows of a resampled galaxy ride along together."""
    keys = np.asarray(keys)
    _uk = np.unique(keys)
    _where = {k: np.where(keys == k)[0] for k in _uk}
    _rng = np.random.default_rng(seed)
    _pt = statfn(np.arange(len(keys)))
    _s = np.array([statfn(np.concatenate(
        [_where[k] for k in _rng.choice(_uk, len(_uk), replace=True)])) for _ in range(n)])
    return float(_pt), float(np.nanpercentile(_s, 16)), float(np.nanpercentile(_s, 84))

def _gboot_med(vals, keys, **kw):
    vals = np.asarray(vals, float)
    return _gboot_idx(keys, lambda i: (np.nanmedian(vals[i])
                                       if np.isfinite(vals[i]).any() else np.nan), **kw)

def _gal_median(vals, keys):
    """Per-galaxy median over sightlines -> (per-gal values, per-gal keys)."""
    vals, keys = np.asarray(vals, float), np.asarray(keys)
    _uk = np.unique(keys)
    return np.array([np.nanmedian(vals[keys == k]) for k in _uk]), _uk

def _cls_of_keys(keys):
    _c = {k: c for k, c in zip(_GKEY, P8["agn_class"])}
    return np.array([_c[k] for k in keys])

# ── T1: decompose the central colour excess into dust vs intrinsic ──
def _lab_map(col, lab):
    _m = _ap8 == lab
    return {(k, str(i)): float(v)
            for k, i, v in zip(_GKEY[_m], P8["incl"][_m], np.asarray(P8[col], float)[_m])}

_T1 = {}
for _core in (CORE_LABEL, CORE_ECHO):
    _con, _cof = _lab_map("UV_on", _core), _lab_map("UV_off", _core)
    _oon, _oof = _lab_map("UV_on", OUT_REF_LABEL), _lab_map("UV_off", OUT_REF_LABEL)
    _rows = []
    for _k in _con:
        _dt = _con[_k] - _oon.get(_k, np.nan)          # total central excess (dusty view)
        _di = _cof[_k] - _oof.get(_k, np.nan)          # intrinsic (age/Z) part
        _rows.append((_k[0], _dt, _dt - _di,
                      (_dt - _di) / _dt if (np.isfinite(_dt) and _dt > RED_CORE_MIN) else np.nan))
    _gk = np.array([r[0] for r in _rows])
    _T1[_core] = dict(gkey=_gk,
                      D_tot=np.array([r[1] for r in _rows]),
                      D_dust=np.array([r[2] for r in _rows]),
                      f_dust=np.array([r[3] for r in _rows]),
                      cls=_cls_of_keys(_gk))

_t1 = _T1[CORE_LABEL]
_nred = int(np.isfinite(_t1["f_dust"]).sum())
print(f"\nT1 [{CORE_LABEL} - {OUT_REF_LABEL}]: {_nred}/{len(_t1['f_dust'])} "
      f"(gal x sightline) rows have a red core (D_tot > {RED_CORE_MIN:g} mag)")
print(f"{'class':>14s} {'n_gal':>5s} {'f_dust_color [16-84]':>24s} {'D_tot med':>10s}")
_fd_gal, _fd_keys = _gal_median(_t1["f_dust"], _t1["gkey"])
_fd_cls = _cls_of_keys(_fd_keys)
for _cl in CLS4 + [c for c in set(_t1["cls"]) if c not in CLS4]:
    _m = _t1["cls"] == _cl
    if not _m.any():
        continue
    _v, _lo, _hi = _gboot_med(_t1["f_dust"][_m], _t1["gkey"][_m])
    print(f"{_cl:>14s} {len(set(_t1['gkey'][_m])):5d} {_v:12.2f} [{_lo:+.2f},{_hi:+.2f}] "
          f"{np.nanmedian(_t1['D_tot'][_m]):10.2f}")
_a, _b = (_fd_gal[(_fd_cls == c) & np.isfinite(_fd_gal)] for c in ("strong", "no_AGN"))
if len(_a) >= 3 and len(_b) >= 3:
    _u, _p = mannwhitneyu(_a, _b, alternative="two-sided")
    print(f"Mann-Whitney f_dust_color strong vs no_AGN (per-galaxy medians): p = {_p:.3f}")

# ── figure: colour profiles, dust contribution, decomposition by class ──
_PROF_LABS = ["ap1kpc"] + ANNULUS_LABELS[1:]          # radial sequence of annuli
_R_MID = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)
_cls_present = [c for c in CLS4 + ["star_forming", "no_event", "unclassified"]
                if c in set(P8["agn_class"])]

def _prof_matrix(col):
    """(n_gal*n_incl, n_annuli) matrix of `col` along the radial annulus sequence."""
    _maps = [_lab_map(col, _l) for _l in _PROF_LABS]
    _rk = sorted(set().union(*[set(m) for m in _maps]))
    return (np.array([[m.get(k, np.nan) for m in _maps] for k in _rk]),
            np.array([k[0] for k in _rk]))

_UVon_M, _pk_on = _prof_matrix("UV_on")
_UVoff_M, _ = _prof_matrix("UV_off")
_pcls = _cls_of_keys(_pk_on)
fig, axs = plt.subplots(1, 3, figsize=(21, 6.5))
for _row in _UVon_M:                                   # context spaghetti
    axs[0].plot(_R_MID, _row, color="0.85", lw=0.5, alpha=0.5, zorder=1)
for _cl in _cls_present:
    _m = _pcls == _cl
    if _m.sum() < 4:
        continue
    axs[0].plot(_R_MID, np.nanmedian(_UVon_M[_m], axis=0), "o-",
                color=AGN_COLORS[_cl], lw=2, zorder=3, label=_cl)
    axs[0].plot(_R_MID, np.nanmedian(_UVoff_M[_m], axis=0), "--",
                color=AGN_COLORS[_cl], lw=1.4, zorder=2)
axs[0].set_xscale("log")
axs[0].set_xlabel("radius [pkpc]")
axs[0].set_ylabel(r"$(U-V)$ [mag]  (solid: dust_on, dashed: dust_off)")
axs[0].legend(fontsize=10, frameon=False)
_DUV = _UVon_M - _UVoff_M                              # dust contribution profile
for _cl in _cls_present:
    _m = _pcls == _cl
    if _m.sum() < 4:
        continue
    _med = np.nanmedian(_DUV[_m], axis=0)
    _ci = np.array([_gboot_med(_DUV[_m][:, _j], _pk_on[_m], n=400)[1:]
                    for _j in range(len(_PROF_LABS))])
    axs[1].plot(_R_MID, _med, "o-", color=AGN_COLORS[_cl], lw=2, label=_cl)
    axs[1].fill_between(_R_MID, _ci[:, 0], _ci[:, 1], color=AGN_COLORS[_cl], alpha=0.15)
axs[1].axhline(0, color="0.5", lw=0.8, ls=":")
axs[1].set_xscale("log")
axs[1].set_xlabel("radius [pkpc]")
axs[1].set_ylabel(r"$\Delta(U-V)_{\rm dust}$ [mag]")
axs[1].legend(fontsize=10, frameon=False)
for _i, _cl in enumerate(_cls_present):                # decomposition strip
    _m = (_fd_cls == _cl) & np.isfinite(_fd_gal)
    if not _m.any():
        continue
    _xj = _i + np.random.default_rng(_i).uniform(-0.16, 0.16, int(_m.sum()))
    axs[2].scatter(_xj, _fd_gal[_m], c=AGN_COLORS[_cl], edgecolor="k",
                   linewidth=0.3, s=34)
    axs[2].hlines(np.nanmedian(_fd_gal[_m]), _i - 0.3, _i + 0.3, color="k", lw=2)
for _y, _lab in ((0, "no dust"), (1, "all dust")):
    axs[2].axhline(_y, color="0.6", lw=0.8, ls="--")
    axs[2].text(len(_cls_present) - 0.4, _y + 0.03, _lab, fontsize=8, color="0.4")
axs[2].set_xticks(range(len(_cls_present)))
axs[2].set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=10)
axs[2].set_ylabel(r"$f_{\rm dust}$ of the central $(U-V)$ excess")
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t1_color_decomposition.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8d — T2: does the surviving dust column carry the attenuation? ──
# Self-contained after Part 8c (P8 in memory). The foreground screen
#     A_V,screen = 1.086 * kappa_V * Sigma_dust
# is the CEILING: stars mixed with (or in front of) the dust attenuate less per
# unit column, so 'surviving ISM dust suffices' predicts a tight A_V-Sigma_dust
# correlation with measured/screen in ~[0.1, 1]. Ratios > 1 falsify it locally.
from scipy.stats import spearmanr

MSUN_KPC2_TO_G_CM2 = 2.089e-10        # 1 Msun/kpc^2 = 2.089e-10 g/cm^2
_ism_meta = Table.read(os.path.join(TABLEDIR, "annulus_ism_truth.fits")).meta
KAPPA = {b: float(_ism_meta[f"KAPPA_{b}"]) for b in ("U", "V", "J")}
# hyperion's kmh94 dust files tabulate chi per gram of GAS (the KMH94 dust-to-gas
# 0.00734 is baked in) — a V-band value of ~200 cm2/g betrays that convention;
# per-dust-mass MW opacity is ~3e4. Convert, or the screen ceiling is x136 low
# (2026-08-12 run: median measured/screen came out 144 for exactly this reason).
KMH94_GDR = 0.00734
if KAPPA["V"] < 1e3:
    KAPPA = {b: v / KMH94_GDR for b, v in KAPPA.items()}
    print(f"[kappa] file chi is per gram of GAS -> /{KMH94_GDR} -> per-dust-mass screen")
print(f"screen opacities [{_ism_meta.get('KAP_SRC', '?')}]: "
      + "  ".join(f"kappa_{b}={v:.4g} cm2/g" for b, v in KAPPA.items()))

_sig = np.asarray(P8["Sigma_dust"], float)
_avv = np.asarray(P8["A_V"], float)
P8["A_V_screen"] = 1.086 * KAPPA["V"] * _sig * MSUN_KPC2_TO_G_CM2
with np.errstate(all="ignore"):
    P8["screen_ratio"] = _avv / np.asarray(P8["A_V_screen"], float)

# convention guard: a per-GAS-mass chi would shift the ratio by ~x150
_ok0 = np.isfinite(np.asarray(P8["screen_ratio"], float)) & (_sig > 0) & (_avv > 0.02)
_rmed = float(np.nanmedian(np.asarray(P8["screen_ratio"], float)[_ok0]))
print(f"median measured/screen (A_V > 0.02 rows): {_rmed:.3g}")
assert 1e-3 < _rmed < 1e3, \
    f"screen ratio {_rmed:.2g} out of range — kappa convention broken (per gas vs per dust?)"

# ── per-annulus correlation + ratio (galaxy-bootstrap CIs) ──
_T2_LABS = ["ap1kpc"] + ANNULUS_LABELS[1:]

def _rho_stat(idx, x, y):
    _f = np.isfinite(x[idx]) & np.isfinite(y[idx])
    return spearmanr(x[idx][_f], y[idx][_f])[0] if _f.sum() >= 5 else np.nan

print(f"\n{'annulus':>10s} {'n':>4s} {'rho(A_V, Sig_dust) [16-84]':>28s} "
      f"{'med ratio [16-84]':>22s} {'>1.2':>5s}")
_t2_sum = {}
for _lab in _T2_LABS:
    _m = (_ap8 == _lab) & np.isfinite(_sig) & np.isfinite(_avv)
    if _m.sum() < 8:
        print(f"{_lab:>10s} {int(_m.sum()):4d}  — too few finite rows")
        continue
    _x, _y, _k = _sig[_m], _avv[_m], _GKEY[_m]
    _r, _rlo, _rhi = _gboot_idx(_k, lambda i: _rho_stat(i, _x, _y))
    _rat = np.asarray(P8["screen_ratio"], float)[_m]
    _q, _qlo, _qhi = _gboot_med(_rat, _k)
    _t2_sum[_lab] = (_r, _q)
    print(f"{_lab:>10s} {int(_m.sum()):4d} {_r:+12.2f} [{_rlo:+.2f},{_rhi:+.2f}] "
          f"{_q:10.3g} [{_qlo:.2g},{_qhi:.2g}] {int(np.nansum(_rat > 1.2)):5d}")

# screen-slope consistency: the SAME dust column must predict A_U/A_V and A_J/A_V
print("\nscreen-slope check (median measured vs KMH94 screen; agreement = "
      "geometry, not opacity, sets the ratio):")
for _b in ("U", "J"):
    _ab = np.asarray(P8[f"A_{_b}"], float)
    _f = np.isfinite(_ab) & np.isfinite(_avv) & (_avv > 0.05)
    if _f.sum() >= 8:
        print(f"   A_{_b}/A_V: measured {np.nanmedian(_ab[_f] / _avv[_f]):.2f}   "
              f"screen {KAPPA[_b] / KAPPA['V']:.2f}")

# falsifiers: attenuation the local dust column cannot supply even as a screen
_bad = _ok0 & (np.asarray(P8["screen_ratio"], float) > 1.2)
if _bad.any():
    print(f"\n[falsifiers] {int(_bad.sum())} rows with A_V > 1.2x the screen ceiling — "
          "per class: " + "  ".join(f"{c}:{int((_bad & (P8['agn_class'] == c)).sum())}"
                                    for c in CLS4 if (_bad & (P8["agn_class"] == c)).any()))
else:
    print("\n[falsifiers] no rows exceed 1.2x the screen ceiling — the surviving "
          "column is always sufficient")

# ── figure: A_V vs Sigma_dust per annulus + the ratio profile ──
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey=False)
_sgrid = np.logspace(2.5, 8, 60)
for _axi, _lab in zip(axs.flat[:len(_T2_LABS)], _T2_LABS):
    _m = (_ap8 == _lab) & np.isfinite(_sig) & np.isfinite(_avv) & (_sig > 0)
    for _cl in _cls_present:
        _s = _m & (P8["agn_class"] == _cl)
        _axi.scatter(_sig[_s], _avv[_s], s=22, c=AGN_COLORS[_cl], alpha=0.55,
                     edgecolor="k", linewidth=0.25, label=_cl)
    _scr = 1.086 * KAPPA["V"] * _sgrid * MSUN_KPC2_TO_G_CM2
    _axi.plot(_sgrid, _scr, "k-", lw=1.4)
    _axi.fill_between(_sgrid, 0.1 * _scr, _scr, color="0.5", alpha=0.15)
    if _lab in _t2_sum:
        _axi.text(0.04, 0.95, f"$\\rho$={_t2_sum[_lab][0]:+.2f}\n"
                  f"ratio={_t2_sum[_lab][1]:.2g}", transform=_axi.transAxes,
                  va="top", fontsize=9,
                  bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    _axi.set_xscale("log")
    _axi.set_yscale("log")
    _axi.set_xlim(3e2, 1e8)
    _axi.set_ylim(1e-3, 5)
    _axi.set_title(_lab, fontsize=11)
    _axi.set_xlabel(r"$\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]")
    _axi.set_ylabel(r"$A_V^{\rm ann}$ [mag]")
axs.flat[0].legend(fontsize=9, loc="lower right", framealpha=0.9)
_axr = axs.flat[-1]                                    # summary: ratio vs radius
_R_MID8 = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)
for _cl in _cls_present:
    _med = []
    for _lab in _T2_LABS:
        _s = (_ap8 == _lab) & (P8["agn_class"] == _cl) & _ok0
        _med.append(np.nanmedian(np.asarray(P8["screen_ratio"], float)[_s])
                    if _s.sum() >= 4 else np.nan)
    _axr.plot(_R_MID8, _med, "o-", color=AGN_COLORS[_cl], lw=2, label=_cl)
_axr.axhline(1.0, color="k", lw=1.2)
_axr.axhspan(0.1, 1.0, color="0.5", alpha=0.15)
_axr.set_xscale("log")
_axr.set_yscale("log")
_axr.set_xlabel("radius [pkpc]")
_axr.set_ylabel("measured / screen $A_V$")
_axr.legend(fontsize=9, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t2_av_vs_sigma_dust.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8e — T3: the observable clock — binned CIGALE A_V vs stellar age ──
# Pure read of the Part 7e/7f caches on top of P8 (8c in memory). Builds G8 —
# ONE row per GALAXY (every quantity is the nanmedian over the 4 sightlines) —
# which 8f/8g reuse. No rank statistics anywhere in 8e–8g: binned medians with
# galaxy-bootstrap bands, confounders split one per panel (mass | z | class).
# The CIGALE A_V comes from the two REGION fits that carry the science: the
# core (0–3.2 kpc disc) and the outskirt (3.2–31.6 kpc), both actually fitted
# on their own photometry + SFH (Parts 7b2/7c/7d). Positive-definite
# quantities (Sigma, mass fractions) are log10 with zeros -> NaN; all axes use
# robust percentile limits so outliers cannot stretch the view.
from scipy.stats import spearmanr, theilslopes

CLS3          = ["strong", "intermediate", "weak"]  # no_AGN/no_event: too few for tracks
N_AGE_BINS    = 4     # equal-count bins per subset (auto-reduced when thin)
MIN_PER_BIN   = 5
SLOPE_MIN_PTS = 3     # finite annuli (of 5) needed for a radial slope

_resf = os.path.join(TABLEDIR, "cigale_region_results.fits")
_truf = os.path.join(TABLEDIR, "aperture_truth.fits")
for _f8 in (_resf, _truf):
    if not os.path.exists(_f8):
        raise FileNotFoundError(f"{_f8} missing — run Parts 7d–7f first")
_TC = Table.read(_resf)
_TC = _TC[np.char.strip(np.asarray(_TC["arm"], str)) == "dust_on"]
_TT = Table.read(_truf)
for _t8 in (_TC, _TT):
    for _c in ("aperture", "incl"):
        _t8[_c] = np.char.strip(np.asarray(_t8[_c], str))
_TC["Av_corr"] = (np.asarray(_TC["bayes.attenuation.Av_ISM"], float)
                  - np.asarray(_TC["Av_zp"], float))

def _map3(_t, _col, _ap):
    _m = np.asarray(_t["aperture"], str) == _ap
    return {(int(s), int(g), str(i)): float(v) for s, g, i, v in
            zip(_t["snap"][_m], _t["gal_id"][_m], _t["incl"][_m],
                np.asarray(_t[_col], float)[_m])}

def _l10m(_m3):
    """log10 map; zero/negative/non-finite -> NaN (log axes must not fake a floor)"""
    return {_k: (np.log10(_v) if np.isfinite(_v) and _v > 0 else np.nan)
            for _k, _v in _m3.items()}

def _fmap(_num, _den):
    """element-wise ratio of two (snap, gal, incl)->value maps"""
    _out = {}
    for _k, _v in _num.items():
        _d = _den.get(_k, np.nan)
        _out[_k] = _v / _d if (np.isfinite(_v) and np.isfinite(_d) and _d > 0) else np.nan
    return _out

_gals8 = sorted(set(zip(np.asarray(P8["snap"], int), np.asarray(P8["gal_id"], int))))

def _gmed(_m3):
    """per-galaxy nanmedian over the 4 sightlines of a (snap, gal, incl)->value map"""
    _out = np.full(len(_gals8), np.nan)
    for _j, (_s, _g) in enumerate(_gals8):
        _v = np.array([_m3.get((_s, _g, _i), np.nan) for _i in INCL_LABELS])
        if np.isfinite(_v).any():
            _out[_j] = np.nanmedian(_v)
    return _out

def _slope_map(_col, log10=False):
    """per (galaxy, sightline) Theil-Sen slope of `col` along the 5-annulus profile"""
    _maps = [_map3(P8, _col, _l) for _l in _PROF_LABS]
    if log10:
        _maps = [_l10m(_m) for _m in _maps]
    _lr, _out = np.log10(_R_MID), {}
    for _s, _g in _gals8:
        for _i in INCL_LABELS:
            _p = np.array([_m.get((_s, _g, _i), np.nan) for _m in _maps])
            _ok = np.isfinite(_p)
            if _ok.sum() >= SLOPE_MIN_PTS:
                _out[(_s, _g, _i)] = float(theilslopes(_p[_ok], _lr[_ok])[0])
    return _out

# ── G8: one row per galaxy — the single table 8e–8g plot from ──
G8 = Table()
G8["snap"]    = np.array([s for s, _ in _gals8], int)
G8["gal_id"]  = np.array([g for _, g in _gals8], int)
G8["gkey"]    = np.array([f"{s}_{g}" for s, g in _gals8])
G8["z"]       = np.array([_z_of_snap[s] for s, _ in _gals8])
G8["cls"]     = _cls_of_keys(np.asarray(G8["gkey"], str))
G8["lms"]     = np.array([_selnum(_md[(s, g)], "log_mstar") if (s, g) in _md
                          else np.nan for s, g in _gals8])
# CIGALE A_V (zp-corrected) in the two science regions
G8["Av_obs"]     = _gmed(_map3(_TC, "Av_corr", "core"))
G8["Av_obs_out"] = _gmed(_map3(_TC, "Av_corr", "outskirt"))
# mass-weighted ages in the matching geometries [Gyr]
G8["age_core"]    = _gmed(_map3(_TT, "age_m_star_myr", CORE_LABEL)) / 1e3
G8["age_out_reg"] = _gmed(_map3(_TT, "age_m_star_myr", "outskirt")) / 1e3  # region
G8["age_out"]     = _gmed(_map3(_TT, "age_m_star_myr", OUT_REF_LABEL)) / 1e3  # annulus
G8["age_tot"]     = _gmed(_map3(_TT, "age_m_star_myr", "ap100kpc")) / 1e3
# RT-truth attenuation + ISM columns (annuli: core disc vs 10–32 kpc annulus)
G8["AvT_core"]  = _gmed(_map3(P8, "A_V", CORE_LABEL))
G8["AvT_out"]   = _gmed(_map3(P8, "A_V", OUT_REF_LABEL))
G8["lSd_core"]  = _gmed(_l10m(_map3(P8, "Sigma_dust", CORE_LABEL)))
G8["lSd_out"]   = _gmed(_l10m(_map3(P8, "Sigma_dust", OUT_REF_LABEL)))
G8["lSh2_core"] = _gmed(_l10m(_map3(P8, "Sigma_H2", CORE_LABEL)))
G8["lSh2_out"]  = _gmed(_l10m(_map3(P8, "Sigma_H2", OUT_REF_LABEL)))
G8["slope_Av"]  = _gmed(_slope_map("A_V"))                          # mag/dex
G8["slope_lSd"] = _gmed(_slope_map("Sigma_dust", log10=True))       # dex/dex
G8["slope_lSh2"] = _gmed(_slope_map("Sigma_H2", log10=True))        # dex/dex
# dust / H2 mass fractions in the core aperture (M from 8a, M* truth from 7e)
_msC = _map3(_TT, "mstar", CORE_LABEL)
G8["lfd_core"]  = _gmed(_l10m(_fmap(_map3(P8, "M_dust", CORE_LABEL), _msC)))
G8["lfh2_core"] = _gmed(_l10m(_fmap(_map3(P8, "M_H2", CORE_LABEL), _msC)))

# time since the quench end: anchor-epoch cosmic age − t_QT (selection FITS, yr)
_tqt = np.array([_selnum(_md[(s, g)], "t_qt") if (s, g) in _md else np.nan
                 for s, g in _gals8])
_dtq = np.array([COSMO.age(_z_of_snap[s]).value for s, _ in _gals8]) - _tqt / 1e9
_bad = np.isfinite(_dtq) & (_dtq < -0.05)
if _bad.any():
    print(f"[warn] {int(_bad.sum())} galaxies have t_QT AFTER the anchor epoch "
          "(Δt < -0.05 Gyr) -> dt_q set NaN; check t_qt units/selection")
    _dtq[_bad] = np.nan
G8["dt_q"] = np.clip(_dtq, 0.0, None)                               # Gyr

_c8, _sn8 = np.asarray(G8["cls"], str), np.asarray(G8["snap"], int)
_lms8, _snaps8 = np.asarray(G8["lms"], float), sorted(set(_sn8))
print(f"G8: {len(G8)} galaxies — classes " +
      "  ".join(f"{c}:{int((_c8 == c).sum())}" for c in CLS3) +
      f"  (dropped from class panels: {int((~np.isin(_c8, CLS3)).sum())})")
print("finite per column: " + "  ".join(
    f"{_c}:{int(np.isfinite(np.asarray(G8[_c], float)).sum())}"
    for _c in ("Av_obs", "Av_obs_out", "age_core", "age_out_reg", "age_out",
               "age_tot", "AvT_core", "AvT_out", "lSd_core", "lSd_out",
               "lSh2_core", "lSh2_out", "slope_Av", "slope_lSd", "slope_lSh2",
               "lfd_core", "lfh2_core", "dt_q")))

# ── shared track/stat/axis helpers (8e–8g) ──
def _binned_track(x, y, nbins=N_AGE_BINS, min_per_bin=MIN_PER_BIN, n=1000, seed=0):
    """Equal-count bins in x -> per-bin (x_med, y_med, lo16, hi84); CI = bootstrap
    over the galaxies in the bin (G8 is one row per galaxy already)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    _ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[_ok], y[_ok]
    _nb = min(nbins, len(x) // min_per_bin)
    if _nb < 2:
        return (np.array([]),) * 4
    _e = np.quantile(x, np.linspace(0, 1, _nb + 1))
    _ib = np.clip(np.searchsorted(_e, x, side="right") - 1, 0, _nb - 1)
    _rng, _res = np.random.default_rng(seed), []
    for _b in range(_nb):
        _yb = y[_ib == _b]
        _bm = np.median(_rng.choice(_yb, (n, len(_yb))), axis=1)
        _res.append((np.median(x[_ib == _b]), np.median(_yb),
                     np.percentile(_bm, 16), np.percentile(_bm, 84)))
    return tuple(np.array(_v) for _v in zip(*_res))

def _track(ax, x, y, color, label=None, **kw):
    _xm, _ym, _lo, _hi = _binned_track(x, y, **kw)
    if len(_xm):
        ax.plot(_xm, _ym, "o-", color=color, lw=2, ms=6, zorder=4, label=label)
        ax.fill_between(_xm, _lo, _hi, color=color, alpha=0.18, zorder=3)
    elif label is not None:
        ax.plot([], [], "o-", color=color, lw=2, label=f"{label} (too few)")

def _rho(x, y):
    _m = np.isfinite(np.asarray(x, float)) & np.isfinite(np.asarray(y, float))
    if _m.sum() < 8:
        return f"n={int(_m.sum())}"
    _r, _p = spearmanr(np.asarray(x, float)[_m], np.asarray(y, float)[_m])
    return f"n={int(_m.sum())} rho={_r:+.2f} p={_p:.3f}"

def _rlim(*arrs, pad=0.06, k=1.7):
    """robust axis range over pooled finite values: Tukey fences (quartiles
    ± k*IQR, clipped to the data) — outliers cannot stretch the view"""
    _v = np.concatenate([np.asarray(a, float).ravel() for a in arrs])
    _v = _v[np.isfinite(_v)]
    if _v.size < 4:
        return None
    _q1, _q3 = np.percentile(_v, [25, 75])
    if _q3 > _q1:
        _lo = max(float(_v.min()), _q1 - k * (_q3 - _q1))
        _hi = min(float(_v.max()), _q3 + k * (_q3 - _q1))
    else:
        _lo, _hi = np.percentile(_v, [1, 99])
    if not _hi > _lo:
        return None
    return _lo - pad * (_hi - _lo), _hi + pad * (_hi - _lo)

# ── figure: <A_V^CIGALE> vs age in the core (top) and the outskirt (bottom),
#            split by mass | redshift | class ──
_mcut = np.nanmedian(_lms8)
_MBINS = [(_lms8 < _mcut, "#1b9e77", f"log M* < {_mcut:.2f}"),
          (_lms8 >= _mcut, "#7570b3", f"log M* ≥ {_mcut:.2f}")]
_zcol = dict(zip(_snaps8, plt.cm.viridis(np.linspace(0.1, 0.9, len(_snaps8)))))

fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey="row", sharex=True)
for _row, (_yn, _xn, _yl) in enumerate(
        (("Av_obs", "age_core", r"$A_V^{\rm CIGALE}$ (core, <3.2 kpc) [mag]"),
         ("Av_obs_out", "age_out_reg",
          r"$A_V^{\rm CIGALE}$ (outskirt, 3.2–32 kpc) [mag]"))):
    _x, _y = np.asarray(G8[_xn], float), np.asarray(G8[_yn], float)
    print(f"\nSpearman rho({_yn}, {_xn}) — all: {_rho(_x, _y)}")
    for _ax in axs[_row]:
        _ax.scatter(_x, _y, s=22, c="0.82", edgecolor="none", zorder=1)
    for _m, _co, _lab in _MBINS:                        # (a) mass halves
        _track(axs[_row][0], _x[_m], _y[_m], _co, _lab if _row == 0 else None)
        print(f"   {_lab:>16s}: {_rho(_x[_m], _y[_m])}")
    for _sn in _snaps8:                                 # (b) redshift anchors
        _m = _sn8 == _sn
        _track(axs[_row][1], _x[_m], _y[_m], _zcol[_sn],
               f"z ≈ {_z_of_snap[_sn]:.1f}" if _row == 0 else None)
        print(f"   {'z~%.1f' % _z_of_snap[_sn]:>16s}: {_rho(_x[_m], _y[_m])}")
    for _cl in CLS3:                                    # (c) coupling class
        _m = _c8 == _cl
        _track(axs[_row][2], _x[_m], _y[_m], AGN_COLORS[_cl],
               _cl if _row == 0 else None)
        print(f"   {_cl:>16s}: {_rho(_x[_m], _y[_m])}")
    _yr = _rlim(_y)
    if _yr:
        axs[_row][0].set_ylim(*_yr)
    axs[_row][0].set_ylabel(_yl)
_xr = _rlim(G8["age_core"], G8["age_out_reg"])
if _xr:
    axs[0][0].set_xlim(*_xr)
for _ax in axs[1]:
    _ax.set_xlabel("mass-weighted stellar age of the same region [Gyr]")
for _ax in axs[0]:
    _ax.legend(fontsize=10, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t3_av_vs_age_binned.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8f — T4: where the dust (and H2) sits — core vs outskirt vs slope, by class ──
# Truth-side structure at matched LOCAL ages (annuli were never CIGALE-fitted, so
# everything here is the RT / 8a truth). Rows: A_V, log Sigma_dust, log Sigma_H2;
# columns: core (<3.2 kpc) | outskirt annulus (10–32 kpc) | radial Theil-Sen slope
# vs TOTAL age. Core and outskirt panels of a row share the y-range, so the
# core-vs-outskirt offset is read directly; all ranges are robust percentiles.
# Needs 8e in memory (G8, _track, _rho, _rlim).
_ROWS8F = [
    ("AvT_core", "AvT_out", "slope_Av",
     r"$A_V^{\rm RT}$ [mag]", r"d$A_V$/d$\log r$ [mag/dex]"),
    ("lSd_core", "lSd_out", "slope_lSd",
     r"$\log\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]", r"d$\log\Sigma_{\rm d}$/d$\log r$"),
    ("lSh2_core", "lSh2_out", "slope_lSh2",
     r"$\log\Sigma_{\rm H_2}$ [M$_\odot$ kpc$^{-2}$]", r"d$\log\Sigma_{\rm H_2}$/d$\log r$"),
]
_XCOLS8F = [("age_core", "core age [Gyr]"), ("age_out", "outskirt age [Gyr]"),
            ("age_tot", "total age [Gyr]")]
fig, axs = plt.subplots(3, 3, figsize=(21, 16.5), sharex="col")
print(f"{'panel':>10s}  per-class Spearman (per-galaxy points)")
for _r, (_ync, _yno, _yns, _yl, _yls) in enumerate(_ROWS8F):
    for _cc, _yn in enumerate((_ync, _yno, _yns)):
        _ax = axs[_r][_cc]
        _x = np.asarray(G8[_XCOLS8F[_cc][0]], float)
        _y = np.asarray(G8[_yn], float)
        _stats = []
        for _cl in CLS3:
            _m = _c8 == _cl
            _ax.scatter(_x[_m], _y[_m], s=26, c=AGN_COLORS[_cl], alpha=0.6,
                        edgecolor="k", linewidth=0.3, zorder=2)
            _track(_ax, _x[_m], _y[_m], AGN_COLORS[_cl],
                   _cl if (_r, _cc) == (0, 0) else None, nbins=3)
            _stats.append(f"{_cl[:5]}: {_rho(_x[_m], _y[_m])}")
        print(f"{_yn:>10s}  " + " | ".join(_stats))
    # core+outskirt share the row's y-range; the slope panel gets its own + a zero guide
    _yr = _rlim(G8[_ync], G8[_yno])
    if _yr:
        axs[_r][0].set_ylim(*_yr)
        axs[_r][1].set_ylim(*_yr)
    _yrs = _rlim(G8[_yns])
    if _yrs:
        axs[_r][2].set_ylim(*_yrs)
    axs[_r][2].axhline(0, color="0.5", lw=0.8, ls=":")
    axs[_r][0].set_ylabel(_yl)
    axs[_r][2].set_ylabel(_yls)
for _cc, (_xn, _xl) in enumerate(_XCOLS8F):
    _xr = _rlim(G8[_xn])
    if _xr:
        axs[0][_cc].set_xlim(*_xr)
    axs[2][_cc].set_xlabel(_xl)
for _ax, _ttl in zip(axs[0], ("core (<3.2 kpc)", "outskirt annulus (10–32 kpc)",
                              "radial slope (5-annulus Theil–Sen)")):
    _ax.set_title(_ttl, fontsize=11)
axs[0][0].legend(fontsize=10, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t4_structure_vs_age.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)


In [ ]:
# ── Part 8g — T5: the dust clock — everything vs time since quenching ──
# x = cosmic time elapsed between the quench end (t_QT, selection FITS) and the
# anchor epoch. If AGN coupling clears dust FASTER, the strong track must fall
# more steeply than the weak one — one readable Theil-Sen slope per class with
# a galaxy-bootstrap 16–84 CI. Top row: the attenuation clock (CIGALE in the
# core and outskirt regions + RT truth). Bottom row: the ISM clock — dust
# column, dust mass fraction and H2 mass fraction in the core. Needs 8e in
# memory (G8 etc.).
from scipy.stats import theilslopes

_xq = np.asarray(G8["dt_q"], float)
print(f"time since quench: finite on {int(np.isfinite(_xq).sum())}/{len(_xq)} galaxies; "
      f"range {np.nanmin(_xq):.2f}–{np.nanmax(_xq):.2f} Gyr")

def _ts_ci(x, y, n=1000, seed=0):
    """pooled Theil-Sen slope + bootstrap-over-galaxies 16–84 CI -> (s, lo, hi, n)"""
    x, y = np.asarray(x, float), np.asarray(y, float)
    _m = np.isfinite(x) & np.isfinite(y)
    x, y = x[_m], y[_m]
    if len(x) < 8:
        return None
    _rng, _b = np.random.default_rng(seed), []
    for _ in range(n):
        _i = _rng.integers(0, len(x), len(x))
        if np.unique(x[_i]).size > 2:
            _b.append(theilslopes(y[_i], x[_i])[0])
    return (float(theilslopes(y, x)[0]),
            float(np.percentile(_b, 16)), float(np.percentile(_b, 84)), len(x))

_PANELS8G = [
    ("Av_obs",     r"$A_V^{\rm CIGALE}$ (core, <3.2 kpc) [mag]", "mag/Gyr"),
    ("Av_obs_out", r"$A_V^{\rm CIGALE}$ (outskirt, 3.2–32 kpc) [mag]", "mag/Gyr"),
    ("AvT_core",   r"core $A_V^{\rm RT}$ [mag]", "mag/Gyr"),
    ("lSd_core",   r"core $\log\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]", "dex/Gyr"),
    ("lfd_core",   r"core $\log\,M_{\rm dust}/M_\star$", "dex/Gyr"),
    ("lfh2_core",  r"core $\log\,M_{\rm H_2}/M_\star$", "dex/Gyr"),
]
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharex=True)
for _ax, (_yn, _yl, _un) in zip(axs.ravel(), _PANELS8G):
    _y = np.asarray(G8[_yn], float)
    print(f"\n{_yn} vs time since quench — Theil-Sen per class:")
    for _cl in CLS3:
        _m = _c8 == _cl
        _ax.scatter(_xq[_m], _y[_m], s=26, c=AGN_COLORS[_cl], alpha=0.6,
                    edgecolor="k", linewidth=0.3, zorder=2)
        _track(_ax, _xq[_m], _y[_m], AGN_COLORS[_cl],
               _cl if _ax is axs[0][0] else None, nbins=3)
        _r = _ts_ci(_xq[_m], _y[_m])
        print(f"   {_cl:>14s}: " + (f"{_r[0]:+.3f} [{_r[1]:+.3f}, {_r[2]:+.3f}] {_un} "
                                    f"(n = {_r[3]})" if _r else "too few finite rows"))
    _yr = _rlim(_y)
    if _yr:
        _ax.set_ylim(*_yr)
    _ax.set_ylabel(_yl)
_xr = _rlim(_xq, pad=0.04)
if _xr:
    axs[0][0].set_xlim(max(0.0, _xr[0]), _xr[1])
for _ax in axs[1]:
    _ax.set_xlabel(r"time since quenching, $t(z_{\rm snap}) - t_{\rm QT}$ [Gyr]")
axs[0][0].legend(fontsize=10, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t5_dust_clock.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

# Part 8h — T6: is longer-living dust what keeps cores red, and how fast does the peak erode?

Three pure-read figures on top of the 8c/8e caches (P8, G8), each split **low-z vs high-z** at
the median anchor redshift:

- **8h1 (the shape)** — stacked annular profiles of the RT $A_V$ and of the dust-only colour
  gradient $\Delta(U-V)_{\rm dust}(R)$, in terciles of time since quenching $\Delta t_{\rm q}$.
  Answers: does attenuation *peak* in the centre, and does the peak erode outside-in,
  inside-out, or self-similarly?
- **8h2 (the speed)** — the concentration $C_{A_V} = A_V(<3.2\,{\rm kpc}) -
  A_V(10\text{–}32\,{\rm kpc})$, the 5-annulus radial slope, and the CIGALE-observable analogue,
  each vs $\Delta t_{\rm q}$ with one Theil–Sen slope per z half: the **erosion rate of central
  attenuation in mag/Gyr**, and whether it is faster at high z.
- **8h3 (the link)** — central dust-attributed reddening (T1) and $C_{A_V}$ vs the surviving
  core dust fraction $M_{\rm dust}/M_\star$, coloured by $\Delta t_{\rm q}$: if long-lived dust
  is what keeps cores red, the trend runs along the dust axis, not along the clock.

**Caveats.** The outer annulus inherits the outskirts' shot noise (these galaxies hold ~30–90
gas particles there), so $C_{A_V}$ requires the 10–32 kpc $A_V$ finite on ≥2 sightlines.
$\Delta t_{\rm q}$ is the only clock used — mass-weighted age mixes in the pre-quench history
and compresses toward small values at high z, which would *fake* "faster evolution at high z";
galaxies without a quench event are drawn as open markers but never enter terciles or slopes.
The observable concentration uses the two fitted **cumulative** apertures ($<3.2$ and $<32$
kpc), so the core is inside both and the contrast is diluted — the sign and trend carry, the
amplitude does not.


In [ ]:
# ── Part 8h1 — T6: stacked attenuation profiles — the shape, by quench clock × z ──
# Needs 8c + 8e in memory (P8, _prof_matrix, _R_MID, G8, _gboot_med, _rlim).
# Row 1: the annular RT A_V profile; row 2: the dust-only colour gradient
# Delta(U-V)_dust(R) = (U-V)_on - (U-V)_off (T1's decomposition, all annuli).
# Lines: terciles of time since quenching dt_q (one sequential ramp, young ->
# old = light -> dark; galaxies without a quench event are excluded and
# counted); columns: low-z vs high-z anchors, split at the median galaxy z.
_AV_M8H, _pk8h = _prof_matrix("A_V")
_UVON8H, _ = _prof_matrix("UV_on")
_UVOFF8H, _ = _prof_matrix("UV_off")
_DUV8H = _UVON8H - _UVOFF8H

# per-galaxy clock and z -> per-profile-row via gkey
_g8map = {str(k): (float(d), float(z)) for k, d, z in
          zip(G8["gkey"], np.asarray(G8["dt_q"], float), np.asarray(G8["z"], float))}
_row_dtq = np.array([_g8map.get(str(k), (np.nan, np.nan))[0] for k in _pk8h])
_row_z   = np.array([_g8map.get(str(k), (np.nan, np.nan))[1] for k in _pk8h])

_dtq_g = np.asarray(G8["dt_q"], float)
if not np.isfinite(_dtq_g).any():
    raise RuntimeError("no finite dt_q in G8 — check t_qt in the selection FITS (8e prints why)")
Z_SPLIT8H = float(np.nanmedian(np.asarray(G8["z"], float)))
_z8h_g = np.asarray(G8["z"], float)
print("anchor membership: " + "  ".join(
    f"z~{_z_of_snap[_s]:.2f}({'lo' if _z_of_snap[_s] <= Z_SPLIT8H else 'hi'}):"
    f"{int((np.asarray(G8['snap'], int) == _s).sum())}" for _s in _snaps8)
    + f"   [split at median z = {Z_SPLIT8H:.2f}]")
print(f"no quench event (never in terciles/slopes): "
      f"{int((~np.isfinite(_dtq_g)).sum())}/{len(_dtq_g)} galaxies")

_te8h = np.nanpercentile(_dtq_g, [100 / 3, 200 / 3])
_TER_LABS8H = [rf"$\Delta t_q$ < {_te8h[0]:.2f} Gyr",
               rf"{_te8h[0]:.2f}–{_te8h[1]:.2f} Gyr",
               rf"$\Delta t_q$ > {_te8h[1]:.2f} Gyr"]
_TER_COLS8H = plt.cm.Purples([0.45, 0.68, 0.92])     # sequential: young -> old
_row_ter = np.full(len(_pk8h), -1)
_fin8h = np.isfinite(_row_dtq)
_row_ter[_fin8h] = np.searchsorted(_te8h, _row_dtq[_fin8h], side="right")

_ZB8H = [(f"z ≤ {Z_SPLIT8H:.2f}", _row_z <= Z_SPLIT8H),
         (f"z > {Z_SPLIT8H:.2f}", _row_z > Z_SPLIT8H)]
_ZB8H = [(_n, _m) for _n, _m in _ZB8H if len(set(_pk8h[_m])) >= 6]
if not _ZB8H:
    raise RuntimeError("no z bin holds >= 6 galaxies — lower the threshold or check G8['z']")

_i_out8h = _PROF_LABS.index(OUT_REF_LABEL)
fig, axs = plt.subplots(2, len(_ZB8H), figsize=(7.5 * len(_ZB8H), 11),
                        sharex=True, sharey="row", squeeze=False)
print(f"\n{'z bin':>10s} {'tercile':>10s} {'n_gal':>5s} {'A_V(1-3kpc)':>11s} "
      f"{'A_V(10-32)':>10s} {'core-out':>8s}")
for _cc, (_zlab, _zm) in enumerate(_ZB8H):
    for _rr, (_M8, _zero) in enumerate(((_AV_M8H, False), (_DUV8H, True))):
        _ax = axs[_rr][_cc]
        for _row in _M8[_zm]:                          # context spaghetti
            _ax.plot(_R_MID, _row, color="0.88", lw=0.5, alpha=0.5, zorder=1)
        for _t in range(3):
            _m = _zm & (_row_ter == _t)
            _ng = len(set(_pk8h[_m]))
            if _ng < 4:
                continue
            _med = np.nanmedian(_M8[_m], axis=0)
            _ci = np.array([_gboot_med(_M8[_m][:, _j], _pk8h[_m], n=400)[1:]
                            for _j in range(len(_PROF_LABS))])
            _ax.plot(_R_MID, _med, "o-", color=_TER_COLS8H[_t], lw=2, zorder=3,
                     label=f"{_TER_LABS8H[_t]} (n={_ng})" if (_rr, _cc) == (0, 0) else None)
            _ax.fill_between(_R_MID, _ci[:, 0], _ci[:, 1],
                             color=_TER_COLS8H[_t], alpha=0.2, zorder=2)
            if _rr == 0:
                print(f"{_zlab:>10s} {'T%d' % (_t + 1):>10s} {_ng:5d} {_med[1]:11.3f} "
                      f"{_med[_i_out8h]:10.3f} {_med[1] - _med[_i_out8h]:+8.3f}")
        if _zero:
            _ax.axhline(0, color="0.5", lw=0.8, ls=":")
        _ax.set_xscale("log")
    axs[0][_cc].set_title(f"{_zlab}  ({len(set(_pk8h[_zm]))} galaxies)", fontsize=11)
    axs[1][_cc].set_xlabel("radius [pkpc]")
for _rr, (_M8, _yl) in enumerate(((_AV_M8H, r"annular $A_V^{\rm RT}$ [mag]"),
                                  (_DUV8H, r"$\Delta(U-V)_{\rm dust}$ [mag]"))):
    _yr = _rlim(_M8)
    if _yr:
        axs[_rr][0].set_ylim(*_yr)
    axs[_rr][0].set_ylabel(_yl)
axs[0][0].legend(fontsize=10, frameon=False, title="time since quenching")
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t6_profile_stacks.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)


In [ ]:
# ── Part 8h2 — T6: attenuation concentration vs the quench clock, per z half ──
# Needs 8c + 8e + 8h1 in memory (P8, G8, _gals8, _map3, _track, _rlim, _rho,
# Z_SPLIT8H). C_Av = A_V(<3.2 kpc) - A_V(10-32 kpc annulus) per sightline, then
# the galaxy median — the outer annulus must be finite on >= 2 sightlines or
# the metric inherits the outskirts' shot noise. The Theil-Sen slope per z half
# IS the erosion rate of central attenuation [mag/Gyr]. Third panel: the
# OBSERVABLE concentration Av_obs(core) - Av_obs(outskirt) — the two fitted
# REGIONS, geometrically parallel to C_Av (core minus outer zone) rather than
# the old diluted cumulative contrast. Residual mismatch vs C_Av: the outskirt
# region additionally covers 3.2-10 kpc, which the 10-32 annulus reference does
# not. C_Av > 0 = centrally peaked.
_avc3_8h = _map3(P8, "A_V", CORE_LABEL)
_avo3_8h = _map3(P8, "A_V", OUT_REF_LABEL)
_cav8h = np.full(len(_gals8), np.nan)
for _j, (_s, _g) in enumerate(_gals8):
    _pr = np.array([[_avc3_8h.get((_s, _g, _i), np.nan),
                     _avo3_8h.get((_s, _g, _i), np.nan)] for _i in INCL_LABELS])
    if np.isfinite(_pr[:, 1]).sum() >= 2:
        _d = _pr[:, 0] - _pr[:, 1]
        if np.isfinite(_d).any():
            _cav8h[_j] = np.nanmedian(_d)
G8["C_Av"] = _cav8h
G8["C_obs"] = np.asarray(G8["Av_obs"], float) - np.asarray(G8["Av_obs_out"], float)
print(f"C_Av finite on {int(np.isfinite(_cav8h).sum())}/{len(G8)} galaxies "
      f"(needs the 10-32 kpc A_V on >= 2 sightlines); "
      f"C_obs on {int(np.isfinite(np.asarray(G8['C_obs'], float)).sum())}")

def _ts8h(x, y, n=1000, seed=0):
    """Theil-Sen slope + bootstrap 16-84 CI (G8 = one row per galaxy already)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    _m = np.isfinite(x) & np.isfinite(y)
    x, y = x[_m], y[_m]
    if len(x) < 8:
        return None
    _rng, _b = np.random.default_rng(seed), []
    for _ in range(n):
        _i = _rng.integers(0, len(x), len(x))
        if np.unique(x[_i]).size > 2:
            _b.append(theilslopes(y[_i], x[_i])[0])
    return (float(theilslopes(y, x)[0]),
            float(np.percentile(_b, 16)), float(np.percentile(_b, 84)), len(x))

_z8h2 = np.asarray(G8["z"], float)
_ZG8H = [(f"z ≤ {Z_SPLIT8H:.2f}", _z8h2 <= Z_SPLIT8H, plt.cm.viridis(0.25)),
         (f"z > {Z_SPLIT8H:.2f}", _z8h2 > Z_SPLIT8H, plt.cm.viridis(0.80))]
# definition in the panel TITLE, short y-labels: the long LaTeX y/x-labels
# collided across panels and clipped at the figure edge under tight_layout
_PANELS8H2 = [
    ("C_Av", r"$A_V^{\rm RT}(<3.2\,{\rm kpc}) - A_V^{\rm RT}(10$–$32\,{\rm kpc})$",
     r"$C_{A_V}$ [mag]", "mag/Gyr"),
    ("slope_Av", "5-annulus Theil–Sen radial slope",
     r"d$A_V^{\rm RT}$/d$\log r$ [mag/dex]", "mag dex$^{-1}$/Gyr"),
    ("C_obs", r"$A_V^{\rm CIGALE}({\rm core}) - A_V^{\rm CIGALE}(3.2$–$32\,{\rm kpc})$",
     r"$C_{A_V}^{\rm CIGALE}$ [mag]", "mag/Gyr"),
]
_xq8h = np.asarray(G8["dt_q"], float)
fig, axs = plt.subplots(1, 3, figsize=(22, 6.5), sharex=True,
                        constrained_layout=True)
for _ax, (_yn, _ttl, _yl, _un) in zip(axs, _PANELS8H2):
    _y = np.asarray(G8[_yn], float)
    print(f"\n{_yn} vs time since quench — Theil-Sen erosion rate [{_un}]:")
    _txt = []
    for _zlab, _zm, _co in _ZG8H:
        _ax.scatter(_xq8h[_zm], _y[_zm], s=26, color=_co, alpha=0.65,
                    edgecolor="k", linewidth=0.3, zorder=2)
        _track(_ax, _xq8h[_zm], _y[_zm], _co, _zlab if _ax is axs[0] else None, nbins=3)
        _r = _ts8h(_xq8h[_zm], _y[_zm])
        _s = (f"{_r[0]:+.3f} [{_r[1]:+.3f}, {_r[2]:+.3f}] (n = {_r[3]})"
              if _r else "too few finite rows")
        _txt.append(f"{_zlab}: " + (f"{_r[0]:+.3f} [{_r[1]:+.3f},{_r[2]:+.3f}]"
                                    if _r else "n < 8"))
        print(f"   {_zlab:>10s}: {_s}")
    _rp = _ts8h(_xq8h, _y)
    print("   {:>10s}: ".format("pooled")
          + (f"{_rp[0]:+.3f} [{_rp[1]:+.3f}, {_rp[2]:+.3f}] (n = {_rp[3]})"
             if _rp else "too few finite rows"))
    _ax.axhline(0, color="0.5", lw=0.8, ls=":")
    _ax.text(0.03, 0.03, "\n".join(_txt) + f"  [{_un}]",
             transform=_ax.transAxes, fontsize=9, va="bottom")
    _yr = _rlim(_y)
    if _yr:
        _ax.set_ylim(*_yr)
    _ax.set_title(_ttl, fontsize=12)
    _ax.set_ylabel(_yl)
    _ax.set_xlabel("time since quenching [Gyr]")
axs[0].legend(fontsize=10, frameon=False, loc="upper right")
_f = os.path.join(PLOTDIR, "p8_t6_concentration_clock.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8h3 — T6: does longer-living dust keep the core red? ──
# Needs 8c + 8e + 8h2 in memory (_T1, _gal_median, G8 incl. C_Av, _ZG8H, _rho).
# x = core dust-mass fraction (the surviving-dust proxy), y = the dust-only
# part of the central colour excess (T1's D_dust, top) and the attenuation
# concentration C_Av (bottom); colour = time since quenching, same sequential
# ramp as 8h1 (open grey = no quench event). If long-lived dust is what keeps
# cores red, the trend runs along x (dust) and the per-panel Spearman vs dust
# beats the one vs the clock; old-but-dusty galaxies stay in the upper right.
# Colormap floor at 0.30: an un-floored Purples renders small dt_q near-white,
# indistinguishable from the open no-quench-event markers.
from matplotlib.colors import LinearSegmentedColormap
_CMAP8H3 = LinearSegmentedColormap.from_list(
    "Purples_floor", plt.cm.Purples(np.linspace(0.30, 1.0, 256)))
_dd_gal8h, _dd_keys8h = _gal_median(_T1[CORE_LABEL]["D_dust"], _T1[CORE_LABEL]["gkey"])
_dd_of8h = dict(zip(_dd_keys8h, _dd_gal8h))
G8["D_dust_core"] = np.array([_dd_of8h.get(str(_k), np.nan)
                              for _k in np.asarray(G8["gkey"], str)])

_x8h3 = np.asarray(G8["lfd_core"], float)
_c8h3 = np.asarray(G8["dt_q"], float)
_vlim = np.nanpercentile(_c8h3, [5, 95]) if np.isfinite(_c8h3).any() else (0.0, 1.0)
_ROWS8H3 = [
    ("D_dust_core", r"central $\Delta(U-V)_{\rm dust}$ [mag]"),
    ("C_Av",        r"$C_{A_V}$ (core $-$ outskirt) [mag]"),
]
fig, axs = plt.subplots(2, len(_ZG8H), figsize=(7.5 * len(_ZG8H), 11),
                        sharex=True, sharey="row", squeeze=False,
                        constrained_layout=True)
_sc8h = None
for _cc, (_zlab, _zm, _) in enumerate(_ZG8H):
    for _rr, (_yn, _yl) in enumerate(_ROWS8H3):
        _ax = axs[_rr][_cc]
        _y = np.asarray(G8[_yn], float)
        _mq = _zm & np.isfinite(_c8h3)
        _mn = _zm & ~np.isfinite(_c8h3)
        _sc8h = _ax.scatter(_x8h3[_mq], _y[_mq], c=_c8h3[_mq], cmap=_CMAP8H3,
                            vmin=_vlim[0], vmax=_vlim[1], s=36,
                            edgecolor="k", linewidth=0.35, zorder=3)
        _ax.scatter(_x8h3[_mn], _y[_mn], facecolor="none", edgecolor="0.55",
                    s=32, zorder=2,
                    label="no quench event" if (_rr, _cc) == (0, 0) else None)
        _ax.axhline(0, color="0.5", lw=0.8, ls=":")
        _ax.text(0.03, 0.97,
                 f"vs dust:  {_rho(_x8h3[_zm], _y[_zm])}\n"
                 f"vs clock: {_rho(_c8h3[_zm], _y[_zm])}",
                 transform=_ax.transAxes, fontsize=8.5, va="top")
        print(f"{_zlab:>10s} {_yn:>12s}   vs lfd_core: {_rho(_x8h3[_zm], _y[_zm])}"
              f"   vs dt_q: {_rho(_c8h3[_zm], _y[_zm])}")
        if _cc == 0:
            _ax.set_ylabel(_yl)
    axs[0][_cc].set_title(f"{_zlab}  ({int(_zm.sum())} galaxies)", fontsize=11)
    axs[1][_cc].set_xlabel(r"core $\log\,M_{\rm dust}/M_\star$ (<3.2 kpc)")
for _rr in range(2):
    _yr = _rlim(G8[_ROWS8H3[_rr][0]])
    if _yr:
        axs[_rr][0].set_ylim(*_yr)
_xr = _rlim(_x8h3)
if _xr:
    axs[0][0].set_xlim(*_xr)
axs[0][0].legend(fontsize=10, frameon=False, loc="lower right")
fig.colorbar(_sc8h, ax=axs.ravel().tolist(), fraction=0.03, pad=0.02,
             label="time since quenching [Gyr]")
_f = os.path.join(PLOTDIR, "p8_t6_dust_longevity_redcores.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)


# Run order (cheat sheet)

1. **cluster** — `BUILD_MULTI_Z=True` → run Parts 0–1 (histories); then `BUILD_BH=True` → Part 1
   BH cell. Flip both back to `False` afterwards.
2. Parts 2–3 (selection, SFT/QT, AGN split, statistics → `powderday_quenched_selection.fits`) —
   needs only the HDF5s from step 1. Then **Part 3b** (mass–size QC): `flag_too_large` /
   `flag_unresolved` into `SELECTION_FITS` (carried into every Part 7 catalog).
3. **cluster** — Part 4 (Stage 0 particle files), apply the **powderday aperture patch** (Part 5
   markdown), Part 5 cell, then `bash submit_all_snaps.sh` in **all three** run trees under
   `output/cis25/sed_quenched_regions/<run_tag>/powderday_sed_out/` (`dusty_simdust`,
   `nodust_1e-12`, `dusty_simdust_agn`). Any time after Stage 0: **Part 4b** (star/gas/dust counts
   per projected annulus × sightline) and **Part 4c** (mass-weighted Z_star/Z_gas per aperture &
   annulus → `tables/aperture_metallicities.fits`, the Part 7d metallicity pins).
4. When `.rtout.sed` files exist: Part 6 QC (must show 5 apertures × 4 inclinations + MC
   uncertainties), then Part 7 → the rest-frame per-aperture catalogs. Then:
   - **Part 7a (required)** — the TRUE attenuation $A_V=-2.5\log_{10}F_{\rm on}/F_{\rm off}$ vs the
     ISM and quench/AGN diagnostics (`tables/attenuation_vs_ism.fits`, fiducial sightline). This is
     the only consumer of `dust_off` outside 7f and the origin of the $A_V$ reference — no CIGALE
     needed.
   - **Part 7b (required)** — the observed-frame per-aperture intermediates, all three arms × 5
     cumulative rungs × 4 sightlines. Then **Part 7b2 (required)** — differences them into the
     **3 region catalogs** (`core` 0–3.2 / `outskirt` 3.2–31.6 / `cgm` 31.6–100 kpc) that Part 7d
     actually fits; prints the star-free and NaN-band censuses (cgm-heavy is expected).
5. **Part 7c (required)** — the region-matched **formed-mass** SFH archive
   (`cigale/sfh_smoothed_regions.h5`; the builder lives in `simbanator.analysis.sfh_utils` now).
   Needs `fsps` in the kernel; confirm `sfh_mass_kind == 'formed'` in the printed attrs, or Part
   7d refuses to build. Then **Part 7c2 (required reading)** — the region-sum closure (raises on a
   geometry bug) and the sightline spread that licenses (or forbids) `SFH_REGION_IL`.
6. **cluster** — **Part 7e** (`tables/aperture_truth.fits`: aperture + annulus + **region** rows;
   also the mass weights for the region Z pins — set `OVERWRITE_APERTURE_TRUTH=True` once after
   this redesign) and **Part 8a** (`tables/annulus_ism_truth.fits`; first verify one cutout carries
   the PartType0 thermo fields). Both precede 7d now: **Part 7d0** turns the Part 8a dust masses +
   the `.rtout.sed` dust luminosities into per-region **dl2014 umin pins**
   (`tables/region_umin_pins.fits`; the emissivity table is exported once by the CIGALE env
   python, subprocessed automatically).
7. **cluster** — **Part 7d**: one run dir per (galaxy, region, sightline, chain, Z node, umin pin)
   under `output/cis25/cigale_runs_regions/`. Each run injects the region's own SFH; `dust_on` +
   `dust_off` share the dust chain, `agn_on` gets its own (`skirtor2016`). Pinned runs fit 2 592
   models (free-umin fallback 28 512). Pre-flight with `PILOT=True` + `cg.check()` + one
   interactive `cg.run()`, then `sbatch cigale_runs_regions/submit_cigale_regions.job`
   (`SKIP_IF_DONE=True`, so a resubmit only runs the gaps).
8. After the array drains — **Part 7f**: collect every `results.fits`, join the region truth, the
   per-(region, sightline) $A_V$ and the dust-mass anchor closure (`dlogMdust` vs the pin's
   `offset_dex`), write `tables/cigale_region_results.fits` + `cigale_region_av_stats.fits` and
   the figure set.
9. **Part 8 (red cores)** — **Part 8b** (`tables/annulus_av_allincl.fits`, needs the Part 7
   catalogs). **Parts 8c–8d** are pure reads of the caches; **8e** additionally needs
   `aperture_truth.fits` (step 6), `cigale_region_results.fits` (step 8) and `t_qt` from
   `SELECTION_FITS`, and builds the per-galaxy table `G8` (CIGALE $A_V$ now = the `core` and
   `outskirt` region fits) that **8f/8g** reuse — run 8c → 8e → 8f → 8g → 8h1–8h3 in order
   (8h2 must precede 8h3).

10. **2026-08-14 expansion — 9 anchors + star-forming control.** `TARGET_REDSHIFTS` is now
    z = 0.1–2 (snaps 145/134/125/116/110/105/100/090/078). The 5 new anchors need
    **BUILD_MULTI_Z** then **BUILD_BH** (cluster) before Parts 2–3. Parts 2–3 add a mass-matched
    star-forming control per anchor (`pop` = 'Q'/'SF', `match_gal_id`, `agn_class='star_forming'`,
    quench columns NaN). Part 4 skips existing cutouts; Part 5 stages `agn_on` only for pop='Q' at
    `AGN_ARM_SNAPS`, and the regenerated master jobs exit early on any galaxy whose `.rtout.sed`
    already exists. After the new RT: rebuild the caches with their OVERWRITE flags (4b, 4c, 7e,
    8a, 8b) and re-run Part 7 + 7a–7f (CIGALE 7d skips completed run dirs by default), then Part 8.

Old → new part labels: 2026-08-10 prune (`7d2→7c`, `7e→7d`, `7f→7e`, `7g→7f`; deleted parts in
`powderday_flux_quenched_m25.ipynb.pre-cleanup.bak`); **2026-08-15 region redesign** — Part 7b2
(region inputs) and Part 7d0 (umin pins) are new, Part 7c/7d/7f are region-based, and the
galaxy-pinned cumulative campaign under `cigale_runs_pinned/` is superseded (tree kept on disk;
delete manually together with the stale `cigale/sfh_smoothed_aperture.h5` and
`tables/cigale_pinned_*.fits` when no longer needed).

**Caveats.**
- **The pinned fit is a best case, not an observation.** With the SFH, the mass-weighted age, the
  metallicity and now the dust mass (umin pin) all fixed from the simulation, the recovered $A_V$
  is the *floor* of the systematic: a real fit with a free SFH will do worse. That is the point —
  it isolates dust from the age–metallicity–dust degeneracy — but the paper text has to say so.
- **The SPS-library mismatch is still the dominant systematic.** powderday renders with FSPS,
  CIGALE fits with BC03; at fixed age and $Z$ they differ by ~0.05–0.15 mag in optical colour, and
  $A_V$ is the only free knob left to absorb it. The `dust_off` set (truth $A_V\equiv0$) measures
  the zero-point **per region** (`Av_zp`, `dAv_zpcorr` in Part 7f). Do not quote an $A_V$ offset
  smaller than it.
- **Regions are broad zones, not closed systems.** A region does not contain all the dust heated
  by its own stars, nor only its own heating, so CIGALE's energy balance — the pathway that sets
  $A_V$ — is only approximately closed per region. With 3 broad zones the leakage is far smaller
  than for the abandoned 5-annulus design and it is accepted: core+outskirt carry the science, and
  the **cgm region is allowed to fail** (SFH `nstar_min` skips, NaN-band drops, large $\chi^2$ —
  all ledgered in `cigale_region_skipped.fits` and the 7f censuses, never fatal).
- **Coverage is stellar AND photometric per region.** `SFH_NSTAR_MIN = 20` excludes a region from
  the Part 7c archive → that (galaxy, region, sightline) gets no run; `MIN_FIT_BANDS` drops
  photometrically thin rows. Both censuses are printed; cgm dominates both by design.
- **`bc03.metallicity` is coarse where these galaxies live** (0.008 / 0.02 / 0.05) and a single
  pinned node marginalises over nothing. Runs are keyed by their own region node, so a region that
  straddles one costs an extra run rather than a wrong pin. If $\Delta A_V$ comes out bimodal,
  colour it by `zs_idx`.
- **The umin pin anchors, it does not measure.** `dust.mass` in the pinned runs is anchored to the
  Part 8a truth by construction — quote `dlogMdust` as the pin-closure check, not as a recovery.
  Runs tagged `_ufree` (no usable pin: `no_dust`/`no_lum`, cgm-dominated) are genuine recoveries
  on the free 11-node grid. The pin is computed at qpah=2.5, γ=0.02; Part 7d0 prints how often
  another γ of the free grid would shift the node.
- Stage 0 writes **100 pkpc region cutouts** (CGM + satellites, periodic-wrap safe); the RT grid is
  ±100 kpc (`zoom_box_len`), and the 5 hyperion-log-spaced apertures (1, 3.16, 10, 31.6, 100 kpc)
  sample central → outskirts. `N_AP/AP_MIN_KPC/AP_MAX_KPC` + `THETA_DEG/PHI_DEG` here must match
  `SED_APERTURE_*` / `THETA/PHI` in `simbanator/sed/parameters_master*.py` at RT time. The 3
  regions are built by differencing those rungs — changing `REGION_DEFS` needs no new RT as long
  as the boundaries stay on rung radii.
- The **`agn_on` run** is `dust_on` + AGN point sources (`parameters_master-agn.py`: `BH_SED=True`,
  Hopkins+2007 template, `BH_var=False`). It needs `PartType5` in the Stage-0 cutouts — re-run
  Part 4 first. Only its chain carries `skirtor2016`; Part 7f's `agn_bias.png` shows the mix of
  real AGN light and extra model freedom together.
- `<filter>_err` is the Hyperion **Monte-Carlo photon noise** propagated through the filter
  convolution — an RT-convergence error, not a mock observational depth. Differencing rungs can
  drive a region flux negative at that noise level → NaN band (counted in 7b2), and the region
  error is `sqrt(err_out² − err_in²)` with a quadrature-sum fallback.
- Part 7 fluxes are rest-frame convolved; Part 7b re-extracts observed-frame for CIGALE (same
  `extract_flux_set` helper, `redshift=True`). Part 7a's $A_V$ is therefore a **rest-frame**
  attenuation, directly comparable across anchors.
- The dust_off run uses 1 dust-RT photon (`parameters_master-nodust.py`) — some galaxies can
  crash/truncate; the Part 7 cross-check + `missing_sources_*.txt` make any loss explicit.
- CIGALE error budget: the input files carry raw MC errors; the fit adds `additionalerror = 0.1`
  (10 %) in quadrature via `prepare_run` — change it there, not in Part 7b/7b2.
- **Negative errors** = "upper limit" to CIGALE. Catalogs written before the
  `convolveFilterWithSED` sign fix are all-negative → all-NaN fits; Part 7d's preflight
  (`cigale.sanitize_input_errors`) repairs them in place.
- An **all-zero SFH column** would make `sfhfromfile`'s `normalise=True` divide by zero and produce
  an all-NaN `results.fits` that looks converged. `cigale.write_sfhfromfile` refuses to write it,
  `cigale.validate_sfh_file` (called from `prepare_run`) double-checks, and the affected rows land
  in `cigale_region_skipped.fits` rather than silently vanishing. A cgm SFH that is near-zero but
  not exactly zero passes the guard with a noise-dominated shape — watch `sfr_hold_frac` and the
  cgm $\chi^2$ before quoting anything there.

- Part 8's $\Sigma$/DGR/$f_{\rm cold}$ are masked where a projected annulus holds fewer
  than `NGAS_ANN_MIN = 10` gas particles (Part 8a prints the fraction); the 1 kpc rung is
  the worst sampled, so every T2 headline number and every 8e–8g core quantity is quoted
  at `ap3kpc`. The T2 screen uses the RT's own KMH94 opacity read from
  `$POWDERDAY_ROOT/hyperion-dust/dust_files/kmh94_3.1_hg.hdf5` — if Part 8a printed the
  MW-like fallback instead, the ratios carry a ~×2 opacity uncertainty.